# Daily Systematic Gold Macro Model

Fixed. **Nothing removed** — all 37 cells kept.

| Fix | Why |
|---|---|
| **13 hardcoded `/Users/elena_nael/...` save paths** → `outputs/` | `plt.savefig` raised `FileNotFoundError` on any other machine, and on yours if the folder moved. |
| **3 nbconvert cells** → portable | Hardcoded `/opt/anaconda3/bin/jupyter` and a hardcoded notebook path. Now uses `sys.executable -m jupyter` and finds the notebook in the current folder. |
| **2 `conda install` cells** → `!conda install` | A bare shell command in a code cell is a `SyntaxError`. |
| **yfinance MultiIndex** flattened after every `yf.download` | Newer yfinance returns MultiIndex columns; `df['Close']` then returns a DataFrame, not a Series. |
| **`float(...).iloc[n]`** → `float(np.asarray(...)[n])` | pandas FutureWarning; becomes a `TypeError` in a future release. |

**Already correct — left alone:** every data source is a live `yf.download`,
so it updates to the latest bar each time you run it. The `np.random` calls in
cells 13, 25 and 29 are Monte Carlo path generation (GBM shocks, `N_PATHS`),
not fake data — seeded for reproducibility, which is right.

Regime based gold trading framework
¶

The "Weather Forecast"
¶
This model combines three macro factors to determine the overall environment for gold:


Real Yield Signal (40% weight): Are inflation-protected bonds (TIPs) rising or falling relative to nominal yields? Currently neutral (0) — no clear tailwind or headwind from real rates.
DXY Signal (35% weight): Is the dollar strengthening or weakening? Currently bearish (-1) — a weaker dollar is normally gold-positive, but it's being dragged down by other signals.
Gold Momentum (25% weight): Is gold's price trend positive? Currently neutral (0).


Current Score: -0.35 → NEUTRAL
The model says there's no strong macro edge right now. The dollar weakness is the only bullish input, but it's not enough to push the composite into "bull" territory. The model says wait for clarity rather than press a trade.

In [ ]:
# ============================================================
# MACRO REGIME MODEL (MRM) FOR GOLD — FIXED
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a1a'
plt.rcParams['axes.edgecolor']   = '#333333'
plt.rcParams['axes.labelcolor']  = '#cccccc'
plt.rcParams['xtick.color']      = '#888888'
plt.rcParams['ytick.color']      = '#888888'
plt.rcParams['text.color']       = '#cccccc'
plt.rcParams['grid.color']       = '#2a2a2a'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['font.size']        = 11

GOLD_COLOR   = '#FFD700'
GREEN_COLOR  = '#00C896'
RED_COLOR    = '#FF4D4D'
BLUE_COLOR   = '#4D9FFF'
ORANGE_COLOR = '#FF8C42'
GRAY_COLOR   = '#888888'

# ── DOWNLOAD EACH TICKER SEPARATELY (fixes column mix-up) ────
print('Downloading data...')

def get(ticker, period='3y'):
    df = yf.download(ticker, period=period, auto_adjust=True, progress=False)['Close']
    if hasattr(df, 'columns') and isinstance(df.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
        df.columns = df.columns.get_level_values(0)
    df = df.squeeze()  # ensure Series not DataFrame
    df.name = ticker
    return df.dropna()

gold = get('GC=F')
tnx  = get('^TNX')
tip  = get('TIP')
dxy  = get('DX-Y.NYB')
rinf = get('RINF')

# Combine on common dates
raw = pd.DataFrame({
    'Gold': gold,
    'TNX' : tnx,
    'TIP' : tip,
    'DXY' : dxy,
    'RINF': rinf,
}).dropna(how='all').ffill().dropna()

# ── SANITY CHECK ─────────────────────────────────────────────
print(f'\nSanity check (last values):')
print(f'  Gold : ${raw["Gold"].iloc[-1]:,.0f}   ← should be ~$2,900-3,200')
print(f'  TNX  : {raw["TNX"].iloc[-1]:.2f}%    ← should be ~4-5%')
print(f'  DXY  : {raw["DXY"].iloc[-1]:.2f}     ← should be ~100-110')
print(f'  TIP  : ${raw["TIP"].iloc[-1]:.2f}   ← should be ~$100-115')
print(f'Rows   : {len(raw)}  |  {raw.index[0].date()} → {raw.index[-1].date()}\n')

# ── REAL YIELD SIGNAL ─────────────────────────────────────────
raw['TIP_Return']  = raw['TIP'].pct_change(20)
raw['TNX_20d_chg'] = raw['TNX'].diff(20)

raw['RealYield_Signal'] = 0
raw.loc[(raw['TIP_Return'] < -0.01) & (raw['TNX_20d_chg'] > 0.1),  'RealYield_Signal'] = -1
raw.loc[(raw['TIP_Return'] >  0.01) & (raw['TNX_20d_chg'] < -0.1), 'RealYield_Signal'] = +1

# ── DXY SIGNAL ────────────────────────────────────────────────
raw['DXY_MA20']   = raw['DXY'].rolling(20).mean()
raw['DXY_ROC']    = raw['DXY'].pct_change(10)
raw['DXY_Signal'] = np.where(raw['DXY'] > raw['DXY_MA20'], -1, +1)
raw.loc[raw['DXY_ROC'].abs() < 0.005, 'DXY_Signal'] = 0

# ── GOLD MOMENTUM SIGNAL ──────────────────────────────────────
raw['Gold_MA20'] = raw['Gold'].rolling(20).mean()
raw['Gold_MA50'] = raw['Gold'].rolling(50).mean()
raw['Gold_Signal'] = np.where(
    (raw['Gold'] > raw['Gold_MA20']) & (raw['Gold_MA20'] > raw['Gold_MA50']), +1,
    np.where(
        (raw['Gold'] < raw['Gold_MA20']) & (raw['Gold_MA20'] < raw['Gold_MA50']), -1, 0
    )
)

# ── VOLATILITY REGIME ─────────────────────────────────────────
raw['Gold_Vol20'] = raw['Gold'].pct_change().rolling(20).std() * np.sqrt(252)
raw['Gold_VolMA'] = raw['Gold_Vol20'].rolling(60).mean()
raw['Vol_Regime'] = np.where(raw['Gold_Vol20'] > raw['Gold_VolMA'] * 1.3, 'HIGH', 'NORMAL')

# ── COMPOSITE MRM SCORE ───────────────────────────────────────
raw['MRM_Score'] = (
    raw['RealYield_Signal'] * 0.40 +
    raw['DXY_Signal']       * 0.35 +
    raw['Gold_Signal']      * 0.25
)
raw['MRM_MA5']  = raw['MRM_Score'].rolling(5).mean()
raw['MRM_MA20'] = raw['MRM_Score'].rolling(20).mean()

def classify_regime(score):
    if score >= 0.4:    return 'BULL'
    elif score <= -0.4: return 'BEAR'
    else:               return 'NEUTRAL'

raw['Regime'] = raw['MRM_Score'].apply(classify_regime)
df = raw.dropna().copy()

# ── PLOT ──────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True)
fig.suptitle('MACRO REGIME MODEL (MRM) — Gold Analysis', fontsize=15, color=GOLD_COLOR, y=0.99)

ax = axes[0]
ax.plot(df.index, df['Gold'], color=GOLD_COLOR, linewidth=1.5, label='Gold Futures')
ax.plot(df.index, df['Gold_MA20'], color=BLUE_COLOR,   linewidth=1, linestyle='--', alpha=0.7, label='MA20')
ax.plot(df.index, df['Gold_MA50'], color=ORANGE_COLOR, linewidth=1, linestyle='--', alpha=0.7, label='MA50')
for i in range(1, len(df)):
    if df['Regime'].iloc[i] == 'BULL':
        ax.axvspan(df.index[i-1], df.index[i], alpha=0.07, color=GREEN_COLOR)
    elif df['Regime'].iloc[i] == 'BEAR':
        ax.axvspan(df.index[i-1], df.index[i], alpha=0.07, color=RED_COLOR)
ax.set_title('Gold Price + Macro Regime (Green=Bull, Red=Bear, No shade=Neutral)', fontsize=10)
ax.set_ylabel('Price (USD)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True)

ax2 = axes[1]
ax2.plot(df.index, df['TIP'], color=BLUE_COLOR, linewidth=1.2, label='TIP ETF')
ax2r = ax2.twinx()
ax2r.plot(df.index, df['TNX'], color=RED_COLOR, linewidth=1, linestyle='--', alpha=0.7, label='10Y Yield %')
ax2.set_title('Real Yield: TIP ETF (left) vs 10Y Yield % (right) — TIP falling + Yield rising = Bearish Gold', fontsize=10)
ax2.set_ylabel('TIP Price', color=BLUE_COLOR)
ax2r.set_ylabel('10Y Yield %', color=RED_COLOR)
ax2.legend(loc='upper left', fontsize=9)
ax2r.legend(loc='upper right', fontsize=9)
ax2.grid(True)

ax3 = axes[2]
ax3.plot(df.index, df['DXY'], color=ORANGE_COLOR, linewidth=1.2, label='DXY')
ax3.plot(df.index, df['DXY_MA20'], color='white', linewidth=1, linestyle='--', alpha=0.5, label='DXY MA20')
ax3.fill_between(df.index, df['DXY'], df['DXY_MA20'],
                 where=(df['DXY'] > df['DXY_MA20']), alpha=0.15, color=RED_COLOR,   label='Dollar Strong → Gold ↓')
ax3.fill_between(df.index, df['DXY'], df['DXY_MA20'],
                 where=(df['DXY'] < df['DXY_MA20']), alpha=0.15, color=GREEN_COLOR, label='Dollar Weak → Gold ↑')
ax3.set_title('DXY Dollar Index vs MA20', fontsize=10)
ax3.set_ylabel('DXY')
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(True)

ax4 = axes[3]
colors_mrm = [GREEN_COLOR if v >= 0.4 else (RED_COLOR if v <= -0.4 else GRAY_COLOR) for v in df['MRM_Score']]
ax4.bar(df.index, df['MRM_Score'], color=colors_mrm, alpha=0.6, width=1)
ax4.plot(df.index, df['MRM_MA5'],  color='white',      linewidth=1.2, label='5d MA')
ax4.plot(df.index, df['MRM_MA20'], color=ORANGE_COLOR, linewidth=1,   linestyle='--', label='20d MA')
ax4.axhline( 0.4, color=GREEN_COLOR, linewidth=0.8, linestyle=':')
ax4.axhline(-0.4, color=RED_COLOR,   linewidth=0.8, linestyle=':')
ax4.axhline( 0,   color='white',     linewidth=0.5, linestyle='--')
ax4.set_title('MRM Composite Score', fontsize=10)
ax4.set_ylabel('Score')
ax4.legend(loc='upper left', fontsize=9)
ax4.grid(True)

plt.tight_layout()
plt.show()

# ── CURRENT READING ───────────────────────────────────────────
latest = df.iloc[-1]
print()
print('=' * 60)
print('  MACRO REGIME MODEL — CURRENT READING')
print('=' * 60)
print(f'  Date:              {df.index[-1].date()}')
print(f'  Gold Price:        ${latest["Gold"]:,.0f}')
print(f'  10Y Yield:         {latest["TNX"]:.2f}%')
print(f'  DXY:               {latest["DXY"]:.2f}')
print(f'  Vol Regime:        {latest["Vol_Regime"]}')
print()
print(f'  Real Yield Signal: {int(latest["RealYield_Signal"]):+d}  (weight 40%)')
print(f'  DXY Signal:        {int(latest["DXY_Signal"]):+d}  (weight 35%)')
print(f'  Gold Momentum:     {int(latest["Gold_Signal"]):+d}  (weight 25%)')
print()
print(f'  MRM SCORE:         {latest["MRM_Score"]:+.2f}')
print(f'  REGIME:            {latest["Regime"]}')
print()
regime = latest['Regime']
vol    = latest['Vol_Regime']
if   regime == 'BULL'    and vol == 'NORMAL': print('  ACTION: FULL LONG BIAS — Macro tailwind, size normally')
elif regime == 'BULL'    and vol == 'HIGH':   print('  ACTION: LONG BIAS — Reduce size 30-50%, high vol')
elif regime == 'BEAR'    and vol == 'NORMAL': print('  ACTION: SHORT BIAS — Macro headwind, size normally')
elif regime == 'BEAR'    and vol == 'HIGH':   print('  ACTION: SHORT BIAS — Reduce size 30-50%, high vol')
elif regime == 'NEUTRAL':                     print('  ACTION: NEUTRAL — No clear macro edge, wait for clarity')
print('=' * 60)


The "Pattern Recognition" Model
¶
This is a statistical model that looks at the hidden state gold is in, based on price behavior patterns rather than macro inputs. It essentially asks: "Does gold's recent price action look more like past bull, bear, or neutral periods?"
Current Reading: BULL with 100% confidence
This is a strong signal. The model is completely certain gold is in a bull regime right now. The transition matrix (Image 3) shows:


If you're in BULL, there's a 96% chance you stay in BULL tomorrow
There's essentially 0% chance of jumping directly to BEAR from BULL
Bull regimes are sticky and tend to persist


Strategy implication: Buy dips to the 20-day moving average. Don't short against the trend.

In [ ]:
# ============================================================
# HIDDEN MARKOV MODEL (HMM) FOR GOLD
# Detects hidden market regimes automatically:
# State 0 = Trending Bull | State 1 = Choppy | State 2 = Trending Bear
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a1a'
plt.rcParams['axes.edgecolor']   = '#333333'
plt.rcParams['axes.labelcolor']  = '#cccccc'
plt.rcParams['xtick.color']      = '#888888'
plt.rcParams['ytick.color']      = '#888888'
plt.rcParams['text.color']       = '#cccccc'
plt.rcParams['grid.color']       = '#2a2a2a'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['font.size']        = 11

GOLD_COLOR   = '#FFD700'
GREEN_COLOR  = '#00C896'
RED_COLOR    = '#FF4D4D'
BLUE_COLOR   = '#4D9FFF'
ORANGE_COLOR = '#FF8C42'

# ── DOWNLOAD DATA ────────────────────────────────────────────
print('Downloading gold data...')
gold = yf.download('GC=F', period='5y', auto_adjust=True)['Close']
if hasattr(gold, 'columns') and isinstance(gold.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gold.columns = gold.columns.get_level_values(0)
gold = gold.dropna()
gold.name = 'Gold'
print(f'Loaded {len(gold)} rows  |  {gold.index[0].date()} → {gold.index[-1].date()}')

# ── FEATURE ENGINEERING ──────────────────────────────────────
df = pd.DataFrame(index=gold.index)
df['Gold']      = gold.values.flatten()
df['Return']    = df['Gold'].pct_change()
df['Return5']   = df['Gold'].pct_change(5)
df['Vol5']      = df['Return'].rolling(5).std()
df['Vol20']     = df['Return'].rolling(20).std()
df['VolRatio']  = df['Vol5'] / (df['Vol20'] + 1e-8)   # vol spike detector
df['MA20']      = df['Gold'].rolling(20).mean()
df['MA50']      = df['Gold'].rolling(50).mean()
df['Trend']     = (df['Gold'] - df['MA20']) / (df['MA20'] + 1e-8)  # distance from MA20
df['Momentum']  = df['Return'].rolling(10).mean()

df = df.dropna()

# ── TRAIN HMM ────────────────────────────────────────────────
N_STATES = 3   # Bull / Neutral / Bear

features = ['Return', 'Vol5', 'VolRatio', 'Trend', 'Momentum']
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Training HMM with {N_STATES} states on {len(X_scaled)} observations...')

best_model = None
best_score = -np.inf

# Run multiple times — HMM can get stuck in local optima
for seed in range(10):
    try:
        model = GaussianHMM(
            n_components=N_STATES,
            covariance_type='full',
            n_iter=200,
            random_state=seed,
            tol=1e-4
        )
        model.fit(X_scaled)
        score = model.score(X_scaled)
        if score > best_score:
            best_score = score
            best_model = model
    except Exception:
        continue

print(f'HMM trained. Best log-likelihood: {best_score:.2f}')

# ── DECODE STATES ────────────────────────────────────────────
states = best_model.predict(X_scaled)
df['State'] = states

# Get state probabilities (soft assignment)
state_probs = best_model.predict_proba(X_scaled)
for i in range(N_STATES):
    df[f'P_State{i}'] = state_probs[:, i]

# ── LABEL STATES by mean return ──────────────────────────────
state_stats = df.groupby('State').agg(
    Mean_Return=('Return', 'mean'),
    Mean_Vol=('Vol5', 'mean'),
    Count=('Return', 'count')
).reset_index()

state_stats['Annualized_Return'] = state_stats['Mean_Return'] * 252
state_stats['Annualized_Vol']    = state_stats['Mean_Vol'] * np.sqrt(252)
state_stats = state_stats.sort_values('Annualized_Return', ascending=False).reset_index(drop=True)

print('\nState Statistics:')
print(state_stats[['State', 'Annualized_Return', 'Annualized_Vol', 'Count']].to_string(index=False))

# Map states to labels
sorted_states = state_stats['State'].values
state_labels  = {}
state_colors  = {}
label_names   = ['BULL', 'NEUTRAL', 'BEAR']
color_map     = [GREEN_COLOR, ORANGE_COLOR, RED_COLOR]

for i, s in enumerate(sorted_states):
    state_labels[s] = label_names[i]
    state_colors[s] = color_map[i]

df['Regime_Label'] = df['State'].map(state_labels)
df['Regime_Color'] = df['State'].map(state_colors)

# ── TRANSITION MATRIX ────────────────────────────────────────
trans = best_model.transmat_
trans_df = pd.DataFrame(
    trans,
    index=[state_labels[i] for i in range(N_STATES)],
    columns=[state_labels[i] for i in range(N_STATES)]
)

# ── PLOT ─────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True)
fig.suptitle('HIDDEN MARKOV MODEL (HMM) — Gold Regime Detection', fontsize=15, color=GOLD_COLOR, y=0.99)

# Panel 1: Gold price with regime shading
ax = axes[0]
ax.plot(df.index, df['Gold'], color=GOLD_COLOR, linewidth=1.5, label='Gold', zorder=3)

for i in range(1, len(df)):
    color = df['Regime_Color'].iloc[i]
    ax.axvspan(df.index[i-1], df.index[i], alpha=0.15, color=color, linewidth=0)

from matplotlib.patches import Patch
legend_patches = [
    Patch(color=GREEN_COLOR,  alpha=0.5, label='Bull Regime'),
    Patch(color=ORANGE_COLOR, alpha=0.5, label='Neutral Regime'),
    Patch(color=RED_COLOR,    alpha=0.5, label='Bear Regime'),
]
ax.legend(handles=legend_patches + [ax.lines[0]], loc='upper left', fontsize=9)
ax.set_title('Gold Price — HMM Regime Background', fontsize=10)
ax.set_ylabel('Price (USD)')
ax.grid(True)

# Panel 2: State probabilities stacked
ax2 = axes[1]
bull_state = [k for k, v in state_labels.items() if v == 'BULL'][0]
neut_state = [k for k, v in state_labels.items() if v == 'NEUTRAL'][0]
bear_state = [k for k, v in state_labels.items() if v == 'BEAR'][0]

ax2.fill_between(df.index, df[f'P_State{bull_state}'], 0,
                 color=GREEN_COLOR, alpha=0.5, label='P(Bull)')
ax2.fill_between(df.index, df[f'P_State{neut_state}'], 0,
                 color=ORANGE_COLOR, alpha=0.4, label='P(Neutral)')
ax2.fill_between(df.index, df[f'P_State{bear_state}'], 0,
                 color=RED_COLOR, alpha=0.4, label='P(Bear)')
ax2.set_title('State Probabilities — Confidence in Current Regime', fontsize=10)
ax2.set_ylabel('Probability')
ax2.set_ylim(0, 1)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True)

# Panel 3: Volatility with regime
ax3 = axes[2]
ax3.plot(df.index, df['Vol5']  * np.sqrt(252), color=RED_COLOR,   linewidth=1,   label='Vol 5d (ann.)')
ax3.plot(df.index, df['Vol20'] * np.sqrt(252), color=BLUE_COLOR,  linewidth=1.2, label='Vol 20d (ann.)')
ax3.fill_between(df.index, 0, df['Vol5'] * np.sqrt(252),
                 where=(df['Vol5'] > df['Vol20']),
                 alpha=0.2, color=RED_COLOR, label='Vol Spike (reduce size)')
ax3.set_title('Annualized Volatility — Red fill = Vol spike = reduce position size', fontsize=10)
ax3.set_ylabel('Annualized Vol')
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(True)

# Panel 4: Discrete state over time
ax4 = axes[3]
regime_num = df['Regime_Label'].map({'BULL': 1, 'NEUTRAL': 0, 'BEAR': -1})
colors_reg = [GREEN_COLOR if v == 1 else (RED_COLOR if v == -1 else ORANGE_COLOR) for v in regime_num]
ax4.bar(df.index, regime_num, color=colors_reg, alpha=0.7, width=1)
ax4.axhline(0, color='white', linewidth=0.5, linestyle='--')
ax4.set_yticks([-1, 0, 1])
ax4.set_yticklabels(['BEAR', 'NEUTRAL', 'BULL'])
ax4.set_title('Detected Regime Over Time', fontsize=10)
ax4.grid(True, axis='y')

plt.tight_layout()
plt.show()

# ── TRANSITION MATRIX PLOT ───────────────────────────────────
fig2, ax = plt.subplots(1, 1, figsize=(6, 5))
fig2.patch.set_facecolor('#0f0f0f')
ax.set_facecolor('#1a1a1a')

im = ax.imshow(trans_df.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(N_STATES))
ax.set_yticks(range(N_STATES))
ax.set_xticklabels(trans_df.columns, fontsize=11)
ax.set_yticklabels(trans_df.index, fontsize=11)
ax.set_title('HMM Transition Matrix\n(Probability of moving FROM row TO column)', color=GOLD_COLOR)

for i in range(N_STATES):
    for j in range(N_STATES):
        ax.text(j, i, f'{trans_df.values[i,j]:.2f}', ha='center', va='center',
                fontsize=12, fontweight='bold',
                color='black' if trans_df.values[i,j] > 0.5 else 'white')

plt.tight_layout()
plt.show()

# ── CURRENT REGIME READING ────────────────────────────────────
latest     = df.iloc[-1]
current    = latest['Regime_Label']
p_bull     = latest[f'P_State{bull_state}']
p_neut     = latest[f'P_State{neut_state}']
p_bear     = latest[f'P_State{bear_state}']
persistence = trans_df.loc[current, current]

print()
print('=' * 60)
print('  HMM — CURRENT REGIME READING')
print('=' * 60)
print(f'  Date:              {df.index[-1].date()}')
print(f'  Gold Price:        ${latest["Gold"]:,.0f}')
print(f'  Current Regime:    {current}')
print()
print(f'  P(Bull):           {p_bull:.1%}')
print(f'  P(Neutral):        {p_neut:.1%}')
print(f'  P(Bear):           {p_bear:.1%}')
print()
print(f'  Regime Persistence: {persistence:.1%} chance of staying in {current}')
print()
print('  Transition Matrix:')
print(trans_df.round(2).to_string())
print()

if current == 'BULL':
    print('  STRATEGY: Trade long-only. Buy dips to MA20.')
    print('  SIZE:     Full size if Vol regime = NORMAL')
    print('  AVOID:    Shorting against trend in this regime')
elif current == 'BEAR':
    print('  STRATEGY: Short bias or stay flat. Sell rallies to MA20.')
    print('  SIZE:     Full size if Vol regime = NORMAL')
    print('  AVOID:    Buying dips — they become lower lows')
else:
    print('  STRATEGY: Range trade. Buy support, sell resistance.')
    print('  SIZE:     Reduce to 50% — choppy conditions, tight stops')
    print('  AVOID:    Trend-following entries — they get chopped out')

print('=' * 60)


_______________________________________________
¶

The "Position Sizing" Model
¶
GARCH measures how turbulent conditions are, regardless of direction. High volatility doesn't mean bearish — it means dangerous and choppy, so you should trade smaller.
Current Reading: HIGH VOLATILITY — 0.5x normal size


GARCH vol: 25.4% annualized (above the 19.2% high threshold)
Historical 20-day vol: 28.2% — even higher
The next 5 days are all forecast to stay in HIGH vol territory (~23-25%)


This means even if you're bullish on gold, the model says cut your position in half because the swings are large enough to stop you out even when you're directionally correct. This is pure risk management.

In [ ]:
# ============================================================
# VOLATILITY REGIME + GARCH FOR GOLD
# GARCH tells you: when to size up vs size down
# High vol clusters = stay small. Low vol trending = size up.
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a1a'
plt.rcParams['axes.edgecolor']   = '#333333'
plt.rcParams['axes.labelcolor']  = '#cccccc'
plt.rcParams['xtick.color']      = '#888888'
plt.rcParams['ytick.color']      = '#888888'
plt.rcParams['text.color']       = '#cccccc'
plt.rcParams['grid.color']       = '#2a2a2a'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['font.size']        = 11

GOLD_COLOR   = '#FFD700'
GREEN_COLOR  = '#00C896'
RED_COLOR    = '#FF4D4D'
BLUE_COLOR   = '#4D9FFF'
ORANGE_COLOR = '#FF8C42'

# ── DOWNLOAD DATA ─────────────────────────────────────────────
print('Downloading gold data...')
gold = yf.download('GC=F', period='5y', auto_adjust=True, progress=False)['Close'].squeeze()
if hasattr(gold, 'columns') and isinstance(gold.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gold.columns = gold.columns.get_level_values(0)
gold = gold.dropna()

returns = gold.pct_change().dropna() * 100  # in percent for GARCH

print(f'Loaded {len(gold)} rows  |  {gold.index[0].date()} → {gold.index[-1].date()}')

# ── FIT GARCH(1,1) MODEL ──────────────────────────────────────
print('Fitting GARCH(1,1) model...')

am = arch_model(
    returns,
    vol='Garch',
    p=1, q=1,
    mean='Constant',
    dist='normal'
)
res = am.fit(disp='off')
print(f'GARCH fitted. AIC: {res.aic:.2f}  BIC: {res.bic:.2f}')
print(res.summary().tables[1])

# ── EXTRACT CONDITIONAL VOLATILITY ────────────────────────────
cond_vol = res.conditional_volatility  # daily vol in %
annualized_vol = cond_vol * np.sqrt(252)

# ── FORECAST NEXT 5 DAYS ──────────────────────────────────────
forecast = res.forecast(horizon=5, reindex=False)
forecast_vol = np.sqrt(forecast.variance.values[-1]) * np.sqrt(252)
print(f'\n5-Day Volatility Forecast (annualized):')
for i, v in enumerate(forecast_vol):
    print(f'  Day {i+1}: {v:.1f}%')

# ── ROLLING HISTORICAL VOL ─────────────────────────────────────
df = pd.DataFrame(index=gold.index)
df['Gold']    = gold.values
df['Return']  = returns.reindex(gold.index)
df['Vol5']    = df['Return'].rolling(5).std()  * np.sqrt(252)
df['Vol20']   = df['Return'].rolling(20).std() * np.sqrt(252)
df['Vol60']   = df['Return'].rolling(60).std() * np.sqrt(252)
df['GARCH_Vol'] = annualized_vol.reindex(gold.index)
df = df.dropna()

# ── VOLATILITY REGIME CLASSIFICATION ─────────────────────────
vol_mean = df['GARCH_Vol'].mean()
vol_std  = df['GARCH_Vol'].std()

low_threshold  = vol_mean - 0.5 * vol_std
high_threshold = vol_mean + 0.5 * vol_std

def vol_regime(v):
    if v < low_threshold:   return 'LOW'
    elif v > high_threshold: return 'HIGH'
    else:                    return 'NORMAL'

df['Vol_Regime'] = df['GARCH_Vol'].apply(vol_regime)

# Position sizing multiplier based on vol regime
sizing = {'LOW': 1.5, 'NORMAL': 1.0, 'HIGH': 0.5}
df['Size_Multiplier'] = df['Vol_Regime'].map(sizing)

# ── PLOT ──────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True)
fig.suptitle('VOLATILITY REGIME + GARCH — Gold Position Sizing', fontsize=15, color=GOLD_COLOR, y=0.99)

# Panel 1: Gold price
ax1 = axes[0]
ax1.plot(df.index, df['Gold'], color=GOLD_COLOR, linewidth=1.5)
for i in range(1, len(df)):
    regime = df['Vol_Regime'].iloc[i]
    if regime == 'HIGH':
        ax1.axvspan(df.index[i-1], df.index[i], alpha=0.1, color=RED_COLOR)
    elif regime == 'LOW':
        ax1.axvspan(df.index[i-1], df.index[i], alpha=0.1, color=GREEN_COLOR)
ax1.set_title('Gold Price — Red=High Vol (reduce size), Green=Low Vol (size up)', fontsize=10)
ax1.set_ylabel('Price (USD)')
ax1.grid(True)

# Panel 2: GARCH vs Historical Vol
ax2 = axes[1]
ax2.plot(df.index, df['GARCH_Vol'], color=ORANGE_COLOR, linewidth=1.5, label='GARCH Conditional Vol')
ax2.plot(df.index, df['Vol20'],     color=BLUE_COLOR,   linewidth=1.0, linestyle='--', alpha=0.7, label='20d Historical Vol')
ax2.axhline(low_threshold,  color=GREEN_COLOR, linewidth=1, linestyle=':', label=f'Low threshold ({low_threshold:.1f}%)')
ax2.axhline(high_threshold, color=RED_COLOR,   linewidth=1, linestyle=':', label=f'High threshold ({high_threshold:.1f}%)')
ax2.axhline(vol_mean,       color='white',     linewidth=0.7, linestyle='--', alpha=0.5)
ax2.fill_between(df.index, df['GARCH_Vol'], high_threshold,
                 where=(df['GARCH_Vol'] > high_threshold), color=RED_COLOR, alpha=0.2, label='High Vol Zone')
ax2.fill_between(df.index, df['GARCH_Vol'], low_threshold,
                 where=(df['GARCH_Vol'] < low_threshold), color=GREEN_COLOR, alpha=0.2, label='Low Vol Zone')
ax2.set_title('GARCH Conditional Volatility (annualized) — Size UP in green, SIZE DOWN in red', fontsize=10)
ax2.set_ylabel('Annualized Vol %')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True)

# Panel 3: Position size multiplier
ax3 = axes[2]
colors_size = [GREEN_COLOR if v > 1 else (RED_COLOR if v < 1 else BLUE_COLOR) for v in df['Size_Multiplier']]
ax3.bar(df.index, df['Size_Multiplier'], color=colors_size, alpha=0.7, width=1)
ax3.axhline(1.0, color='white', linewidth=0.8, linestyle='--', label='Normal size (1.0x)')
ax3.set_title('Recommended Position Size Multiplier — 1.5x (low vol) / 1.0x (normal) / 0.5x (high vol)', fontsize=10)
ax3.set_ylabel('Size Multiplier')
ax3.set_ylim(0, 2)
ax3.legend(fontsize=9)
ax3.grid(True)

# Panel 4: Return distribution
ax4 = axes[3]
ax4.bar(df.index, df['Return'], color=[GREEN_COLOR if r > 0 else RED_COLOR for r in df['Return']], alpha=0.6, width=1)
ax4.set_title('Daily Returns (%)', fontsize=10)
ax4.set_ylabel('Return %')
ax4.axhline(0, color='white', linewidth=0.5)
ax4.grid(True)

plt.tight_layout()
plt.show()

# ── FORECAST PLOT ─────────────────────────────────────────────
fig2, ax = plt.subplots(figsize=(10, 4))
fig2.patch.set_facecolor('#0f0f0f')
ax.set_facecolor('#1a1a1a')
days = [f'Day {i+1}' for i in range(5)]
bar_colors = [RED_COLOR if v > high_threshold else (GREEN_COLOR if v < low_threshold else BLUE_COLOR) for v in forecast_vol]
ax.bar(days, forecast_vol, color=bar_colors, alpha=0.8)
ax.axhline(high_threshold, color=RED_COLOR,   linewidth=1, linestyle='--', label=f'High vol threshold ({high_threshold:.1f}%)')
ax.axhline(low_threshold,  color=GREEN_COLOR, linewidth=1, linestyle='--', label=f'Low vol threshold ({low_threshold:.1f}%)')
ax.set_title('GARCH 5-Day Volatility Forecast (annualized %)', color=GOLD_COLOR, fontsize=12)
ax.set_ylabel('Forecast Vol %')
ax.legend(fontsize=9)
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()

# ── CURRENT READING ───────────────────────────────────────────
latest      = df.iloc[-1]
current_vol = latest['GARCH_Vol']
regime      = latest['Vol_Regime']
size_mult   = latest['Size_Multiplier']
next_day_vol = forecast_vol[0]

print()
print('=' * 60)
print('  GARCH VOLATILITY REGIME — CURRENT READING')
print('=' * 60)
print(f'  Date:                  {df.index[-1].date()}')
print(f'  Gold Price:            ${latest["Gold"]:,.0f}')
print(f'  Current GARCH Vol:     {current_vol:.1f}% annualized')
print(f'  Historical Vol (20d):  {latest["Vol20"]:.1f}% annualized')
print(f'  Vol Regime:            {regime}')
print(f'  Size Multiplier:       {size_mult}x your normal size')
print()
print(f'  5-Day Vol Forecast:')
for i, v in enumerate(forecast_vol):
    tag = 'HIGH' if v > high_threshold else ('LOW' if v < low_threshold else 'NORMAL')
    print(f'    Day {i+1}: {v:.1f}%  [{tag}]')
print()
if regime == 'LOW':
    print('  ACTION: LOW VOL TRENDING — Size up to 1.5x normal')
    print('  Gold is in quiet trending mode. This is when trends extend.')
    print('  Widen stops slightly — less noise means less need for tight stops.')
elif regime == 'HIGH':
    print('  ACTION: HIGH VOL — Cut size to 0.5x normal')
    print('  Choppy dangerous conditions. Whipsaws will stop you out.')
    print('  Tighten stops. Take profits faster. Wait for vol to compress.')
else:
    print('  ACTION: NORMAL VOL — Trade standard size (1.0x)')
    print('  Normal conditions. Follow MRM + HMM regime for direction.')
print('=' * 60)


_________________________________________
¶

The "Fair Value" Model
¶
This model uses real yields (TIPs + 10Y yield) to estimate what gold should be worth mathematically. It tracks how gold's sensitivity to real yields (the "beta") changes over time.
Current Reading: FAIRLY VALUED at $5,068 (actual: $5,062)


Gold is only -0.1% below fair value — essentially at fair value
The Z-score is -0.33σ, well within the ±1.5σ bands that would trigger a mean-reversion trade
Dynamic beta (3.60) is very close to static beta (3.57) — the relationship between gold and real yields is stable


Implication: No contrarian edge here. Gold isn't stretched in either direction relative to yields, so you don't have a fundamental reason to fade or chase. Follow the directional models (MRM + HMM) instead.

In [ ]:
# ============================================================
# KALMAN FILTER — Dynamic Gold / Real Yield Relationship
# Tracks the SHIFTING relationship between gold and real yields
# When gold disconnects from fair value = mean reversion trade
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pykalman import KalmanFilter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a1a'
plt.rcParams['axes.edgecolor']   = '#333333'
plt.rcParams['axes.labelcolor']  = '#cccccc'
plt.rcParams['xtick.color']      = '#888888'
plt.rcParams['ytick.color']      = '#888888'
plt.rcParams['text.color']       = '#cccccc'
plt.rcParams['grid.color']       = '#2a2a2a'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['font.size']        = 11

GOLD_COLOR   = '#FFD700'
GREEN_COLOR  = '#00C896'
RED_COLOR    = '#FF4D4D'
BLUE_COLOR   = '#4D9FFF'
ORANGE_COLOR = '#FF8C42'

# ── DOWNLOAD DATA ─────────────────────────────────────────────
print('Downloading data...')

def get(ticker, period='5y'):
    s = yf.download(ticker, period=period, auto_adjust=True, progress=False)['Close'].squeeze()
    if hasattr(s, 'columns') and isinstance(s.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
        s.columns = s.columns.get_level_values(0)
    return s.dropna()

gold = get('GC=F')
tip  = get('TIP')   # TIPS ETF = real yield proxy (price moves inverse to real yield)
tnx  = get('^TNX')  # 10Y nominal

df = pd.DataFrame({'Gold': gold, 'TIP': tip, 'TNX': tnx}).dropna()

# Real yield proxy: when TIP falls and TNX rises = real yields rising = bearish gold
# We use TIP price directly as our "real yield proxy" (inverse relationship)
df['RealYield_Proxy'] = -df['TIP']   # negate so it moves WITH real yields
df['LogGold'] = np.log(df['Gold'])
df['LogTIP']  = np.log(df['TIP'])

print(f'Data: {len(df)} rows  |  {df.index[0].date()} → {df.index[-1].date()}')

# ── STATIC OLS REGRESSION (baseline) ─────────────────────────
from numpy.linalg import lstsq
X_ols = np.column_stack([np.ones(len(df)), df['LogTIP'].values])
y_ols = df['LogGold'].values
beta_ols, _, _, _ = lstsq(X_ols, y_ols, rcond=None)
df['Gold_FairValue_Static'] = np.exp(beta_ols[0] + beta_ols[1] * df['LogTIP'])

print(f'Static OLS: Gold = exp({beta_ols[0]:.2f} + {beta_ols[1]:.2f} * log(TIP))')

# ── KALMAN FILTER — DYNAMIC RELATIONSHIP ─────────────────────
# State: [intercept, beta_TIP] — both allowed to change over time
# This captures the SHIFTING relationship as macro regimes change

print('Fitting Kalman Filter...')

n = len(df)
obs   = df['LogGold'].values
state = df['LogTIP'].values

# Build observation matrix: each row = [1, log(TIP_t)]
obs_matrices = np.column_stack([np.ones(n), state]).reshape(n, 1, 2)

kf = KalmanFilter(
    n_dim_state=2,
    n_dim_obs=1,
    initial_state_mean=[beta_ols[0], beta_ols[1]],
    initial_state_covariance=np.eye(2) * 1.0,
    transition_matrices=np.eye(2),
    observation_matrices=obs_matrices,
    transition_covariance=np.eye(2) * 0.001,   # how fast relationship shifts
    observation_covariance=np.array([[0.01]])
)

state_means, state_covs = kf.filter(obs)

df['KF_Intercept'] = state_means[:, 0]
df['KF_Beta_TIP']  = state_means[:, 1]

# Dynamic fair value
df['Gold_FairValue_KF'] = np.exp(df['KF_Intercept'] + df['KF_Beta_TIP'] * df['LogTIP'])

# Spread = actual gold minus fair value
df['Spread']    = df['Gold'] - df['Gold_FairValue_KF']
df['Spread_Z']  = (df['Spread'] - df['Spread'].rolling(60).mean()) / (df['Spread'].rolling(60).std() + 1e-8)

# Signal: gold too far above fair value = overvalued = fade
# Gold too far below fair value = undervalued = buy
df['KF_Signal'] = 0
df.loc[df['Spread_Z'] >  1.5, 'KF_Signal'] = -1  # overvalued vs real yields
df.loc[df['Spread_Z'] < -1.5, 'KF_Signal'] = +1  # undervalued vs real yields

# ── PLOT ──────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True)
fig.suptitle('KALMAN FILTER — Dynamic Gold / Real Yield Fair Value', fontsize=15, color=GOLD_COLOR, y=0.99)

# Panel 1: Gold vs fair values
ax1 = axes[0]
ax1.plot(df.index, df['Gold'],               color=GOLD_COLOR,   linewidth=1.5,          label='Gold Actual')
ax1.plot(df.index, df['Gold_FairValue_KF'],  color=GREEN_COLOR,  linewidth=1.2,          label='Kalman Fair Value (dynamic)')
ax1.plot(df.index, df['Gold_FairValue_Static'], color=GRAY_COLOR, linewidth=1, linestyle='--', alpha=0.6, label='Static OLS Fair Value')
ax1.fill_between(df.index, df['Gold'], df['Gold_FairValue_KF'],
                 where=(df['Gold'] > df['Gold_FairValue_KF']),
                 alpha=0.15, color=RED_COLOR, label='Overvalued vs yields')
ax1.fill_between(df.index, df['Gold'], df['Gold_FairValue_KF'],
                 where=(df['Gold'] < df['Gold_FairValue_KF']),
                 alpha=0.15, color=GREEN_COLOR, label='Undervalued vs yields')
ax1.set_title('Gold Actual vs Kalman Fair Value — Divergence = Mean Reversion Trade', fontsize=10)
ax1.set_ylabel('Price (USD)')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True)

# Panel 2: Dynamic beta (relationship strength)
ax2 = axes[1]
ax2.plot(df.index, df['KF_Beta_TIP'], color=BLUE_COLOR, linewidth=1.5, label='Dynamic Beta (TIP sensitivity)')
ax2.axhline(beta_ols[1], color=GRAY_COLOR, linewidth=1, linestyle='--', label=f'Static Beta ({beta_ols[1]:.2f})')
ax2.axhline(0, color='white', linewidth=0.5, linestyle='--')
ax2.set_title('Dynamic Beta — How Gold Responds to Real Yields (shifts over time)', fontsize=10)
ax2.set_ylabel('Beta')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True)

# Panel 3: Spread Z-score
ax3 = axes[2]
ax3.plot(df.index, df['Spread_Z'], color=ORANGE_COLOR, linewidth=1.2, label='Spread Z-Score')
ax3.axhline( 1.5, color=RED_COLOR,   linewidth=1.2, linestyle='--', label='+1.5σ Overvalued (SELL)')
ax3.axhline(-1.5, color=GREEN_COLOR, linewidth=1.2, linestyle='--', label='-1.5σ Undervalued (BUY)')
ax3.axhline( 0,   color='white',     linewidth=0.5, linestyle='--')
ax3.fill_between(df.index, df['Spread_Z'],  1.5,
                 where=(df['Spread_Z'] >  1.5), color=RED_COLOR,   alpha=0.3)
ax3.fill_between(df.index, df['Spread_Z'], -1.5,
                 where=(df['Spread_Z'] < -1.5), color=GREEN_COLOR, alpha=0.3)
ax3.set_title('Gold vs Fair Value Spread Z-Score — Extremes = Mean Reversion', fontsize=10)
ax3.set_ylabel('Z-Score (σ)')
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(True)

# Panel 4: TIP vs TNX
ax4 = axes[3]
ax4.plot(df.index, df['TIP'], color=BLUE_COLOR, linewidth=1.2, label='TIP ETF (TIPS proxy)')
ax4r = ax4.twinx()
ax4r.plot(df.index, df['TNX'], color=RED_COLOR, linewidth=1, linestyle='--', alpha=0.7, label='10Y Yield %')
ax4.set_title('TIP ETF vs 10Y Yield — The two inputs driving fair value', fontsize=10)
ax4.set_ylabel('TIP Price', color=BLUE_COLOR)
ax4r.set_ylabel('10Y Yield %', color=RED_COLOR)
ax4.legend(loc='upper left', fontsize=9)
ax4r.legend(loc='upper right', fontsize=9)
ax4.grid(True)

plt.tight_layout()
plt.show()

# ── CURRENT READING ───────────────────────────────────────────
latest     = df.iloc[-1]
spread_z   = latest['Spread_Z']
fair_val   = latest['Gold_FairValue_KF']
actual     = latest['Gold']
gap_pct    = (actual - fair_val) / fair_val * 100
dyn_beta   = latest['KF_Beta_TIP']

print()
print('=' * 60)
print('  KALMAN FILTER — CURRENT READING')
print('=' * 60)
print(f'  Date:              {df.index[-1].date()}')
print(f'  Gold Actual:       ${actual:,.0f}')
print(f'  Gold Fair Value:   ${fair_val:,.0f}   (Kalman dynamic model)')
print(f'  Gap:               {gap_pct:+.1f}%  (+ = overvalued vs real yields)')
print(f'  Spread Z-Score:    {spread_z:+.2f}σ')
print(f'  Dynamic Beta:      {dyn_beta:.2f}  (static was {beta_ols[1]:.2f})')
print()
if spread_z > 1.5:
    print(f'  SIGNAL: OVERVALUED vs real yields by {gap_pct:.1f}%')
    print('  Gold is trading TOO HIGH relative to where yields say it should be.')
    print('  Mean reversion risk — fade rallies, tighten stops on longs.')
elif spread_z < -1.5:
    print(f'  SIGNAL: UNDERVALUED vs real yields by {abs(gap_pct):.1f}%')
    print('  Gold is trading TOO LOW relative to where yields say it should be.')
    print('  Mean reversion opportunity — look for long entries on dips.')
else:
    print(f'  SIGNAL: FAIRLY VALUED (Z = {spread_z:+.2f}σ)')
    print('  Gold is trading in line with real yield model.')
    print('  No mean reversion edge — follow MRM + HMM for direction.')
print('=' * 60)


____________________________________________________________________
¶

The "Probability Distribution" Model
¶
This runs 10,000 random simulations of where gold could be in 20 trading days (~1 month), based on current price ($5,062) and current volatility.
Key numbers:
ScenarioPriceBase case (median)$5,347 (+5.6%)Bull case (95th pct)$6,366 (+25.8%)Bear case (5th pct)$4,482 (-11.5%)Probability higher69.4%
The distribution is skewed to the upside — the median outcome is already above current price, and the bull tail is much larger than the bear tail. This reflects the current bullish trend baked into the drift assumption.

In [ ]:
# ============================================================
# MONTE CARLO SIMULATION FOR GOLD
# Runs 10,000 price paths forward
# Gives probability-weighted targets — not just one forecast
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a1a'
plt.rcParams['axes.edgecolor']   = '#333333'
plt.rcParams['axes.labelcolor']  = '#cccccc'
plt.rcParams['xtick.color']      = '#888888'
plt.rcParams['ytick.color']      = '#888888'
plt.rcParams['text.color']       = '#cccccc'
plt.rcParams['grid.color']       = '#2a2a2a'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['font.size']        = 11

GOLD_COLOR   = '#FFD700'
GREEN_COLOR  = '#00C896'
RED_COLOR    = '#FF4D4D'
BLUE_COLOR   = '#4D9FFF'
ORANGE_COLOR = '#FF8C42'

# ── SETTINGS ──────────────────────────────────────────────────
N_SIMULATIONS = 10000   # number of price paths
HORIZON_DAYS  = 20      # trading days forward (4 weeks)
SEED          = 42

# ── DOWNLOAD DATA ─────────────────────────────────────────────
print('Downloading gold data...')
gold = yf.download('GC=F', period='2y', auto_adjust=True, progress=False)['Close'].squeeze()
if hasattr(gold, 'columns') and isinstance(gold.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gold.columns = gold.columns.get_level_values(0)
gold = gold.dropna()

returns = gold.pct_change().dropna()
S0      = gold.iloc[-1]   # current price

print(f'Current Gold Price: ${S0:,.0f}')
print(f'Running {N_SIMULATIONS:,} simulations over {HORIZON_DAYS} trading days...')

# ── CALIBRATE PARAMETERS FROM RECENT DATA ────────────────────
# Use last 60 days for recent vol (more responsive than full history)
recent_returns = returns.iloc[-60:]
mu_daily    = recent_returns.mean()
sigma_daily = recent_returns.std()

# Also compute full-history parameters
mu_full    = returns.mean()
sigma_full = returns.std()

# Fat tails: fit Student-t distribution
t_params = stats.t.fit(returns)
df_t, loc_t, scale_t = t_params

print(f'\nCalibration (last 60 days):')
print(f'  Daily mu:    {mu_daily*100:.4f}%')
print(f'  Daily sigma: {sigma_daily*100:.4f}%')
print(f'  Annual vol:  {sigma_daily*np.sqrt(252)*100:.1f}%')
print(f'  t-dist df:   {df_t:.2f}  (lower = fatter tails)')

# ── MONTE CARLO — GEOMETRIC BROWNIAN MOTION ───────────────────
np.random.seed(SEED)

# GBM with recent parameters
shocks_gbm = np.random.normal(
    mu_daily - 0.5 * sigma_daily**2,
    sigma_daily,
    (HORIZON_DAYS, N_SIMULATIONS)
)
log_returns  = np.cumsum(shocks_gbm, axis=0)
sim_paths    = S0 * np.exp(log_returns)

# Add row 0 = current price
sim_paths = np.vstack([np.full(N_SIMULATIONS, S0), sim_paths])

# ── MONTE CARLO — FAT-TAIL (Student-t) ────────────────────────
shocks_t = stats.t.rvs(df_t, loc=loc_t, scale=scale_t,
                        size=(HORIZON_DAYS, N_SIMULATIONS))
log_returns_t = np.cumsum(shocks_t, axis=0)
sim_paths_t   = S0 * np.exp(log_returns_t)
sim_paths_t   = np.vstack([np.full(N_SIMULATIONS, S0), sim_paths_t])

# ── COMPUTE PERCENTILE BANDS ──────────────────────────────────
pcts = [5, 10, 25, 50, 75, 90, 95]
bands     = {p: np.percentile(sim_paths,   p, axis=1) for p in pcts}
bands_t   = {p: np.percentile(sim_paths_t, p, axis=1) for p in pcts}

# Final day distribution
final_prices   = sim_paths[-1]
final_prices_t = sim_paths_t[-1]

# ── PROBABILITY TARGETS ───────────────────────────────────────
target_levels = {
    'Up 1%':  S0 * 1.01,
    'Up 2%':  S0 * 1.02,
    'Up 3%':  S0 * 1.03,
    'Up 5%':  S0 * 1.05,
    'Down 1%': S0 * 0.99,
    'Down 2%': S0 * 0.98,
    'Down 3%': S0 * 0.97,
    'Down 5%': S0 * 0.95,
}

# ── PLOT ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'MONTE CARLO SIMULATION — Gold  |  {N_SIMULATIONS:,} Paths  |  {HORIZON_DAYS}-Day Horizon',
             fontsize=14, color=GOLD_COLOR, y=0.99)

days_axis = np.arange(HORIZON_DAYS + 1)

# Top-left: GBM Fan chart
ax1 = axes[0, 0]
# Plot sample paths (faint)
for i in range(0, min(200, N_SIMULATIONS), 1):
    ax1.plot(days_axis, sim_paths[:, i], color=GOLD_COLOR, alpha=0.02, linewidth=0.5)
# Percentile bands
ax1.fill_between(days_axis, bands[5],  bands[95], color=BLUE_COLOR,  alpha=0.15, label='5-95th pct')
ax1.fill_between(days_axis, bands[10], bands[90], color=BLUE_COLOR,  alpha=0.20, label='10-90th pct')
ax1.fill_between(days_axis, bands[25], bands[75], color=GREEN_COLOR, alpha=0.25, label='25-75th pct')
ax1.plot(days_axis, bands[50], color='white', linewidth=2, label='Median path')
ax1.axhline(S0, color=GOLD_COLOR, linewidth=1, linestyle='--', label=f'Current ${S0:,.0f}')
ax1.set_title('GBM Simulation — Fan Chart (Normal distribution)', fontsize=10)
ax1.set_xlabel('Trading Days')
ax1.set_ylabel('Gold Price (USD)')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True)

# Top-right: Fat-tail simulation
ax2 = axes[0, 1]
for i in range(0, min(200, N_SIMULATIONS), 1):
    ax2.plot(days_axis, sim_paths_t[:, i], color=ORANGE_COLOR, alpha=0.02, linewidth=0.5)
ax2.fill_between(days_axis, bands_t[5],  bands_t[95], color=RED_COLOR,   alpha=0.15, label='5-95th pct')
ax2.fill_between(days_axis, bands_t[10], bands_t[90], color=RED_COLOR,   alpha=0.20, label='10-90th pct')
ax2.fill_between(days_axis, bands_t[25], bands_t[75], color=ORANGE_COLOR,alpha=0.25, label='25-75th pct')
ax2.plot(days_axis, bands_t[50], color='white', linewidth=2, label='Median path')
ax2.axhline(S0, color=GOLD_COLOR, linewidth=1, linestyle='--', label=f'Current ${S0:,.0f}')
ax2.set_title('Fat-Tail Simulation — Student-t (accounts for gold spikes)', fontsize=10)
ax2.set_xlabel('Trading Days')
ax2.set_ylabel('Gold Price (USD)')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True)

# Bottom-left: Final price distribution
ax3 = axes[1, 0]
ax3.hist(final_prices,   bins=80, color=BLUE_COLOR,   alpha=0.5, label='GBM (normal)', density=True)
ax3.hist(final_prices_t, bins=80, color=ORANGE_COLOR, alpha=0.5, label='Fat-tail (t-dist)', density=True)
ax3.axvline(S0,                    color=GOLD_COLOR,  linewidth=2, linestyle='--', label=f'Current ${S0:,.0f}')
ax3.axvline(np.median(final_prices), color='white',   linewidth=1.5, label=f'Median ${np.median(final_prices):,.0f}')
ax3.axvline(np.percentile(final_prices, 5),  color=RED_COLOR,   linewidth=1, linestyle=':', label=f'5th pct ${np.percentile(final_prices, 5):,.0f}')
ax3.axvline(np.percentile(final_prices, 95), color=GREEN_COLOR, linewidth=1, linestyle=':', label=f'95th pct ${np.percentile(final_prices, 95):,.0f}')
ax3.set_title(f'Final Price Distribution at Day {HORIZON_DAYS}', fontsize=10)
ax3.set_xlabel('Gold Price (USD)')
ax3.set_ylabel('Density')
ax3.legend(fontsize=8)
ax3.grid(True, axis='y')

# Bottom-right: Probability table
ax4 = axes[1, 1]
ax4.set_facecolor('#0f0f0f')
ax4.axis('off')

table_data = []
for label, level in target_levels.items():
    if 'Up' in label:
        p_gbm = np.mean(final_prices   >= level) * 100
        p_t   = np.mean(final_prices_t >= level) * 100
    else:
        p_gbm = np.mean(final_prices   <= level) * 100
        p_t   = np.mean(final_prices_t <= level) * 100
    table_data.append([label, f'${level:,.0f}', f'{p_gbm:.1f}%', f'{p_t:.1f}%'])

col_labels = ['Move', 'Price Target', 'P(GBM)', 'P(Fat-tail)']
table = ax4.table(
    cellText=table_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

for (row, col), cell in table.get_celld().items():
    cell.set_facecolor('#1a1a1a')
    cell.set_edgecolor('#333333')
    cell.set_text_props(color='#cccccc')
    if row == 0:
        cell.set_facecolor('#2a2a2a')
        cell.set_text_props(color=GOLD_COLOR, fontweight='bold')
    elif row > 0 and row <= 4:
        cell.set_facecolor('#0d2b1a')  # green tint for up moves
    elif row > 4:
        cell.set_facecolor('#2b0d0d')  # red tint for down moves

ax4.set_title('Probability Table — Chance of hitting each level', fontsize=10, color=GOLD_COLOR, pad=20)

plt.tight_layout()
plt.show()

# ── CURRENT READING ───────────────────────────────────────────
p5   = np.percentile(final_prices, 5)
p25  = np.percentile(final_prices, 25)
p50  = np.percentile(final_prices, 50)
p75  = np.percentile(final_prices, 75)
p95  = np.percentile(final_prices, 95)
p_up = np.mean(final_prices > S0) * 100

print()
print('=' * 60)
print(f'  MONTE CARLO — {HORIZON_DAYS}-DAY FORWARD DISTRIBUTION')
print('=' * 60)
print(f'  Current Gold Price:    ${S0:,.0f}')
print(f'  Simulation:            {N_SIMULATIONS:,} paths (GBM normal)')
print()
print(f'  Price Targets ({HORIZON_DAYS} trading days):')
print(f'    95th percentile:     ${p95:,.0f}  (bull scenario)')
print(f'    75th percentile:     ${p75:,.0f}')
print(f'    50th percentile:     ${p50:,.0f}  (median / base case)')
print(f'    25th percentile:     ${p25:,.0f}')
print(f'     5th percentile:     ${p5:,.0f}   (bear scenario)')
print()
print(f'  Probability gold is HIGHER in {HORIZON_DAYS} days: {p_up:.1f}%')
print(f'  Probability gold is LOWER  in {HORIZON_DAYS} days: {100-p_up:.1f}%')
print()
print(f'  Expected Range (25-75th):   ${p25:,.0f} — ${p75:,.0f}')
print(f'  Tail Risk Range (5-95th):   ${p5:,.0f} — ${p95:,.0f}')
print()
print(f'  Probability Table:')
for label, level in target_levels.items():
    if 'Up' in label:
        p = np.mean(final_prices >= level) * 100
        print(f'    P(gold >= ${level:,.0f})  [{label}]:  {p:.1f}%')
    else:
        p = np.mean(final_prices <= level) * 100
        print(f'    P(gold <= ${level:,.0f})  [{label}]:  {p:.1f}%')
print('=' * 60)


___________________________________________
¶

Correlation Monitor
¶
What it does:
Bar chart . Current 30-day correlations at a glance. Rolling charts . How correlations change over time. AlertsFlags when relationships break down. 30D vs 90D. Short vs longer term comparison

In [ ]:
# ── portable output dir (was hardcoded /Users/elena_nael/...) ─────────
from pathlib import Path as _Path
OUTDIR = _Path.cwd() / "outputs"; OUTDIR.mkdir(exist_ok=True)

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta

# ============================================================
# CORRELATION MONITOR — Gold vs Key Assets
# ============================================================

# Define assets
tickers = {
    'Gold': 'GC=F',
    'DXY': 'DX-Y.NYB',
    '10Y Yield': '^TNX',
    'S&P 500': '^GSPC',
    'Oil': 'CL=F',
    'Silver': 'SI=F',
    'TIP ETF': 'TIP'
}

# Download data
end = datetime.today()
start = end - timedelta(days=365)

print("Downloading data...")
data = yf.download(list(tickers.values()), start=start, end=end)['Close']
if hasattr(data, 'columns') and isinstance(data.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    data.columns = data.columns.get_level_values(0)
data.columns = list(tickers.keys())

# Calculate returns
returns = data.pct_change().dropna()

# Calculate rolling correlations with Gold (30-day)
window = 30
rolling_corr = pd.DataFrame()
for col in returns.columns:
    if col != 'Gold':
        rolling_corr[col] = returns['Gold'].rolling(window).corr(returns[col])

# Current correlations (last 30 days)
current_corr = returns.tail(30).corr()['Gold'].drop('Gold')

# 90-day correlations
corr_90d = returns.tail(90).corr()['Gold'].drop('Gold')

# ============================================================
# PLOT
# ============================================================

fig = plt.figure(figsize=(16, 14), facecolor='#0d0d0d')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.3)

fig.suptitle('CORRELATION MONITOR — Gold vs Key Assets',
             fontsize=16, fontweight='bold', color='gold', y=0.98)

# ── Plot 1: Current Correlation Bar Chart ──
ax1 = fig.add_subplot(gs[0, :])
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in current_corr.values]
bars = ax1.bar(current_corr.index, current_corr.values, color=colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax1.axhline(y=0, color='white', linewidth=0.5)
ax1.axhline(y=0.5, color='#2ecc71', linewidth=0.5, linestyle='--', alpha=0.5, label='+0.5 Strong Positive')
ax1.axhline(y=-0.5, color='#e74c3c', linewidth=0.5, linestyle='--', alpha=0.5, label='-0.5 Strong Negative')
ax1.set_facecolor('#0d0d0d')
ax1.set_title('Current 30-Day Correlations with Gold', color='white', fontsize=12)
ax1.set_ylabel('Correlation', color='white')
ax1.tick_params(colors='white')
ax1.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
ax1.spines['bottom'].set_color('#333333')
ax1.spines['left'].set_color('#333333')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for bar, val in zip(bars, current_corr.values):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
             f'{val:.2f}', ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')

# ── Plot 2-6: Rolling Correlations ──
assets = [col for col in rolling_corr.columns]
positions = [(1,0), (1,1), (2,0), (2,1)]
asset_colors = {
    'DXY': '#e74c3c',
    '10Y Yield': '#e67e22',
    'S&P 500': '#3498db',
    'Oil': '#9b59b6',
    'Silver': '#95a5a6',
    'TIP ETF': '#1abc9c'
}

for idx, asset in enumerate(assets[:4]):
    row, col = positions[idx]
    ax = fig.add_subplot(gs[row, col])
    color = asset_colors.get(asset, 'white')
    ax.plot(rolling_corr.index, rolling_corr[asset], color=color, linewidth=1.5, label=f'Gold vs {asset}')
    ax.axhline(y=0, color='white', linewidth=0.5, linestyle='-')
    ax.axhline(y=0.5, color='#2ecc71', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axhline(y=-0.5, color='#e74c3c', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.fill_between(rolling_corr.index, rolling_corr[asset], 0,
                    where=rolling_corr[asset] > 0, alpha=0.2, color='#2ecc71')
    ax.fill_between(rolling_corr.index, rolling_corr[asset], 0,
                    where=rolling_corr[asset] < 0, alpha=0.2, color='#e74c3c')
    ax.set_facecolor('#0d0d0d')
    ax.set_title(f'Gold vs {asset} (30d Rolling)', color='white', fontsize=10)
    ax.set_ylabel('Correlation', color='white')
    ax.tick_params(colors='white', labelsize=7)
    ax.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
    ax.spines['bottom'].set_color('#333333')
    ax.spines['left'].set_color('#333333')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(-1, 1)

plt.savefig(str(OUTDIR / 'correlation_monitor.png'), dpi=150, bbox_inches='tight',
            facecolor='#0d0d0d')
plt.show()

# ============================================================
# PRINT CURRENT READINGS
# ============================================================

print("\n" + "="*60)
print("CORRELATION MONITOR — CURRENT READINGS")
print("="*60)
print(f"  Date: {datetime.today().strftime('%Y-%m-%d')}")
print(f"  Gold Price: ${data['Gold'].iloc[-1]:,.0f}")
print(f"\n  30-Day Correlations with Gold:")
print(f"  {'Asset':<15} {'30D Corr':>10} {'90D Corr':>10} {'Signal'}")
print(f"  {'-'*50}")

for asset in current_corr.index:
    corr_30 = current_corr[asset]
    corr_90 = corr_90d[asset]
    
    if asset == 'DXY':
        signal = "⚠️ BROKEN" if corr_30 > -0.3 else "✅ NORMAL (inverse)"
    elif asset == '10Y Yield':
        signal = "⚠️ BROKEN" if corr_30 > 0.3 else "✅ NORMAL (inverse)"
    elif asset == 'Silver':
        signal = "✅ STRONG" if corr_30 > 0.5 else "⚠️ WEAK"
    elif asset == 'S&P 500':
        signal = "🔵 RISK-ON" if corr_30 > 0.3 else "🟡 NEUTRAL"
    else:
        signal = "✅ POSITIVE" if corr_30 > 0 else "❌ NEGATIVE"
    
    print(f"  {asset:<15} {corr_30:>10.2f} {corr_90:>10.2f}   {signal}")

print(f"\n  KEY ALERTS:")
dxy_corr = current_corr.get('DXY', 0)
yield_corr = current_corr.get('10Y Yield', 0)
silver_corr = current_corr.get('Silver', 0)

if dxy_corr > -0.3:
    print(f"  🚨 Gold-DXY correlation broken! ({dxy_corr:.2f}) — Models may be unreliable")
else:
    print(f"  ✅ Gold-DXY relationship normal ({dxy_corr:.2f})")

if yield_corr > 0.3:
    print(f"  🚨 Gold-Yield correlation broken! ({yield_corr:.2f}) — Unusual behavior")
else:
    print(f"  ✅ Gold-Yield relationship normal ({yield_corr:.2f})")

if silver_corr < 0.4:
    print(f"  ⚠️  Gold-Silver diverging ({silver_corr:.2f}) — Watch for mean reversion")
else:
    print(f"  ✅ Gold-Silver moving together ({silver_corr:.2f})")

print("="*60)


_____________________________
¶

In [ ]:
# ── portable output dir (was hardcoded /Users/elena_nael/...) ─────────
from pathlib import Path as _Path
OUTDIR = _Path.cwd() / "outputs"; OUTDIR.mkdir(exist_ok=True)

import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
from IPython.display import display, HTML

# ============================================================
# ECONOMIC CALENDAR — High Impact Events for Gold
# ============================================================

# Key events that move gold
GOLD_EVENTS = [
    'CPI', 'inflation', 'Fed', 'FOMC', 'Powell', 'interest rate',
    'NFP', 'nonfarm', 'GDP', 'PCE', 'jobs', 'unemployment',
    'PPI', 'retail sales', 'ISM', 'PMI', 'treasury', 'debt'
]

def get_calendar():
    """Fetch economic calendar from investing.com via pandas"""
    today = datetime.today()
    week_end = today + timedelta(days=7)
    
    # Use investpy or manual key dates
    events = []
    
    # Try fetching from public API
    try:
        url = "https://nfs.faireconomy.media/ff_calendar_thisweek.json"
        response = requests.get(url, timeout=10)
        data = response.json()
        
        for event in data:
            # Filter for USD events and high impact
            if event.get('country', '').upper() == 'USD':
                events.append({
                    'date': event.get('date', ''),
                    'time': event.get('time', ''),
                    'event': event.get('title', ''),
                    'impact': event.get('impact', ''),
                    'forecast': event.get('forecast', 'N/A'),
                    'previous': event.get('previous', 'N/A'),
                })
    except Exception as e:
        print(f"Could not fetch live calendar: {e}")
        print("Using manual key dates instead...")
        events = get_manual_events()
    
    return pd.DataFrame(events)

def get_manual_events():
    """Fallback — manually curated upcoming gold-relevant events"""
    today = datetime.today()
    events = [
        {'date': today.strftime('%Y-%m-%d'), 'time': '14:00', 
         'event': 'Check Fed Calendar at federalreserve.gov', 
         'impact': 'HIGH', 'forecast': '-', 'previous': '-'},
        {'date': today.strftime('%Y-%m-%d'), 'time': '08:30',
         'event': 'Check BLS for CPI/NFP at bls.gov',
         'impact': 'HIGH', 'forecast': '-', 'previous': '-'},
    ]
    return events

def classify_gold_impact(event_name):
    """Classify how much an event impacts gold"""
    event_lower = str(event_name).lower()
    
    high_impact = ['cpi', 'inflation', 'fomc', 'fed', 'powell', 'rate decision',
                   'nonfarm', 'nfp', 'pce', 'gdp']
    medium_impact = ['ppi', 'retail', 'ism', 'pmi', 'unemployment', 'jobs',
                     'treasury', 'auction', 'debt']
    
    for term in high_impact:
        if term in event_lower:
            return '🔴 HIGH'
    for term in medium_impact:
        if term in event_lower:
            return '🟡 MEDIUM'
    return '🟢 LOW'

# ============================================================
# FETCH AND PROCESS
# ============================================================

print("Fetching economic calendar...")
df = get_calendar()

if not df.empty:
    # Add gold impact classification
    df['gold_impact'] = df['event'].apply(classify_gold_impact)
    
    # Filter for high and medium impact only
    high_medium = df[df['impact'].isin(['High', 'Medium', 'HIGH', 'MEDIUM', 'high', 'medium'])]
    
    if high_medium.empty:
        high_medium = df  # show all if filter returns empty

    # ============================================================
    # DISPLAY AS STYLED TABLE
    # ============================================================
    
    print("\n" + "="*60)
    print("ECONOMIC CALENDAR — THIS WEEK")
    print("="*60)
    print(f"  Date pulled: {datetime.today().strftime('%Y-%m-%d %H:%M')}")
    print(f"  Showing: USD High & Medium Impact Events")
    print(f"  Total events found: {len(df)}")
    print("="*60)
    
    # Color code by impact
    def color_impact(impact):
        impact = str(impact).upper()
        if 'HIGH' in impact:
            return '🔴 HIGH'
        elif 'MEDIUM' in impact or 'MED' in impact:
            return '🟡 MEDIUM'
        else:
            return '🟢 LOW'
    
    display_df = high_medium[['date', 'time', 'event', 'impact', 'forecast', 'previous']].copy()
    display_df['impact'] = display_df['impact'].apply(color_impact)
    display_df['gold_impact'] = display_df['event'].apply(classify_gold_impact)
    display_df.columns = ['Date', 'Time', 'Event', 'Impact', 'Forecast', 'Previous', 'Gold Impact']
    
    # Print table
    for _, row in display_df.iterrows():
        print(f"\n  📅 {row['Date']} {row['Time']}")
        print(f"     Event:       {row['Event']}")
        print(f"     Impact:      {row['Impact']}")
        print(f"     Gold Impact: {row['Gold Impact']}")
        print(f"     Forecast:    {row['Forecast']}")
        print(f"     Previous:    {row['Previous']}")
    
    # ============================================================
    # PLOT CALENDAR
    # ============================================================
    
    fig, ax = plt.subplots(figsize=(16, max(4, len(display_df) * 0.8)), facecolor='#0d0d0d')
    ax.set_facecolor('#0d0d0d')
    ax.axis('off')
    
    fig.suptitle('ECONOMIC CALENDAR — This Week (USD Events)',
                 fontsize=14, fontweight='bold', color='gold', y=1.02)
    
    # Table data
    table_data = []
    for _, row in display_df.iterrows():
        table_data.append([
            row['Date'],
            row['Time'],
            row['Event'][:50],  # truncate long names
            row['Impact'],
            row['Gold Impact'],
            str(row['Forecast']),
            str(row['Previous'])
        ])
    
    if table_data:
        table = ax.table(
            cellText=table_data,
            colLabels=['Date', 'Time', 'Event', 'Impact', 'Gold Impact', 'Forecast', 'Previous'],
            cellLoc='left',
            loc='center',
            bbox=[0, 0, 1, 1]
        )
        
        # Style the table
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        
        for (row, col), cell in table.get_celld().items():
            cell.set_facecolor('#1a1a1a')
            cell.set_text_props(color='white')
            cell.set_edgecolor('#333333')
            
            if row == 0:  # header
                cell.set_facecolor('#2d2d2d')
                cell.set_text_props(color='gold', fontweight='bold')
            
            # Color code impact column
            if col == 3 and row > 0:
                text = cell.get_text().get_text()
                if 'HIGH' in text.upper():
                    cell.set_facecolor('#4a1a1a')
                elif 'MEDIUM' in text.upper():
                    cell.set_facecolor('#4a3a1a')
    
    plt.tight_layout()
    plt.savefig(str(OUTDIR / 'economic_calendar.png'), dpi=150, bbox_inches='tight',
                facecolor='#0d0d0d')
    plt.show()

# ============================================================
# TRADING ALERTS
# ============================================================

print("\n" + "="*60)
print("TRADING ALERTS FOR TODAY")
print("="*60)

today_str = datetime.today().strftime('%Y-%m-%d')
today_events = df[df['date'].str.contains(today_str[:10], na=False)] if not df.empty else pd.DataFrame()

if today_events.empty:
    print("  ✅ No major events today — models are reliable")
    print("  Safe to trade based on MRM + HMM signals")
else:
    high_today = today_events[today_events['impact'].str.upper().isin(['HIGH', 'MEDIUM'])]
    if not high_today.empty:
        print(f"  🚨 {len(high_today)} HIGH/MEDIUM impact events TODAY!")
        print("  ⚠️  Consider reducing position size before events")
        print("  ⚠️  Wait for event to pass before entering new trades")
        for _, event in high_today.iterrows():
            print(f"\n     ⏰ {event.get('time', 'TBD')} — {event.get('event', 'Unknown')}")
    else:
        print("  ✅ No high impact events today")
        print("  Safe to trade based on model signals")

print("="*60)
print("\n  📌 Always verify at: https://www.forexfactory.com/calendar")
print("  📌 Fed calendar:     https://www.federalreserve.gov/monetarypolicy/fomccalendar.htm")
print("="*60)


In [ ]:
# ── portable output dir (was hardcoded /Users/elena_nael/...) ─────────
from pathlib import Path as _Path
OUTDIR = _Path.cwd() / "outputs"; OUTDIR.mkdir(exist_ok=True)

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta

# ============================================================
# KEY LEVELS DASHBOARD — Gold Support & Resistance
# ============================================================

# Download gold data
print("Downloading Gold data...")
gold = yf.download('GC=F', period='6mo', interval='1d')
if hasattr(gold, 'columns') and isinstance(gold.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gold.columns = gold.columns.get_level_values(0)
gold.columns = [col[0] if isinstance(col, tuple) else col for col in gold.columns]

current_price = float(np.asarray(gold['Close'])[-1])
prev_close = float(np.asarray(gold['Close'])[-2])
prev_high = float(np.asarray(gold['High'])[-2])
prev_low = float(np.asarray(gold['Low'])[-2])
daily_change = ((current_price - prev_close) / prev_close) * 100

# ============================================================
# CALCULATE KEY LEVELS
# ============================================================

# 1. Moving Averages
ma20 = float(gold['Close'].rolling(20).mean().iloc[-1])
ma50 = float(gold['Close'].rolling(50).mean().iloc[-1])
ma100 = float(gold['Close'].rolling(100).mean().iloc[-1])
ma200 = float(gold['Close'].rolling(200).mean().iloc[-1])

# 2. Weekly Pivot Points
week_high = float(gold['High'].tail(5).max())
week_low = float(gold['Low'].tail(5).min())
week_close = float(gold['Close'].tail(5).iloc[-1])

pivot = (week_high + week_low + week_close) / 3
r1 = (2 * pivot) - week_low
r2 = pivot + (week_high - week_low)
r3 = week_high + 2 * (pivot - week_low)
s1 = (2 * pivot) - week_high
s2 = pivot - (week_high - week_low)
s3 = week_low - 2 * (week_high - pivot)

# 3. Monthly Pivot Points
month_high = float(gold['High'].tail(22).max())
month_low = float(gold['Low'].tail(22).min())
month_close = float(gold['Close'].tail(22).iloc[-1])

m_pivot = (month_high + month_low + month_close) / 3
m_r1 = (2 * m_pivot) - month_low
m_r2 = m_pivot + (month_high - month_low)
m_s1 = (2 * m_pivot) - month_high
m_s2 = m_pivot - (month_high - month_low)

# 4. Recent Swing Highs and Lows (last 20 days)
recent = gold.tail(20)
swing_high = float(recent['High'].max())
swing_low = float(recent['Low'].min())

# 5. ATR (Average True Range) — for stop loss sizing
high_low = gold['High'] - gold['Low']
high_close = abs(gold['High'] - gold['Close'].shift())
low_close = abs(gold['Low'] - gold['Close'].shift())
true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
atr14 = float(true_range.rolling(14).mean().iloc[-1])
atr20 = float(true_range.rolling(20).mean().iloc[-1])

# 6. Bollinger Bands
bb_middle = float(gold['Close'].rolling(20).mean().iloc[-1])
bb_std = float(gold['Close'].rolling(20).std().iloc[-1])
bb_upper = bb_middle + (2 * bb_std)
bb_lower = bb_middle - (2 * bb_std)

# 7. 52-week High/Low
high_52w = float(gold['High'].tail(252).max())
low_52w = float(gold['Low'].tail(252).min())

# ============================================================
# CLASSIFY CURRENT PRICE POSITION
# ============================================================

def classify_level(price, level, tolerance=0.005):
    """Check if price is near a level"""
    pct_diff = abs(price - level) / level
    if pct_diff < tolerance:
        return '⚡ AT LEVEL'
    elif price > level:
        return '✅ ABOVE'
    else:
        return '❌ BELOW'

def distance_pct(price, level):
    return ((level - price) / price) * 100

# ============================================================
# PLOT
# ============================================================

fig = plt.figure(figsize=(16, 18), facecolor='#0d0d0d')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.3)

fig.suptitle(f'KEY LEVELS DASHBOARD — Gold @ ${current_price:,.0f}',
             fontsize=16, fontweight='bold', color='gold', y=0.98)

# ── Plot 1: Price Chart with All Levels ──
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor('#0d0d0d')

# Candlestick-style (using close prices)
dates = gold.index[-60:]
closes = gold['Close'].tail(60)
highs = gold['High'].tail(60)
lows = gold['Low'].tail(60)

ax1.plot(dates, closes, color='gold', linewidth=1.5, label='Gold', zorder=5)
ax1.fill_between(dates, lows, highs, alpha=0.1, color='gold')

# Moving averages
ax1.plot(dates, gold['Close'].rolling(20).mean().tail(60),
         color='#3498db', linewidth=1, linestyle='--', label='MA20', alpha=0.8)
ax1.plot(dates, gold['Close'].rolling(50).mean().tail(60),
         color='#e67e22', linewidth=1, linestyle='--', label='MA50', alpha=0.8)

# Key levels
ax1.axhline(y=pivot, color='#9b59b6', linewidth=1, linestyle='-', alpha=0.7, label=f'Weekly Pivot {pivot:,.0f}')
ax1.axhline(y=r1, color='#e74c3c', linewidth=0.8, linestyle='--', alpha=0.6, label=f'R1 {r1:,.0f}')
ax1.axhline(y=r2, color='#e74c3c', linewidth=0.8, linestyle=':', alpha=0.6, label=f'R2 {r2:,.0f}')
ax1.axhline(y=s1, color='#2ecc71', linewidth=0.8, linestyle='--', alpha=0.6, label=f'S1 {s1:,.0f}')
ax1.axhline(y=s2, color='#2ecc71', linewidth=0.8, linestyle=':', alpha=0.6, label=f'S2 {s2:,.0f}')
ax1.axhline(y=bb_upper, color='#1abc9c', linewidth=0.8, linestyle='--', alpha=0.5, label=f'BB Upper {bb_upper:,.0f}')
ax1.axhline(y=bb_lower, color='#1abc9c', linewidth=0.8, linestyle='--', alpha=0.5, label=f'BB Lower {bb_lower:,.0f}')

ax1.set_title('Gold Price — Last 60 Days with Key Levels', color='white', fontsize=12)
ax1.set_ylabel('Price (USD)', color='white')
ax1.tick_params(colors='white', labelsize=8)
ax1.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=7, ncol=4, loc='upper left')
ax1.spines['bottom'].set_color('#333333')
ax1.spines['left'].set_color('#333333')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── Plot 2: Distance from Key Levels (Bar Chart) ──
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor('#0d0d0d')

levels = {
    'MA20': ma20,
    'MA50': ma50,
    'MA200': ma200,
    'W.Pivot': pivot,
    'W.R1': r1,
    'W.R2': r2,
    'W.S1': s1,
    'W.S2': s2,
    'BB Upper': bb_upper,
    'BB Lower': bb_lower,
}

distances = {k: distance_pct(current_price, v) for k, v in levels.items()}
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in distances.values()]

bars = ax2.barh(list(distances.keys()), list(distances.values()),
                color=colors, alpha=0.8, edgecolor='white', linewidth=0.3)
ax2.axvline(x=0, color='white', linewidth=1)
ax2.set_title('Distance from Key Levels (%)', color='white', fontsize=10)
ax2.set_xlabel('% from Current Price', color='white')
ax2.tick_params(colors='white', labelsize=8)
ax2.spines['bottom'].set_color('#333333')
ax2.spines['left'].set_color('#333333')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

for bar, val in zip(bars, distances.values()):
    ax2.text(val + (0.05 if val >= 0 else -0.05), bar.get_y() + bar.get_height()/2,
             f'{val:+.1f}%', va='center', ha='left' if val >= 0 else 'right',
             color='white', fontsize=7)

# ── Plot 3: ATR Stop Loss Guide ──
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor('#0d0d0d')

stop_levels = {
    '0.5x ATR14': current_price - (0.5 * atr14),
    '1x ATR14': current_price - atr14,
    '1.5x ATR14': current_price - (1.5 * atr14),
    '2x ATR14': current_price - (2 * atr14),
    '0.5x ATR20': current_price - (0.5 * atr20),
    '1x ATR20': current_price - atr20,
    '1.5x ATR20': current_price - (1.5 * atr20),
    '2x ATR20': current_price - (2 * atr20),
}

stop_colors = ['#e74c3c', '#e67e22', '#f39c12', '#f1c40f',
               '#c0392b', '#d35400', '#e67e22', '#f39c12']

bars3 = ax3.barh(list(stop_levels.keys()), list(stop_levels.values()),
                 color=stop_colors, alpha=0.8, edgecolor='white', linewidth=0.3)
ax3.axvline(x=current_price, color='gold', linewidth=1.5, label=f'Current ${current_price:,.0f}')
ax3.set_title('ATR-Based Stop Loss Levels', color='white', fontsize=10)
ax3.set_xlabel('Price (USD)', color='white')
ax3.tick_params(colors='white', labelsize=8)
ax3.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
ax3.spines['bottom'].set_color('#333333')
ax3.spines['left'].set_color('#333333')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

for bar, val in zip(bars3, stop_levels.values()):
    ax3.text(val - 10, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', ha='right',
             color='white', fontsize=7)

# ── Plot 4: Bollinger Band Position ──
ax4 = fig.add_subplot(gs[2, :])
ax4.set_facecolor('#0d0d0d')

dates60 = gold.index[-60:]
bb_mid = gold['Close'].rolling(20).mean().tail(60)
bb_up = bb_mid + 2 * gold['Close'].rolling(20).std().tail(60)
bb_dn = bb_mid - 2 * gold['Close'].rolling(20).std().tail(60)

ax4.plot(dates60, gold['Close'].tail(60), color='gold', linewidth=1.5, label='Gold', zorder=5)
ax4.plot(dates60, bb_mid, color='white', linewidth=0.8, linestyle='--', label='BB Middle (MA20)', alpha=0.7)
ax4.plot(dates60, bb_up, color='#1abc9c', linewidth=0.8, label=f'BB Upper ${float(bb_upper):,.0f}', alpha=0.8)
ax4.plot(dates60, bb_dn, color='#1abc9c', linewidth=0.8, label=f'BB Lower ${float(bb_lower):,.0f}', alpha=0.8)
ax4.fill_between(dates60, bb_dn, bb_up, alpha=0.05, color='#1abc9c')

ax4.set_title('Bollinger Bands (20,2) — Overbought/Oversold Guide', color='white', fontsize=10)
ax4.set_ylabel('Price (USD)', color='white')
ax4.tick_params(colors='white', labelsize=8)
ax4.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8, ncol=4)
ax4.spines['bottom'].set_color('#333333')
ax4.spines['left'].set_color('#333333')
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

plt.savefig(str(OUTDIR / 'key_levels_dashboard.png'), dpi=150, bbox_inches='tight',
            facecolor='#0d0d0d')
plt.show()

# ============================================================
# PRINT CURRENT READINGS
# ============================================================

print("\n" + "="*60)
print("KEY LEVELS DASHBOARD — CURRENT READINGS")
print("="*60)
print(f"  Date:          {datetime.today().strftime('%Y-%m-%d')}")
print(f"  Gold Price:    ${current_price:,.0f}")
print(f"  Daily Change:  {daily_change:+.2f}%")
print(f"  Prev High:     ${prev_high:,.0f}")
print(f"  Prev Low:      ${prev_low:,.0f}")

print(f"\n  MOVING AVERAGES:")
print(f"  {'Level':<15} {'Price':>10} {'Distance':>10} {'Status'}")
print(f"  {'-'*50}")
for name, level in [('MA20', ma20), ('MA50', ma50), ('MA100', ma100), ('MA200', ma200)]:
    dist = distance_pct(current_price, level)
    status = classify_level(current_price, level)
    print(f"  {name:<15} ${level:>9,.0f} {dist:>+9.1f}%  {status}")

print(f"\n  WEEKLY PIVOT POINTS:")
print(f"  {'Level':<15} {'Price':>10} {'Distance':>10} {'Status'}")
print(f"  {'-'*50}")
for name, level in [('R3', r3), ('R2', r2), ('R1', r1),
                     ('Pivot', pivot), ('S1', s1), ('S2', s2), ('S3', s3)]:
    dist = distance_pct(current_price, level)
    status = classify_level(current_price, level)
    print(f"  {name:<15} ${level:>9,.0f} {dist:>+9.1f}%  {status}")

print(f"\n  BOLLINGER BANDS (20,2):")
print(f"  {'Level':<15} {'Price':>10} {'Distance':>10} {'Status'}")
print(f"  {'-'*50}")
for name, level in [('BB Upper', bb_upper), ('BB Middle', bb_middle), ('BB Lower', bb_lower)]:
    dist = distance_pct(current_price, level)
    status = classify_level(current_price, level)
    print(f"  {name:<15} ${level:>9,.0f} {dist:>+9.1f}%  {status}")

print(f"\n  ATR STOP LOSS GUIDE:")
print(f"  ATR(14): ${atr14:,.0f}  |  ATR(20): ${atr20:,.0f}")
print(f"  {'Stop Type':<20} {'Price':>10} {'Risk $':>10} {'Risk %':>8}")
print(f"  {'-'*55}")
for mult in [0.5, 1.0, 1.5, 2.0]:
    stop = current_price - (mult * atr14)
    risk_dollar = current_price - stop
    risk_pct = (risk_dollar / current_price) * 100
    print(f"  {mult}x ATR14{'':<12} ${stop:>9,.0f} ${risk_dollar:>9,.0f} {risk_pct:>7.1f}%")

print(f"\n  52-WEEK RANGE:")
print(f"  High: ${high_52w:,.0f}  |  Low: ${low_52w:,.0f}")
pct_from_high = ((current_price - high_52w) / high_52w) * 100
pct_from_low = ((current_price - low_52w) / low_52w) * 100
print(f"  From 52W High: {pct_from_high:+.1f}%  |  From 52W Low: {pct_from_low:+.1f}%")

print(f"\n  KEY ALERTS:")
if current_price > bb_upper:
    print(f"  🚨 Gold ABOVE Bollinger Upper Band — Overbought, consider taking profits")
elif current_price < bb_lower:
    print(f"  🚨 Gold BELOW Bollinger Lower Band — Oversold, potential bounce")
else:
    bb_pct = (current_price - float(bb_lower)) / (float(bb_upper) - float(bb_lower)) * 100
    print(f"  ✅ Gold inside Bollinger Bands ({bb_pct:.0f}% from bottom)")

if current_price > ma20:
    print(f"  ✅ Above MA20 — Short term trend is UP")
else:
    print(f"  ❌ Below MA20 — Short term trend is DOWN")

if current_price > ma50:
    print(f"  ✅ Above MA50 — Medium term trend is UP")
else:
    print(f"  ❌ Below MA50 — Medium term trend is DOWN")

if abs(distance_pct(current_price, r1)) < 0.5:
    print(f"  ⚡ Price near R1 resistance (${r1:,.0f}) — Watch for rejection")
if abs(distance_pct(current_price, s1)) < 0.5:
    print(f"  ⚡ Price near S1 support (${s1:,.0f}) — Watch for bounce")

print("="*60)


In [ ]:
# ── portable output dir (was hardcoded /Users/elena_nael/...) ─────────
from pathlib import Path as _Path
OUTDIR = _Path.cwd() / "outputs"; OUTDIR.mkdir(exist_ok=True)

import requests
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf
from io import BytesIO, StringIO
from datetime import datetime

headers_http = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
}

# ============================================================
# FETCH COT DATA — GOLD COMMODITY EXCHANGE INC.
# ============================================================

def fetch_cot_historical_xls(year):
    """Fetch historical COT from Excel zip files"""
    try:
        url = f"https://www.cftc.gov/files/dea/history/fut_disagg_xls_{year}.zip"
        resp = requests.get(url, headers=headers_http, timeout=60)
        z = zipfile.ZipFile(BytesIO(resp.content))
        with z.open(z.namelist()[0]) as f:
            df = pd.read_excel(f, header=0)
        gold = df[df['Market_and_Exchange_Names'].str.strip() ==
                  'GOLD - COMMODITY EXCHANGE INC.'].copy()
        print(f"  ✅ {year}: {len(gold)} weeks found (Excel)")
        return gold
    except Exception as e:
        print(f"  ❌ {year} Excel failed: {e}")
        return None

def fetch_cot_current():
    """Fetch current week from plain text file"""
    try:
        url = "https://www.cftc.gov/dea/newcot/deafut.txt"
        resp = requests.get(url, headers=headers_http, timeout=30)
        df = pd.read_csv(StringIO(resp.text), header=None, low_memory=False)
        gold = df[df[0].astype(str).str.strip() ==
                  'GOLD - COMMODITY EXCHANGE INC.'].copy()

        if len(gold) > 0:
            # Map fields to named columns to match Excel format
            gold_named = pd.DataFrame()
            gold_named['Market_and_Exchange_Names'] = gold[0].values
            gold_named['Report_Date_as_MM_DD_YYYY'] = pd.to_datetime(
                gold[2].values, errors='coerce')
            gold_named['Open_Interest_All'] = pd.to_numeric(
                gold[7].values, errors='coerce')
            # In disagg format: col 8,9 = Prod/Merc, 12,13 = Swap,
            # 16,17 = Managed Money (speculators), 20,21 = Other
            gold_named['M_Money_Positions_Long_ALL'] = pd.to_numeric(
                gold[8].values, errors='coerce')
            gold_named['M_Money_Positions_Short_ALL'] = pd.to_numeric(
                gold[9].values, errors='coerce')
            gold_named['M_Money_Positions_Spread_ALL'] = pd.to_numeric(
                gold[10].values, errors='coerce')
            print(f"  ✅ Current week found (Text)")
            return gold_named
        return None
    except Exception as e:
        print(f"  ❌ Current week failed: {e}")
        return None

# ── Fetch all years ──
print("Fetching Gold COT data from CFTC...")
all_gold = []
current_year = datetime.today().year

# Historical years via Excel
for year in range(current_year - 2, current_year):
    print(f"  Fetching {year}...")
    data = fetch_cot_historical_xls(year)
    if data is not None and len(data) > 0:
        all_gold.append(data)

# Current year via text file — map to same structure
print(f"  Fetching {current_year} (current)...")
current = fetch_cot_current()

# Also try current year Excel if available
if current is None or len(current) < 10:
    print(f"  Trying {current_year} Excel...")
    current_xls = fetch_cot_historical_xls(current_year)
    if current_xls is not None:
        all_gold.append(current_xls)
else:
    # We'll handle current week separately
    pass

# ============================================================
# PROCESS EXCEL DATA (has proper column names)
# ============================================================

if all_gold:
    combined = pd.concat(all_gold, ignore_index=True)

    # Find managed money columns (speculators)
    mm_long_col = [c for c in combined.columns if 'M_Money' in c and 'Long' in c]
    mm_short_col = [c for c in combined.columns if 'M_Money' in c and 'Short' in c]

    print(f"\nManaged Money columns found:")
    print(f"  Long:  {mm_long_col}")
    print(f"  Short: {mm_short_col}")

    combined['date'] = pd.to_datetime(
        combined['Report_Date_as_MM_DD_YYYY'], errors='coerce')
    combined['open_interest'] = pd.to_numeric(
        combined['Open_Interest_All'], errors='coerce')
    combined['spec_long'] = pd.to_numeric(
        combined[mm_long_col[0]], errors='coerce')
    combined['spec_short'] = pd.to_numeric(
        combined[mm_short_col[0]], errors='coerce')
    combined['net_speculator'] = combined['spec_long'] - combined['spec_short']

    # Also get producer/commercial positions
    prod_long = [c for c in combined.columns if 'Prod_Merc' in c and 'Long' in c]
    prod_short = [c for c in combined.columns if 'Prod_Merc' in c and 'Short' in c]
    if prod_long and prod_short:
        combined['comm_long'] = pd.to_numeric(combined[prod_long[0]], errors='coerce')
        combined['comm_short'] = pd.to_numeric(combined[prod_short[0]], errors='coerce')
        combined['net_commercial'] = combined['comm_long'] - combined['comm_short']
    else:
        combined['net_commercial'] = np.nan

    combined = combined.dropna(subset=['date', 'net_speculator'])
    combined = combined.sort_values('date').drop_duplicates(
        subset=['date']).reset_index(drop=True)

    # Add current week if we got it separately
    if current is not None and len(current) > 0:
        current['date'] = pd.to_datetime(
            current['Report_Date_as_MM_DD_YYYY'], errors='coerce')
        current['open_interest'] = current['Open_Interest_All']
        current['spec_long'] = current['M_Money_Positions_Long_ALL']
        current['spec_short'] = current['M_Money_Positions_Short_ALL']
        current['net_speculator'] = current['spec_long'] - current['spec_short']
        current['net_commercial'] = np.nan

        # Only add if date not already present
        existing_dates = combined['date'].values
        new_rows = current[~current['date'].isin(existing_dates)]
        if len(new_rows) > 0:
            combined = pd.concat(
                [combined, new_rows[['date', 'open_interest', 'spec_long',
                                      'spec_short', 'net_speculator',
                                      'net_commercial']]],
                ignore_index=True
            ).sort_values('date').reset_index(drop=True)

    print(f"\n✅ Total weeks: {len(combined)}")
    print(f"   Date range: {combined['date'].iloc[0].date()} → "
          f"{combined['date'].iloc[-1].date()}")

    # ── Last 52 weeks ──
    df_plot = combined.tail(52).copy()
    net_spec = float(np.asarray(df_plot['net_speculator'])[-1])
    net_spec_prev = float(np.asarray(df_plot['net_speculator'])[-2])
    weekly_change = net_spec - net_spec_prev
    net_52w_high = float(df_plot['net_speculator'].max())
    net_52w_low = float(df_plot['net_speculator'].min())
    net_52w_avg = float(df_plot['net_speculator'].mean())
    current_percentile = (
        np.sum(df_plot['net_speculator'].values <= net_spec) / len(df_plot)
    ) * 100

    # ── Gold price ──
    gold_price = yf.download('GC=F', period='3y', interval='1wk', progress=False)
    if hasattr(gold_price, 'columns') and isinstance(gold_price.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
        gold_price.columns = gold_price.columns.get_level_values(0)
    gold_price.columns = [col[0] if isinstance(col, tuple) else col
                          for col in gold_price.columns]
    current_price = float(np.asarray(gold_price['Close'])[-1])

    # ============================================================
    # PLOT
    # ============================================================
    fig = plt.figure(figsize=(16, 20), facecolor='#0d0d0d')
    gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.5, wspace=0.3)

    fig.suptitle(
        'COT REPORT — Gold Managed Money Positioning\n'
        'Commodity Exchange Inc. (COMEX) — REAL CFTC DATA',
        fontsize=15, fontweight='bold', color='gold', y=0.98)

    cot_dates = df_plot['date']
    net_values = df_plot['net_speculator']

    # ── Plot 1: Gold Price + Net Position ──
    ax1 = fig.add_subplot(gs[0, :])
    ax1.set_facecolor('#0d0d0d')
    ax1_twin = ax1.twinx()

    ax1.fill_between(cot_dates, net_values, 0,
                     where=net_values >= 0, alpha=0.4, color='#2ecc71')
    ax1.fill_between(cot_dates, net_values, 0,
                     where=net_values < 0, alpha=0.4, color='#e74c3c')
    ax1.plot(cot_dates, net_values, color='#3498db',
             linewidth=1.5, label='Net Managed Money')
    ax1.axhline(y=0, color='white', linewidth=0.5)
    ax1.axhline(y=net_52w_avg, color='white', linewidth=0.8,
                linestyle='--', alpha=0.5, label=f'52W Avg: {net_52w_avg:,.0f}')
    ax1.set_ylabel('Net Contracts', color='#3498db', fontsize=9)
    ax1.tick_params(colors='white', labelsize=8)
    ax1.spines['bottom'].set_color('#333333')
    ax1.spines['left'].set_color('#333333')
    ax1.spines['top'].set_visible(False)

    ax1_twin.plot(gold_price.index[-52:], gold_price['Close'].tail(52),
                  color='gold', linewidth=2,
                  label=f'Gold ${current_price:,.0f}')
    ax1_twin.set_ylabel('Gold Price (USD)', color='gold', fontsize=9)
    ax1_twin.tick_params(colors='white', labelsize=8)
    ax1_twin.spines['right'].set_color('#333333')
    ax1_twin.spines['top'].set_visible(False)

    ax1.set_title('Net Managed Money Position vs Gold Price',
                  color='white', fontsize=11)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1_twin.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2,
               facecolor='#1a1a1a', labelcolor='white', fontsize=8,
               loc='upper left')

    # ── Plot 2: Longs vs Shorts ──
    ax2 = fig.add_subplot(gs[1, :])
    ax2.set_facecolor('#0d0d0d')
    ax2.plot(cot_dates, df_plot['spec_long'], color='#2ecc71',
             linewidth=1.5, label='Managed Money Longs')
    ax2.plot(cot_dates, df_plot['spec_short'], color='#e74c3c',
             linewidth=1.5, label='Managed Money Shorts')
    ax2.fill_between(cot_dates, df_plot['spec_long'], df_plot['spec_short'],
                     where=df_plot['spec_long'] >= df_plot['spec_short'],
                     alpha=0.2, color='#2ecc71')
    ax2.fill_between(cot_dates, df_plot['spec_long'], df_plot['spec_short'],
                     where=df_plot['spec_long'] < df_plot['spec_short'],
                     alpha=0.2, color='#e74c3c')
    ax2.set_title('Managed Money Longs vs Shorts',
                  color='white', fontsize=10)
    ax2.set_ylabel('Contracts', color='white', fontsize=9)
    ax2.tick_params(colors='white', labelsize=8)
    ax2.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
    ax2.spines['bottom'].set_color('#333333')
    ax2.spines['left'].set_color('#333333')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    # ── Plot 3: Net Position Bars ──
    ax3 = fig.add_subplot(gs[2, 0])
    ax3.set_facecolor('#0d0d0d')
    bar_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in net_values]
    ax3.bar(range(len(net_values)), net_values,
            color=bar_colors, alpha=0.8, width=0.8)
    ax3.axhline(y=net_52w_avg, color='white', linewidth=1,
                linestyle='--', label=f'52W Avg: {net_52w_avg:,.0f}')
    ax3.axhline(y=net_spec, color='gold', linewidth=2,
                label=f'Current: {net_spec:,.0f}')
    ax3.set_title('Net Position — Last 52 Weeks', color='white', fontsize=10)
    ax3.set_ylabel('Net Contracts', color='white')
    ax3.set_xlabel('Weeks (0=oldest)', color='white')
    ax3.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
    ax3.tick_params(colors='white', labelsize=8)
    ax3.spines['bottom'].set_color('#333333')
    ax3.spines['left'].set_color('#333333')
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)

    # ── Plot 4: Percentile Gauge ──
    ax4 = fig.add_subplot(gs[2, 1])
    ax4.set_facecolor('#0d0d0d')
    pct_range = np.linspace(0, 100, 100)
    gauge_colors = []
    for p in pct_range:
        if p < 20:
            gauge_colors.append('#2ecc71')
        elif p < 40:
            gauge_colors.append('#27ae60')
        elif p < 60:
            gauge_colors.append('#f39c12')
        elif p < 80:
            gauge_colors.append('#e67e22')
        else:
            gauge_colors.append('#e74c3c')
    ax4.barh(pct_range, [1]*100, color=gauge_colors, alpha=0.7, height=1.0)
    ax4.axhline(y=current_percentile, color='white', linewidth=3,
                label=f'Current: {current_percentile:.0f}th pct')
    ax4.set_title('Positioning Percentile\n(vs Last 52 Weeks)',
                  color='white', fontsize=10)
    ax4.set_ylabel('Percentile', color='white')
    ax4.set_ylim(0, 100)
    ax4.set_xlim(0, 1)
    ax4.set_xticks([])
    ax4.text(0.5, 10, 'VERY LOW\n(Contrarian BUY)', ha='center',
             va='center', color='white', fontsize=8,
             transform=ax4.get_yaxis_transform())
    ax4.text(0.5, 50, 'NEUTRAL', ha='center', va='center',
             color='white', fontsize=8,
             transform=ax4.get_yaxis_transform())
    ax4.text(0.5, 90, 'VERY HIGH\n(Contrarian SELL)', ha='center',
             va='center', color='white', fontsize=8,
             transform=ax4.get_yaxis_transform())
    ax4.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=9)
    ax4.tick_params(colors='white', labelsize=8)
    ax4.spines['bottom'].set_color('#333333')
    ax4.spines['left'].set_color('#333333')
    ax4.spines['top'].set_visible(False)
    ax4.spines['right'].set_visible(False)

    # ── Plot 5: Weekly Change ──
    ax5 = fig.add_subplot(gs[3, :])
    ax5.set_facecolor('#0d0d0d')
    weekly_changes = df_plot['net_speculator'].diff().fillna(0)
    change_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in weekly_changes]
    ax5.bar(cot_dates, weekly_changes,
            color=change_colors, alpha=0.8, width=5)
    ax5.axhline(y=0, color='white', linewidth=0.5)
    ax5.set_title('Weekly Change in Net Position', color='white', fontsize=10)
    ax5.set_ylabel('Change in Contracts', color='white')
    ax5.tick_params(colors='white', labelsize=8)
    ax5.spines['bottom'].set_color('#333333')
    ax5.spines['left'].set_color('#333333')
    ax5.spines['top'].set_visible(False)
    ax5.spines['right'].set_visible(False)

    plt.savefig(str(OUTDIR / 'cot_positioning.png'), dpi=150,
                bbox_inches='tight', facecolor='#0d0d0d')
    plt.show()

    # ============================================================
    # PRINT READINGS
    # ============================================================
    print("\n" + "="*60)
    print("COT REPORT — CURRENT POSITIONING (REAL CFTC DATA)")
    print("="*60)
    print(f"  Date:                    {datetime.today().strftime('%Y-%m-%d')}")
    print(f"  COT Report Date:         {combined['date'].iloc[-1].strftime('%Y-%m-%d')}")
    print(f"  Gold Price:              ${current_price:,.0f}")
    print(f"\n  MANAGED MONEY POSITIONS:")
    print(f"  Longs:                   {df_plot['spec_long'].iloc[-1]:>10,.0f}")
    print(f"  Shorts:                  {df_plot['spec_short'].iloc[-1]:>10,.0f}")
    print(f"  Net Position:            {net_spec:>+10,.0f}")
    print(f"  Weekly Change:           {weekly_change:>+10,.0f}")
    print(f"  52W High:                {net_52w_high:>10,.0f}")
    print(f"  52W Low:                 {net_52w_low:>10,.0f}")
    print(f"  52W Average:             {net_52w_avg:>10,.0f}")
    print(f"  Percentile:              {current_percentile:>9.0f}th")
    print(f"  Open Interest:           {df_plot['open_interest'].iloc[-1]:>10,.0f}")

    print(f"\n  INTERPRETATION:")
    if current_percentile > 80:
        print(f"  🔴 CROWDED LONG ({current_percentile:.0f}th pct)")
        print(f"     Extreme bullish positioning — contrarian warning!")
        print(f"     Tighten stops, take partial profits, reduce size")
    elif current_percentile > 60:
        print(f"  🟡 ELEVATED LONG ({current_percentile:.0f}th pct)")
        print(f"     Some crowding but not extreme")
        print(f"     Normal size, watch for sudden reversals")
    elif current_percentile > 40:
        print(f"  🟢 NEUTRAL ({current_percentile:.0f}th pct)")
        print(f"     Not stretched — good for trend following")
        print(f"     Follow MRM + HMM signals normally")
    elif current_percentile > 20:
        print(f"  🟢 LOW LONG ({current_percentile:.0f}th pct)")
        print(f"     Room for more buyers — supportive for rally")
        print(f"     Full size on bullish signals")
    else:
        print(f"  🟢 WASHED OUT ({current_percentile:.0f}th pct)")
        print(f"     Speculators capitulated — strong contrarian buy!")
        print(f"     Aggressive long on any bullish catalyst")

    if abs(weekly_change) > 10000:
        direction = "INFLOW 📈" if weekly_change > 0 else "OUTFLOW 📉"
        print(f"\n  ⚡ LARGE WEEKLY {direction}: {weekly_change:+,.0f} contracts")

    print(f"\n  📌 Source: CFTC — Commodity Exchange Inc. (COMEX)")
    print(f"  📌 Disaggregated report — Managed Money = hedge funds")
    print(f"  📌 Released every Friday 3:30pm EST (10:30pm Greece)")
    print(f"  📌 Data as of previous Tuesday")
    print("="*60)


In [ ]:
# ── export this notebook to HTML ──────────────────────────────────────
# FIX: was a hardcoded interpreter (/opt/anaconda3/bin/jupyter) and a
# hardcoded notebook path. Both break on any other machine or env.
import subprocess, sys, glob, os
from pathlib import Path

_here = Path.cwd()
_nb = sorted(glob.glob(str(_here / "Daily_systematic_gold_macro_model*.ipynb")),
             key=os.path.getmtime)
if _nb:
    subprocess.run([sys.executable, "-m", "jupyter", "nbconvert",
                    "--to", "html", "--no-input", "--no-prompt", _nb[-1]])
    print(f"  exported {Path(_nb[-1]).name}")
else:
    print(f"  notebook not found in {_here} — run this from the folder it lives in")

In [ ]:
# ── portable output dir (was hardcoded /Users/elena_nael/...) ─────────
from pathlib import Path as _Path
OUTDIR = _Path.cwd() / "outputs"; OUTDIR.mkdir(exist_ok=True)

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# GOLD MARKET X-RAY — Intraday Order Flow Analysis
# ============================================================
# Uses 1-min OHLCV data to approximate:
# 1. Volume Profile (real)
# 2. Delta / Order Imbalance (estimated via Tick Rule)
# 3. Bid-Ask Heatmap (approximated from price/volume)
# 4. Large Order Detection (statistical)
# ============================================================

# ── Settings ──
TICKER = 'GC=F'
LOOKBACK_DAYS = 3      # days of 1-min data
LARGE_ORDER_MULT = 3.0 # x average volume = large order
POC_ZONES = 10         # number of price zones for volume profile

# ── Download 1-min data ──
print("Downloading 1-minute gold futures data...")
end = datetime.now()
start = end - timedelta(days=LOOKBACK_DAYS)

df = yf.download(TICKER, start=start, end=end, interval='1m', progress=False)
if hasattr(df, 'columns') and isinstance(df.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    df.columns = df.columns.get_level_values(0)
df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
df = df.dropna()

print(f"✅ {len(df)} bars downloaded")
print(f"   From: {df.index[0]}")
print(f"   To:   {df.index[-1]}")

current_price = float(np.asarray(df['Close'])[-1])
print(f"   Current Price: ${current_price:,.2f}")

# ============================================================
# CALCULATIONS
# ============================================================

# ── 1. Delta / Order Imbalance (Tick Rule) ──
# If close > open → buying pressure (positive delta)
# If close < open → selling pressure (negative delta)
df['delta'] = np.where(
    df['Close'] > df['Open'],
    df['Volume'],   # buying volume
    np.where(
        df['Close'] < df['Open'],
        -df['Volume'],  # selling volume
        0
    )
)
df['cumulative_delta'] = df['delta'].cumsum()

# Buy/Sell volume split
df['buy_volume'] = np.where(df['delta'] > 0, df['Volume'], 0)
df['sell_volume'] = np.where(df['delta'] < 0, df['Volume'], 0)

# Rolling delta imbalance (30-bar window)
df['delta_imbalance'] = df['delta'].rolling(30).sum()
df['delta_pct'] = df['delta'].rolling(30).sum() / (
    df['Volume'].rolling(30).sum() + 1e-10) * 100

# ── 2. Large Order Detection ──
vol_avg = df['Volume'].rolling(50).mean()
vol_std = df['Volume'].rolling(50).std()
df['is_large_order'] = df['Volume'] > (vol_avg + LARGE_ORDER_MULT * vol_std)
df['large_buy'] = df['is_large_order'] & (df['delta'] > 0)
df['large_sell'] = df['is_large_order'] & (df['delta'] < 0)

# ── 3. Volume Profile ──
price_min = float(df['Low'].min())
price_max = float(df['High'].max())
price_bins = np.linspace(price_min, price_max, 100)
volume_profile = np.zeros(len(price_bins) - 1)

for _, row in df.iterrows():
    bar_low = float(row['Low'])
    bar_high = float(row['High'])
    bar_vol = float(row['Volume'])
    # Distribute volume across price range touched
    mask = (price_bins[:-1] >= bar_low) & (price_bins[1:] <= bar_high)
    if mask.any():
        volume_profile[mask] += bar_vol / max(mask.sum(), 1)

# Point of Control (POC) = price level with most volume
poc_idx = np.argmax(volume_profile)
poc_price = (price_bins[poc_idx] + price_bins[poc_idx + 1]) / 2

# Value Area (70% of volume)
total_vol = volume_profile.sum()
sorted_idx = np.argsort(volume_profile)[::-1]
cumvol = 0
value_area_idx = []
for idx in sorted_idx:
    cumvol += volume_profile[idx]
    value_area_idx.append(idx)
    if cumvol >= 0.70 * total_vol:
        break
vah = (price_bins[max(value_area_idx)] + price_bins[max(value_area_idx) + 1]) / 2
val = (price_bins[min(value_area_idx)] + price_bins[min(value_area_idx) + 1]) / 2

# ── 4. Bid-Ask Heatmap Approximation ──
# Use high-low spread as proxy for bid-ask width
df['spread_proxy'] = df['High'] - df['Low']
df['spread_ma'] = df['spread_proxy'].rolling(20).mean()
df['spread_zscore'] = (
    (df['spread_proxy'] - df['spread_ma']) /
    (df['spread_proxy'].rolling(20).std() + 1e-10)
)

# ── 5. VWAP ──
df['vwap'] = (df['Close'] * df['Volume']).cumsum() / df['Volume'].cumsum()

# ── 6. Session Stats ──
last_session = df.tail(390)  # ~1 trading day of 1-min bars
session_volume = float(last_session['Volume'].sum())
session_buy_vol = float(last_session['buy_volume'].sum())
session_sell_vol = float(last_session['sell_volume'].sum())
session_delta = session_buy_vol - session_sell_vol
session_delta_pct = (session_buy_vol / (session_volume + 1e-10)) * 100

large_buys_today = last_session['large_buy'].sum()
large_sells_today = last_session['large_sell'].sum()

# ── 7. Imbalance Signal ──
def get_imbalance_signal(delta_pct):
    if delta_pct > 60:
        return '🟢 STRONG BUY PRESSURE'
    elif delta_pct > 55:
        return '🟢 MILD BUY PRESSURE'
    elif delta_pct > 45:
        return '🟡 NEUTRAL / BALANCED'
    elif delta_pct > 40:
        return '🔴 MILD SELL PRESSURE'
    else:
        return '🔴 STRONG SELL PRESSURE'

# ============================================================
# PLOT
# ============================================================

fig = plt.figure(figsize=(20, 24), facecolor='#0d0d0d')
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

fig.suptitle(
    f'GOLD MARKET X-RAY — Intraday Order Flow\n'
    f'GC=F @ ${current_price:,.2f}  |  '
    f'{datetime.now().strftime("%Y-%m-%d %H:%M")} EST',
    fontsize=16, fontweight='bold', color='gold', y=0.98
)

# ── Plot 1: Price + VWAP + Large Orders (wide) ──
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor('#0d0d0d')

plot_df = df.tail(200)  # last ~3 hours

ax1.plot(plot_df.index, plot_df['Close'],
         color='gold', linewidth=1.2, label='Gold Price', zorder=3)
ax1.plot(plot_df.index, plot_df['vwap'],
         color='#9b59b6', linewidth=1, linestyle='--',
         label='VWAP', alpha=0.8, zorder=3)

# Large buy orders
large_buy_df = plot_df[plot_df['large_buy']]
large_sell_df = plot_df[plot_df['large_sell']]

ax1.scatter(large_buy_df.index, large_buy_df['Low'] - 1,
            color='#2ecc71', marker='^', s=100, zorder=5,
            label=f'Large Buy ({len(large_buy_df)})')
ax1.scatter(large_sell_df.index, large_sell_df['High'] + 1,
            color='#e74c3c', marker='v', s=100, zorder=5,
            label=f'Large Sell ({len(large_sell_df)})')

# POC and Value Area
ax1.axhline(y=poc_price, color='#f39c12', linewidth=1.5,
            linestyle='-', label=f'POC ${poc_price:,.1f}', alpha=0.9)
ax1.axhline(y=vah, color='#e74c3c', linewidth=0.8,
            linestyle='--', label=f'VAH ${vah:,.1f}', alpha=0.7)
ax1.axhline(y=val, color='#2ecc71', linewidth=0.8,
            linestyle='--', label=f'VAL ${val:,.1f}', alpha=0.7)

ax1.set_title('Price Action + VWAP + Large Orders + Key Levels',
              color='white', fontsize=11)
ax1.set_ylabel('Price (USD)', color='white')
ax1.tick_params(colors='white', labelsize=7)
ax1.legend(facecolor='#1a1a1a', labelcolor='white',
           fontsize=7, ncol=4, loc='upper left')
ax1.spines['bottom'].set_color('#333333')
ax1.spines['left'].set_color('#333333')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── Plot 2: Volume Profile (horizontal) ──
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor('#0d0d0d')

bin_centers = (price_bins[:-1] + price_bins[1:]) / 2
max_vol = volume_profile.max()

# Color by volume intensity
colors_vp = []
for v in volume_profile:
    ratio = v / max_vol
    if ratio > 0.8:
        colors_vp.append('#f39c12')  # POC zone - orange
    elif ratio > 0.5:
        colors_vp.append('#2ecc71')  # High vol - green
    elif ratio > 0.2:
        colors_vp.append('#3498db')  # Medium vol - blue
    else:
        colors_vp.append('#2d2d2d')  # Low vol - dark

ax2.barh(bin_centers, volume_profile, height=(price_max-price_min)/100,
         color=colors_vp, alpha=0.8)
ax2.axhline(y=poc_price, color='#f39c12', linewidth=2,
            label=f'POC ${poc_price:,.1f}')
ax2.axhline(y=vah, color='#e74c3c', linewidth=1,
            linestyle='--', label=f'VAH ${vah:,.1f}')
ax2.axhline(y=val, color='#2ecc71', linewidth=1,
            linestyle='--', label=f'VAL ${val:,.1f}')
ax2.axhline(y=current_price, color='gold', linewidth=1.5,
            label=f'Price ${current_price:,.1f}')

ax2.set_title('Volume Profile', color='white', fontsize=11)
ax2.set_xlabel('Volume', color='white')
ax2.set_ylabel('Price (USD)', color='white')
ax2.tick_params(colors='white', labelsize=7)
ax2.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=7)
ax2.spines['bottom'].set_color('#333333')
ax2.spines['left'].set_color('#333333')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# ── Plot 3: Delta / Order Imbalance ──
ax3 = fig.add_subplot(gs[1, :2])
ax3.set_facecolor('#0d0d0d')

delta_colors = ['#2ecc71' if v >= 0 else '#e74c3c'
                for v in plot_df['delta']]
ax3.bar(plot_df.index, plot_df['delta'],
        color=delta_colors, alpha=0.6, width=0.0005)
ax3.plot(plot_df.index, plot_df['delta_imbalance'] / 30,
         color='white', linewidth=1.2,
         label='30-bar Rolling Delta', alpha=0.8)
ax3.axhline(y=0, color='white', linewidth=0.5)

ax3_twin = ax3.twinx()
ax3_twin.plot(plot_df.index, plot_df['cumulative_delta'],
              color='#9b59b6', linewidth=1.5,
              label='Cumulative Delta', alpha=0.9)
ax3_twin.set_ylabel('Cumulative Delta', color='#9b59b6', fontsize=9)
ax3_twin.tick_params(colors='white', labelsize=7)
ax3_twin.spines['right'].set_color('#333333')
ax3_twin.spines['top'].set_visible(False)

ax3.set_title('Delta — Order Imbalance (Green=Buying / Red=Selling)',
              color='white', fontsize=11)
ax3.set_ylabel('Delta (contracts)', color='white')
ax3.tick_params(colors='white', labelsize=7)
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2,
           facecolor='#1a1a1a', labelcolor='white', fontsize=8)
ax3.spines['bottom'].set_color('#333333')
ax3.spines['left'].set_color('#333333')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# ── Plot 4: Buy/Sell Volume Ratio ──
ax4 = fig.add_subplot(gs[1, 2])
ax4.set_facecolor('#0d0d0d')

buy_roll = plot_df['buy_volume'].rolling(30).sum()
sell_roll = plot_df['sell_volume'].abs().rolling(30).sum()
total_roll = buy_roll + sell_roll
buy_pct = (buy_roll / (total_roll + 1e-10)) * 100

ax4.fill_between(plot_df.index, buy_pct, 50,
                 where=buy_pct >= 50,
                 alpha=0.4, color='#2ecc71', label='Buy Dominance')
ax4.fill_between(plot_df.index, buy_pct, 50,
                 where=buy_pct < 50,
                 alpha=0.4, color='#e74c3c', label='Sell Dominance')
ax4.plot(plot_df.index, buy_pct,
         color='white', linewidth=1, alpha=0.8)
ax4.axhline(y=50, color='white', linewidth=0.5, linestyle='--')
ax4.axhline(y=60, color='#2ecc71', linewidth=0.5,
            linestyle=':', alpha=0.5, label='60% Buy threshold')
ax4.axhline(y=40, color='#e74c3c', linewidth=0.5,
            linestyle=':', alpha=0.5, label='40% Sell threshold')
ax4.set_ylim(20, 80)
ax4.set_title('Buy/Sell Volume %\n(30-bar rolling)', color='white', fontsize=10)
ax4.set_ylabel('%', color='white')
ax4.tick_params(colors='white', labelsize=7)
ax4.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=7)
ax4.spines['bottom'].set_color('#333333')
ax4.spines['left'].set_color('#333333')
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

# ── Plot 5: Bid-Ask Spread Heatmap ──
ax5 = fig.add_subplot(gs[2, :2])
ax5.set_facecolor('#0d0d0d')

# Create heatmap-style visualization
spread_data = plot_df['spread_zscore'].values.reshape(1, -1)
im = ax5.imshow(
    spread_data,
    aspect='auto',
    cmap='RdYlGn_r',
    vmin=-2, vmax=2,
    extent=[0, len(plot_df), float(plot_df['Low'].min()),
            float(plot_df['High'].max())]
)
ax5.plot(range(len(plot_df)), plot_df['Close'].values,
         color='white', linewidth=1, alpha=0.9, label='Price')
ax5.plot(range(len(plot_df)), plot_df['vwap'].values,
         color='#9b59b6', linewidth=0.8, linestyle='--',
         alpha=0.7, label='VWAP')

plt.colorbar(im, ax=ax5, label='Spread Z-Score',
             orientation='horizontal', fraction=0.02, pad=0.15)
ax5.set_title('Bid-Ask Spread Heatmap (Red=Wide/Illiquid, Green=Tight/Liquid)',
              color='white', fontsize=10)
ax5.set_ylabel('Price (USD)', color='white')
ax5.tick_params(colors='white', labelsize=7)
ax5.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
ax5.spines['bottom'].set_color('#333333')
ax5.spines['left'].set_color('#333333')
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

# ── Plot 6: Large Order Detection ──
ax6 = fig.add_subplot(gs[2, 2])
ax6.set_facecolor('#0d0d0d')

ax6.bar(plot_df.index, plot_df['Volume'],
        color='#2d2d2d', alpha=0.5, width=0.0005, label='Normal Volume')

large_buy_plot = plot_df[plot_df['large_buy']]
large_sell_plot = plot_df[plot_df['large_sell']]

ax6.bar(large_buy_plot.index, large_buy_plot['Volume'],
        color='#2ecc71', alpha=0.9, width=0.0005,
        label=f'Large Buy ({len(large_buy_plot)})')
ax6.bar(large_sell_plot.index, large_sell_plot['Volume'],
        color='#e74c3c', alpha=0.9, width=0.0005,
        label=f'Large Sell ({len(large_sell_plot)})')

ax6.plot(plot_df.index, vol_avg.tail(200),
         color='white', linewidth=0.8, linestyle='--',
         alpha=0.6, label='Avg Volume')
ax6.plot(plot_df.index,
         (vol_avg + LARGE_ORDER_MULT * vol_std).tail(200),
         color='#f39c12', linewidth=0.8, linestyle=':',
         alpha=0.6, label=f'{LARGE_ORDER_MULT}x threshold')

ax6.set_title('Large Order Detection\n(Iceberg Approximation)',
              color='white', fontsize=10)
ax6.set_ylabel('Volume', color='white')
ax6.tick_params(colors='white', labelsize=7)
ax6.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=7)
ax6.spines['bottom'].set_color('#333333')
ax6.spines['left'].set_color('#333333')
ax6.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)

# ── Plot 7: Volume by Hour of Day ──
ax7 = fig.add_subplot(gs[3, 0])
ax7.set_facecolor('#0d0d0d')

df['hour'] = df.index.hour
hourly_vol = df.groupby('hour')['Volume'].mean()
hourly_delta = df.groupby('hour')['delta'].mean()

bar_colors_h = ['#2ecc71' if v >= 0 else '#e74c3c'
                for v in hourly_delta]
ax7.bar(hourly_vol.index, hourly_vol.values,
        color=bar_colors_h, alpha=0.8, width=0.7)
ax7.set_title('Avg Volume by Hour\n(Green=Net Buying / Red=Net Selling)',
              color='white', fontsize=10)
ax7.set_xlabel('Hour (EST)', color='white')
ax7.set_ylabel('Avg Volume', color='white')
ax7.tick_params(colors='white', labelsize=8)
ax7.spines['bottom'].set_color('#333333')
ax7.spines['left'].set_color('#333333')
ax7.spines['top'].set_visible(False)
ax7.spines['right'].set_visible(False)

# ── Plot 8: Cumulative Delta vs Price ──
ax8 = fig.add_subplot(gs[3, 1])
ax8.set_facecolor('#0d0d0d')

ax8.plot(plot_df.index, plot_df['cumulative_delta'],
         color='#3498db', linewidth=1.5, label='Cum. Delta')
ax8.axhline(y=0, color='white', linewidth=0.5)
ax8_twin = ax8.twinx()
ax8_twin.plot(plot_df.index, plot_df['Close'],
              color='gold', linewidth=1, alpha=0.7, label='Price')
ax8_twin.set_ylabel('Price', color='gold', fontsize=8)
ax8_twin.tick_params(colors='white', labelsize=7)
ax8_twin.spines['right'].set_color('#333333')
ax8_twin.spines['top'].set_visible(False)

ax8.set_title('Cumulative Delta vs Price\n(Divergence = Reversal Signal)',
              color='white', fontsize=10)
ax8.set_ylabel('Cumulative Delta', color='#3498db')
ax8.tick_params(colors='white', labelsize=7)
lines1, labels1 = ax8.get_legend_handles_labels()
lines2, labels2 = ax8_twin.get_legend_handles_labels()
ax8.legend(lines1 + lines2, labels1 + labels2,
           facecolor='#1a1a1a', labelcolor='white', fontsize=8)
ax8.spines['bottom'].set_color('#333333')
ax8.spines['left'].set_color('#333333')
ax8.spines['top'].set_visible(False)
ax8.spines['right'].set_visible(False)

# ── Plot 9: Session Summary Gauge ──
ax9 = fig.add_subplot(gs[3, 2])
ax9.set_facecolor('#0d0d0d')
ax9.set_xlim(0, 1)
ax9.set_ylim(0, 10)
ax9.axis('off')

signal = get_imbalance_signal(session_delta_pct)
signal_color = '#2ecc71' if 'BUY' in signal else (
    '#e74c3c' if 'SELL' in signal else '#f39c12')

# Summary text
summary = [
    ('SESSION SUMMARY', 'gold', 14),
    ('', 'white', 8),
    (f'Buy Volume:  {session_buy_vol:,.0f}', '#2ecc71', 9),
    (f'Sell Volume: {session_sell_vol:,.0f}', '#e74c3c', 9),
    (f'Net Delta:   {session_delta:+,.0f}', signal_color, 9),
    (f'Buy %:       {session_delta_pct:.1f}%', signal_color, 9),
    ('', 'white', 8),
    (f'Large Buys:  {large_buys_today}', '#2ecc71', 9),
    (f'Large Sells: {large_sells_today}', '#e74c3c', 9),
    ('', 'white', 8),
    (f'POC:  ${poc_price:,.1f}', '#f39c12', 9),
    (f'VAH:  ${vah:,.1f}', '#e74c3c', 9),
    (f'VAL:  ${val:,.1f}', '#2ecc71', 9),
    (f'VWAP: ${float(np.asarray(df["vwap"])[-1]):,.1f}', '#9b59b6', 9),
    ('', 'white', 8),
    (signal, signal_color, 10),
]

y_pos = 9.5
for text, color, size in summary:
    ax9.text(0.5, y_pos, text, ha='center', va='top',
             color=color, fontsize=size,
             fontweight='bold' if size >= 10 else 'normal',
             transform=ax9.transData)
    y_pos -= 0.6

plt.savefig(str(OUTDIR / 'gold_market_xray_intraday.png'),
            dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

# ============================================================
# PRINT READINGS
# ============================================================

print("\n" + "="*60)
print("GOLD MARKET X-RAY — INTRADAY ORDER FLOW")
print("="*60)
print(f"  Time:            {datetime.now().strftime('%Y-%m-%d %H:%M')} EST")
print(f"  Gold Price:      ${current_price:,.2f}")
print(f"  VWAP:            ${float(np.asarray(df['vwap'])[-1]):,.2f}")
print(f"\n  VOLUME PROFILE:")
print(f"  POC (most vol):  ${poc_price:,.2f}")
print(f"  VAH (70% vol):   ${vah:,.2f}")
print(f"  VAL (70% vol):   ${val:,.2f}")
print(f"\n  DELTA / ORDER FLOW:")
print(f"  Session Buy Vol: {session_buy_vol:>12,.0f}")
print(f"  Session Sell Vol:{session_sell_vol:>12,.0f}")
print(f"  Net Delta:       {session_delta:>+12,.0f}")
print(f"  Buy Dominance:   {session_delta_pct:>11.1f}%")
print(f"  Signal:          {signal}")
print(f"\n  LARGE ORDERS (>{LARGE_ORDER_MULT}x avg volume):")
print(f"  Large Buys:      {large_buys_today:>12}")
print(f"  Large Sells:     {large_sells_today:>12}")

if large_buys_today > large_sells_today * 2:
    print(f"  ⚡ INSTITUTIONAL BUYING DETECTED")
elif large_sells_today > large_buys_today * 2:
    print(f"  ⚡ INSTITUTIONAL SELLING DETECTED")
else:
    print(f"  ✅ Balanced institutional activity")

print(f"\n  PRICE vs KEY LEVELS:")
if current_price > vah:
    print(f"  🔴 Above Value Area High — Extended, watch for pullback")
elif current_price < val:
    print(f"  🟢 Below Value Area Low — Potential mean reversion up")
elif current_price > poc_price:
    print(f"  🟢 Above POC — Bullish bias within value area")
else:
    print(f"  🟡 Below POC — Neutral, watch for direction")

if current_price > float(np.asarray(df['vwap'])[-1]):
    print(f"  ✅ Above VWAP — Buyers in control intraday")
else:
    print(f"  ❌ Below VWAP — Sellers in control intraday")

print("="*60)
print("\n⚠️  NOTE: Delta/spread are approximations from OHLCV data")
print("    For true order flow use Bookmap alongside this analysis")
print("="*60)


In [ ]:
# FIX: this was a bare shell command in a code cell -> SyntaxError.
# Prefixed with ! so Jupyter runs it as a shell command.
# (only needed once, for matplotlib animation export)
!conda install -c conda-forge ffmpeg

In [ ]:
# FIX: this was a bare shell command in a code cell -> SyntaxError.
# Prefixed with ! so Jupyter runs it as a shell command.
# (only needed once, for matplotlib animation export)
!conda install -c conda-forge ffmpeg

In [ ]:
# ── portable output dir (was hardcoded /Users/elena_nael/...) ─────────
from pathlib import Path as _Path
OUTDIR = _Path.cwd() / "outputs"; OUTDIR.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# GOLD — LIQUIDITY MAGNET FORECAST
# Where will price be PULLED next based on cluster gravity
# ============================================================

print("Fetching gold data...")
gold = yf.download('GC=F', period='1y', interval='1d', progress=False)
if hasattr(gold, 'columns') and isinstance(gold.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gold.columns = gold.columns.get_level_values(0)
df   = gold[['Open','High','Low','Close','Volume']].dropna()
df.columns = ['Open','High','Low','Close','Volume']

last_price = float(np.asarray(df['Close'])[-1])
last_date  = df.index[-1]
print(f"  Last close: ${last_price:,.2f}  on  {last_date.date()}")

# ============================================================
# BUILD VOLUME PROFILE
# ============================================================
N_BINS     = 400
p_lo       = float(df['Low'].min())  * 0.998
p_hi       = float(df['High'].max()) * 1.002
price_bins = np.linspace(p_lo, p_hi, N_BINS)
bin_size   = price_bins[1] - price_bins[0]
vol_profile = np.zeros(N_BINS)

for _, row in df.iterrows():
    lo  = float(row['Low'])
    hi  = float(row['High'])
    vol = float(row['Volume'])
    mask = (price_bins >= lo) & (price_bins <= hi)
    n    = mask.sum()
    if n > 0:
        weights = np.ones(n)
        ci = int((float(row['Close']) - lo) / (hi - lo + 1e-9) * (n-1))
        ci = np.clip(ci, 0, n-1)
        weights[ci] = 3.0
        weights /= weights.sum()
        vol_profile[mask] += vol * weights

vol_smooth = gaussian_filter1d(vol_profile, sigma=4)
vol_norm   = vol_smooth / vol_smooth.max()

# ============================================================
# FIND CLUSTERS
# ============================================================
peaks, _ = find_peaks(vol_norm, height=0.20, distance=6, prominence=0.06)
peak_strengths = vol_norm[peaks]
order = np.argsort(peak_strengths)[::-1]
peaks = peaks[order]
peak_strengths = peak_strengths[order]

clusters = []
for pk, st in zip(peaks[:16], peak_strengths[:16]):
    p = price_bins[pk]
    clusters.append({
        'price'    : p,
        'strength' : float(st),
        'tier'     : 'MAJOR' if st > 0.60 else ('MEDIUM' if st > 0.35 else 'MINOR'),
        'dist_pct' : (p - last_price) / last_price * 100,
        'above'    : p > last_price,
    })

print(f"  Clusters found: {len(clusters)}")

# ============================================================
# GRAVITY-BASED FORECAST
# Each cluster exerts gravitational pull on price
# Net force determines most likely next destination
# ============================================================

N_DAYS    = 15
N_PATHS   = 600
SIGMA_ANN = 0.254
MU_ANN    = 0.154
SIGMA_D   = SIGMA_ANN / np.sqrt(252)
MU_D      = MU_ANN / 252

np.random.seed(42)

def gravity_drift(price, clusters, kappa=0.0012):
    """Extra drift from liquidity gravity — pulls toward heavy clusters"""
    drift = 0.0
    for c in clusters:
        dist   = c['price'] - price
        decay  = np.exp(-abs(dist / price) / 0.025)
        drift += kappa * c['strength'] * np.sign(dist) * decay
    return drift

# Simulate paths WITH gravity
future_gravity = np.zeros((N_PATHS, N_DAYS + 1))
future_gravity[:, 0] = last_price

for t in range(1, N_DAYS + 1):
    for i in range(N_PATHS):
        p      = future_gravity[i, t-1]
        g_pull = gravity_drift(p, clusters)
        z      = np.random.randn()
        ret    = (MU_D + g_pull - 0.5*SIGMA_D**2) + SIGMA_D*z
        future_gravity[i, t] = p * np.exp(ret)

# Simulate paths WITHOUT gravity (pure GBM baseline)
future_gbm = np.zeros((N_PATHS, N_DAYS + 1))
future_gbm[:, 0] = last_price
for t in range(1, N_DAYS + 1):
    z = np.random.randn(N_PATHS)
    future_gbm[:, t] = future_gbm[:, t-1] * np.exp(
        (MU_D - 0.5*SIGMA_D**2) + SIGMA_D*z
    )

# Percentiles
med_g  = np.median(future_gravity, axis=0)
p10_g  = np.percentile(future_gravity, 10, axis=0)
p25_g  = np.percentile(future_gravity, 25, axis=0)
p75_g  = np.percentile(future_gravity, 75, axis=0)
p90_g  = np.percentile(future_gravity, 90, axis=0)
med_b  = np.median(future_gbm, axis=0)

future_dates = pd.bdate_range(start=last_date, periods=N_DAYS + 1)
bull_pct = float((future_gravity[:, -1] > last_price).mean() * 100)

# Nearest magnet targets
above_clusters = sorted([c for c in clusters if c['above']],
                          key=lambda x: x['price'])
below_clusters = sorted([c for c in clusters if not c['above']],
                          key=lambda x: x['price'], reverse=True)

target_up   = above_clusters[0] if above_clusters else None
target_down = below_clusters[0] if below_clusters else None

# Which target does median path hit first?
eod_median = float(med_g[-1])
primary_target = target_up if eod_median > last_price else target_down

print(f"  Median 15d target: ${eod_median:,.1f}")
print(f"  Bull probability:  {bull_pct:.0f}%")

# ============================================================
# FIGURE
# ============================================================
fig = plt.figure(figsize=(20, 13), facecolor='#07070f')
fig.patch.set_facecolor('#07070f')

gs = gridspec.GridSpec(
    3, 2, figure=fig,
    height_ratios=[0.65, 3.8, 1.1],
    width_ratios=[3.5, 1.0],
    hspace=0.05, wspace=0.03,
    left=0.05, right=0.97,
    top=0.97,  bottom=0.04
)

ax_eq   = fig.add_subplot(gs[0, :])
ax_main = fig.add_subplot(gs[1, 0])
ax_vp   = fig.add_subplot(gs[1, 1])
ax_bot  = fig.add_subplot(gs[2, :])

BG = '#07070f'
for ax in [ax_eq, ax_main, ax_vp, ax_bot]:
    ax.set_facecolor(BG)
    for sp in ax.spines.values():
        sp.set_color('#111122')
        sp.set_linewidth(0.5)
    ax.tick_params(colors='#444466', labelsize=8)

# ═══════════════════════════════════════
# EQUATIONS
# ═══════════════════════════════════════
ax_eq.axis('off')
ax_eq.text(0.5, 0.97,
           'GOLD  LIQUIDITY  MAGNET  FORECAST   —   GRAVITY-ADJUSTED  MONTE  CARLO',
           transform=ax_eq.transAxes,
           color='#ffd700', fontsize=13, fontweight='bold',
           va='top', ha='center',
           path_effects=[pe.withStroke(linewidth=5, foreground='#332200')])

eqs = [
    (r"$S_{t+1} = S_t \cdot e^{(\mu_{eff} + G_t - \frac{1}{2}\sigma^2)\Delta t + \sigma\sqrt{\Delta t}\,\varepsilon}$",
     0.01, 0.52, '#66ccff', 11.5),
    (r"$G_t = \kappa \sum_{j} \mathcal{L}_j \cdot \mathrm{sign}(p_j - S_t) \cdot e^{-|p_j - S_t|\,/\,(S_t \cdot \lambda)}$",
     0.01, 0.10, '#ff88aa', 11),
    (r"$\mathcal{L}_j = $ cluster strength$\quad \kappa = 0.0012 \quad \lambda = 0.025$",
     0.55, 0.52, '#aaffcc', 10.5),
    (r"$\mu_{eff} = 0.154\%/yr \quad \sigma = 25.4\%/yr \quad N = 600\,paths$",
     0.55, 0.10, '#ddaaff', 10.5),
]
for eq_txt, ex, ey, ecol, esz in eqs:
    ax_eq.text(ex, ey, eq_txt,
               transform=ax_eq.transAxes,
               color=ecol, fontsize=esz, va='center', ha='left',
               path_effects=[pe.withStroke(linewidth=2, foreground='#000008')])
ax_eq.axhline(y=0, color='#1a1a33', linewidth=1.0)

# ═══════════════════════════════════════
# MAIN CHART
# ═══════════════════════════════════════
# Price view range
p_view_lo = min(float(p10_g.min()), float(df['Close'].iloc[-40:].min())) * 0.997
p_view_hi = max(float(p90_g.max()), float(df['Close'].iloc[-40:].max())) * 1.003

ax_main.set_xlim(df.index[-40], future_dates[-1])
ax_main.set_ylim(p_view_lo, p_view_hi)
ax_main.set_facecolor('#07070f')
ax_main.yaxis.grid(True, color='#0d0d1a', linewidth=0.5)
ax_main.set_axisbelow(True)

# ── Cluster zone backgrounds ──
for c in clusters:
    if p_view_lo < c['price'] < p_view_hi:
        col   = '#ff3300' if c['tier']=='MAJOR' else \
                ('#ff9900' if c['tier']=='MEDIUM' else '#cccc00')
        alpha = 0.03 + c['strength'] * 0.07
        ax_main.axhspan(c['price'] - bin_size*5,
                         c['price'] + bin_size*5,
                         color=col, alpha=alpha, zorder=0)

# ── Historical price (last 40 days) ──
hist_c = df['Close'].iloc[-40:].values
hist_d = df.index[-40:]
ax_main.plot(hist_d, hist_c,
             color='#aaaacc', linewidth=1.8,
             alpha=0.9, zorder=4, label='Historical')

# ── Forecast bands ──
ax_main.fill_between(future_dates, p10_g, p90_g,
                      color='#001433', alpha=0.80, zorder=2)
ax_main.fill_between(future_dates, p25_g, p75_g,
                      color='#002a55', alpha=0.75, zorder=3)

# ── Individual gravity paths (faint) ──
for i in range(0, N_PATHS, 8):
    fin = future_gravity[i, -1]
    col = '#00e676' if fin > last_price else '#ff4444'
    ax_main.plot(future_dates, future_gravity[i],
                 color=col, linewidth=0.3, alpha=0.05, zorder=2)

# ── Pure GBM baseline (dotted grey) ──
ax_main.plot(future_dates, med_b,
             color='#444466', linewidth=1.2,
             linestyle=':', alpha=0.7, zorder=5,
             label='GBM baseline (no gravity)')

# ── Gravity-adjusted median — THE FORECAST ──
ax_main.plot(future_dates, med_g,
             color='#ffd700', linewidth=3.0,
             alpha=0.97, zorder=8,
             label=f'Gravity forecast  ${eod_median:,.0f}')
ax_main.scatter(future_dates, med_g,
                color='#ffd700', s=50, zorder=9, alpha=0.95)

# Day labels on forecast line
y_span = p_view_hi - p_view_lo
for i in range(1, N_DAYS + 1):
    chg = (med_g[i] - last_price) / last_price * 100
    sym = '+' if chg >= 0 else ''
    col = '#00e676' if chg >= 0 else '#ff4444'
    if i % 3 == 0 or i == N_DAYS:
        ax_main.text(future_dates[i],
                     float(med_g[i]) + y_span*0.018,
                     f'D{i}\n{sym}{chg:.1f}%',
                     color=col, fontsize=7, ha='center',
                     va='bottom', fontweight='bold', zorder=10)

# ── Cluster lines ──
for c in clusters:
    if p_view_lo < c['price'] < p_view_hi:
        col = '#ff4400' if c['tier']=='MAJOR' else \
              ('#ffaa00' if c['tier']=='MEDIUM' else '#888800')
        lw  = 2.0 if c['tier']=='MAJOR' else \
              (1.3 if c['tier']=='MEDIUM' else 0.8)
        ls  = '-' if c['tier']=='MAJOR' else '--'
        ax_main.axhline(c['price'], color=col, linewidth=lw,
                         linestyle=ls, alpha=0.80, zorder=6)

        # Right-side label
        marker = '▲' if c['above'] else '▼'
        ax_main.text(future_dates[-1],
                     c['price'],
                     f"  {marker} ${c['price']:,.0f}  [{c['tier'][0]}]  {c['dist_pct']:+.1f}%",
                     color=col, fontsize=7.5, va='center', ha='left',
                     path_effects=[pe.withStroke(linewidth=2,
                                                  foreground='#07070f')])

# ── Primary target arrow ──
if primary_target:
    tp = primary_target['price']
    if p_view_lo < tp < p_view_hi:
        tcol = '#00ff88' if primary_target['above'] else '#ff4444'
        ax_main.annotate(
            f"PRIMARY TARGET\n${tp:,.0f}  ({primary_target['dist_pct']:+.1f}%)",
            xy=(future_dates[N_DAYS//2], tp),
            xytext=(future_dates[3], tp + (1 if primary_target['above'] else -1)*y_span*0.06),
            color=tcol, fontsize=9, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=tcol, lw=2.0),
            path_effects=[pe.withStroke(linewidth=2, foreground='#07070f')],
            zorder=11
        )

# ── Today divider ──
ax_main.axvline(last_date, color='#ffffff',
                 linewidth=0.8, linestyle='--', alpha=0.4)
ax_main.text(last_date, p_view_lo + y_span*0.01,
             '  Now', color='#666688', fontsize=8)

# ── Current price ──
ax_main.axhline(last_price, color='#ffd700',
                 linewidth=1.0, linestyle='--', alpha=0.5, zorder=7)
ax_main.scatter([last_date], [last_price],
                color='#ffffff', s=100, zorder=12,
                label=f'Now  ${last_price:,.2f}')

ax_main.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax_main.set_ylabel('Gold Price (USD)', color='#ffd700', fontsize=10)
ax_main.set_title(
    f'Gravity-Adjusted 15-Day Forecast   ·   '
    f'Gold price pulled toward liquidity clusters   ·   '
    f'Bull prob: {bull_pct:.0f}%   ·   '
    f'Median target: ${eod_median:,.0f}',
    color='#888899', fontsize=9, pad=4, loc='left'
)
ax_main.legend(loc='upper left', fontsize=8,
               facecolor='#111122', edgecolor='#222233',
               labelcolor='white', framealpha=0.9)

# Summary box
poc_idx   = int(np.argmax(vol_smooth))
poc_price = price_bins[poc_idx]

lines = [
    '  MAGNET FORECAST',
    '  ' + '─'*22,
    f'  Now        ${last_price:,.2f}',
    f'  15d target ${eod_median:,.0f}  ({(eod_median-last_price)/last_price*100:+.2f}%)',
    f'  Bull prob   {bull_pct:.0f}%',
    '  ' + '─'*22,
]
if target_up:
    lines.append(f'  ▲ R1  ${target_up["price"]:,.0f}  ({target_up["dist_pct"]:+.1f}%)')
if len(above_clusters) > 1:
    c2 = above_clusters[1]
    lines.append(f'  ▲ R2  ${c2["price"]:,.0f}  ({c2["dist_pct"]:+.1f}%)')
if target_down:
    lines.append(f'  ▼ S1  ${target_down["price"]:,.0f}  ({target_down["dist_pct"]:+.1f}%)')
if len(below_clusters) > 1:
    c2 = below_clusters[1]
    lines.append(f'  ▼ S2  ${c2["price"]:,.0f}  ({c2["dist_pct"]:+.1f}%)')
lines.append('  ' + '─'*22)
lines.append(f'  POC   ${poc_price:,.0f}')
lines.append(f'  P90   ${float(p90_g[-1]):,.0f}')
lines.append(f'  P10   ${float(p10_g[-1]):,.0f}')

ax_main.text(0.995, 0.02, '\n'.join(lines),
             transform=ax_main.transAxes,
             color='#ccccee', fontsize=8,
             va='bottom', ha='right',
             fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.6',
                       facecolor='#0a0a1a',
                       edgecolor='#222233',
                       alpha=0.93))

# ═══════════════════════════════════════
# VOLUME PROFILE  (right panel)
# ═══════════════════════════════════════
mask = (price_bins >= p_view_lo) & (price_bins <= p_view_hi)
vp_b = price_bins[mask]
vp_v = vol_norm[mask]

vp_colors = ['#ff4400' if v > 0.60 else
             ('#ff9900' if v > 0.35 else
              ('#cccc00' if v > 0.15 else '#1a2a1a'))
             for v in vp_v]

ax_vp.barh(vp_b, vp_v, height=bin_size*0.9,
           color=vp_colors, alpha=0.88)

# POC
if p_view_lo < poc_price < p_view_hi:
    ax_vp.axhline(poc_price, color='#ff4400', linewidth=2.0, alpha=0.9)
    ax_vp.text(0.98, poc_price, f'POC\n${poc_price:,.0f}',
               transform=ax_vp.get_yaxis_transform(),
               color='#ff4400', fontsize=7,
               ha='right', va='center', fontweight='bold')

# Forecast median target
ax_vp.axhline(eod_median, color='#ffd700', linewidth=1.5,
               linestyle='--', alpha=0.85)
ax_vp.text(0.98, eod_median, f'Target\n${eod_median:,.0f}',
           transform=ax_vp.get_yaxis_transform(),
           color='#ffd700', fontsize=6.5, ha='right', va='center')

# Current price
ax_vp.axhline(last_price, color='#ffffff', linewidth=1.2, alpha=0.7)

ax_vp.set_ylim(p_view_lo, p_view_hi)
ax_vp.set_xlim(0, 1.15)
ax_vp.set_yticklabels([])
ax_vp.set_title('Volume\nProfile', color='#888899', fontsize=8, pad=3)
ax_vp.set_xlabel('Normalized', color='#444466', fontsize=7)
ax_vp.xaxis.set_tick_params(labelsize=6)

# ═══════════════════════════════════════
# BOTTOM — gravity path distribution
# ═══════════════════════════════════════
all_finals = future_gravity[:, -1]
kde_f = gaussian_kde(all_finals, bw_method=0.12)
kx    = np.linspace(all_finals.min(), all_finals.max(), 300)
ky    = kde_f(kx)
ky   /= ky.max()

ax_bot.fill_between(kx, ky, 0,
                     where=kx >= last_price,
                     color='#003322', alpha=0.8, label='Bull paths')
ax_bot.fill_between(kx, ky, 0,
                     where=kx < last_price,
                     color='#330000', alpha=0.8, label='Bear paths')
ax_bot.plot(kx, ky, color='#ffffff', linewidth=1.2, alpha=0.7)

# Mark cluster positions on distribution
for c in clusters:
    if all_finals.min() < c['price'] < all_finals.max():
        col = '#ff4400' if c['tier']=='MAJOR' else \
              ('#ff9900' if c['tier']=='MEDIUM' else '#888800')
        ax_bot.axvline(c['price'], color=col,
                        linewidth=1.5 if c['tier']=='MAJOR' else 0.8,
                        alpha=0.8, linestyle='-' if c['tier']=='MAJOR' else '--')
        ax_bot.text(c['price'], 0.92,
                    f"${c['price']:,.0f}",
                    color=col, fontsize=6.5,
                    ha='center', va='top', rotation=90)

# Median and current
ax_bot.axvline(eod_median, color='#ffd700',
                linewidth=2.0, alpha=0.95, label=f'Median ${eod_median:,.0f}')
ax_bot.axvline(last_price, color='#ffffff',
                linewidth=1.5, linestyle='--',
                alpha=0.7, label=f'Now ${last_price:,.2f}')

ax_bot.set_xlim(all_finals.min(), all_finals.max())
ax_bot.set_ylim(0, 1.15)
ax_bot.yaxis.set_visible(False)
ax_bot.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax_bot.set_title(
    f'15-Day Outcome Distribution   ·   '
    f'Green = bull scenarios   ·   '
    f'Red = bear scenarios   ·   '
    f'Vertical lines = cluster magnets   ·   '
    f'Bull prob: {bull_pct:.0f}%',
    color='#666688', fontsize=8, pad=3, loc='left'
)
ax_bot.legend(loc='upper right', fontsize=8,
              facecolor='#111122', edgecolor='#222233',
              labelcolor='white', framealpha=0.88)

plt.savefig(str(OUTDIR / 'gold_liquidity_forecast.png'),
            dpi=160, facecolor='#07070f', bbox_inches='tight')
print("Saved: /Users/elena_nael/gold_liquidity_forecast.png")
plt.show()


In [ ]:
# ── export this notebook to HTML ──────────────────────────────────────
# FIX: was a hardcoded interpreter (/opt/anaconda3/bin/jupyter) and a
# hardcoded notebook path. Both break on any other machine or env.
import subprocess, sys, glob, os
from pathlib import Path

_here = Path.cwd()
_nb = sorted(glob.glob(str(_here / "Daily_systematic_gold_macro_model*.ipynb")),
             key=os.path.getmtime)
if _nb:
    subprocess.run([sys.executable, "-m", "jupyter", "nbconvert",
                    "--to", "html", "--no-input", "--no-prompt", _nb[-1]])
    print(f"  exported {Path(_nb[-1]).name}")
else:
    print(f"  notebook not found in {_here} — run this from the folder it lives in")

In [ ]:
# ── portable output dir (was hardcoded /Users/elena_nael/...) ─────────
from pathlib import Path as _Path
OUTDIR = _Path.cwd() / "outputs"; OUTDIR.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from scipy.signal import find_peaks, argrelextrema
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# GOLD + DXY — TECHNICAL & SMC ANALYSIS
# ============================================================

print("Fetching data...")
gc  = yf.download('GC=F',  period='6mo', interval='1d', progress=False)
if hasattr(gc, 'columns') and isinstance(gc.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gc.columns = gc.columns.get_level_values(0)
dxy = yf.download('DX-Y.NYB', period='6mo', interval='1d', progress=False)
if hasattr(dxy, 'columns') and isinstance(dxy.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    dxy.columns = dxy.columns.get_level_values(0)

gc  = gc[['Open','High','Low','Close','Volume']].dropna()
dxy = dxy[['Open','High','Low','Close','Volume']].dropna()

# Align dates
common = gc.index.intersection(dxy.index)
gc  = gc.loc[common]
dxy = dxy.loc[common]

print(f"  GC=F  last: ${float(np.asarray(gc['Close'])[-1]):,.2f}")
print(f"  DXY   last: ${float(np.asarray(dxy['Close'])[-1]):,.4f}")

# ============================================================
# TECHNICAL INDICATORS
# ============================================================

def add_indicators(df):
    c = df['Close'].values.flatten()
    h = df['High'].values.flatten()
    l = df['Low'].values.flatten()

    # EMAs
    df['EMA20']  = df['Close'].ewm(span=20).mean()
    df['EMA50']  = df['Close'].ewm(span=50).mean()
    df['EMA200'] = df['Close'].ewm(span=200).mean()

    # Bollinger Bands
    sma20 = df['Close'].rolling(20).mean()
    std20 = df['Close'].rolling(20).std()
    df['BB_UP']  = sma20 + 2*std20
    df['BB_LO']  = sma20 - 2*std20
    df['BB_MID'] = sma20

    # RSI
    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / (loss + 1e-9)
    df['RSI'] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = df['Close'].ewm(span=12).mean()
    ema26 = df['Close'].ewm(span=26).mean()
    df['MACD']        = ema12 - ema26
    df['MACD_signal'] = df['MACD'].ewm(span=9).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']

    # ATR
    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift()).abs(),
        (df['Low']  - df['Close'].shift()).abs()
    ], axis=1).max(axis=1)
    df['ATR'] = tr.rolling(14).mean()

    # Volume SMA
    df['Vol_SMA'] = df['Volume'].rolling(20).mean()

    return df

gc  = add_indicators(gc)
dxy = add_indicators(dxy)

# ============================================================
# SMC ANALYSIS FUNCTIONS
# ============================================================

def find_swing_highs_lows(df, lookback=5):
    h = df['High'].values.flatten()
    l = df['Low'].values.flatten()

    sh_idx = argrelextrema(h, np.greater, order=lookback)[0]
    sl_idx = argrelextrema(l, np.less,    order=lookback)[0]

    swing_highs = [(df.index[i], h[i]) for i in sh_idx]
    swing_lows  = [(df.index[i], l[i]) for i in sl_idx]
    return swing_highs, swing_lows

def find_order_blocks(df, swing_highs, swing_lows, n=3):
    """
    Bullish OB: last bearish candle before a bullish impulse
    Bearish OB: last bullish candle before a bearish impulse
    """
    obs = []
    c   = df['Close'].values.flatten()
    o   = df['Open'].values.flatten()

    for i in range(3, len(df)-3):
        # Bullish OB: bearish candle followed by 3 bullish closes
        if c[i] < o[i]:  # bearish candle
            if all(c[i+k] > c[i+k-1] for k in range(1, 4)):
                obs.append({
                    'type'  : 'bullish',
                    'date'  : df.index[i],
                    'high'  : float(df['High'].iloc[i]),
                    'low'   : float(df['Low'].iloc[i]),
                    'open'  : float(o[i]),
                    'close' : float(c[i]),
                })
        # Bearish OB: bullish candle followed by 3 bearish closes
        if c[i] > o[i]:  # bullish candle
            if all(c[i+k] < c[i+k-1] for k in range(1, 4)):
                obs.append({
                    'type'  : 'bearish',
                    'date'  : df.index[i],
                    'high'  : float(df['High'].iloc[i]),
                    'low'   : float(df['Low'].iloc[i]),
                    'open'  : float(o[i]),
                    'close' : float(c[i]),
                })

    # Keep most recent n of each type
    bull_obs = [o for o in obs if o['type']=='bullish'][-n:]
    bear_obs = [o for o in obs if o['type']=='bearish'][-n:]
    return bull_obs + bear_obs

def find_fvg(df, min_gap_pct=0.001):
    """Fair Value Gaps (imbalances)"""
    fvgs = []
    h = df['High'].values.flatten()
    l = df['Low'].values.flatten()

    for i in range(1, len(df)-1):
        # Bullish FVG: low[i+1] > high[i-1]
        gap = l[i+1] - h[i-1]
        if gap > 0 and gap/h[i-1] > min_gap_pct:
            fvgs.append({'type':'bullish','top':l[i+1],'bot':h[i-1],
                          'date':df.index[i]})
        # Bearish FVG: high[i+1] < low[i-1]
        gap = l[i-1] - h[i+1]
        if gap > 0 and gap/l[i-1] > min_gap_pct:
            fvgs.append({'type':'bearish','top':l[i-1],'bot':h[i+1],
                          'date':df.index[i]})

    return fvgs[-6:]  # last 6 FVGs

def find_bos_choch(df, swing_highs, swing_lows):
    """Break of Structure & Change of Character"""
    signals = []
    c = df['Close'].values.flatten()

    # BOS: price closes above last swing high = bullish BOS
    for i in range(10, len(df)):
        recent_sh = [s[1] for s in swing_highs if s[0] < df.index[i]]
        recent_sl = [s[1] for s in swing_lows  if s[0] < df.index[i]]
        if not recent_sh or not recent_sl:
            continue

        last_sh = recent_sh[-1]
        last_sl = recent_sl[-1]

        if c[i] > last_sh and c[i-1] <= last_sh:
            signals.append({'type':'BOS_bull','date':df.index[i],
                             'price':c[i],'level':last_sh})
        if c[i] < last_sl and c[i-1] >= last_sl:
            signals.append({'type':'BOS_bear','date':df.index[i],
                             'price':c[i],'level':last_sl})

    return signals[-4:]

def find_liquidity_levels(df, swing_highs, swing_lows):
    """Equal highs/lows = liquidity pools"""
    pools = []
    tolerance = 0.002

    sh_prices = [s[1] for s in swing_highs[-10:]]
    sl_prices = [s[1] for s in swing_lows[-10:]]

    for i in range(len(sh_prices)):
        for j in range(i+1, len(sh_prices)):
            if abs(sh_prices[i]-sh_prices[j])/sh_prices[i] < tolerance:
                pools.append({'type':'sell_side','price':(sh_prices[i]+sh_prices[j])/2})

    for i in range(len(sl_prices)):
        for j in range(i+1, len(sl_prices)):
            if abs(sl_prices[i]-sl_prices[j])/sl_prices[i] < tolerance:
                pools.append({'type':'buy_side','price':(sl_prices[i]+sl_prices[j])/2})

    return pools

# Run SMC on GC
gc_sh, gc_sl = find_swing_highs_lows(gc)
gc_obs       = find_order_blocks(gc, gc_sh, gc_sl)
gc_fvgs      = find_fvg(gc)
gc_bos       = find_bos_choch(gc, gc_sh, gc_sl)
gc_liq       = find_liquidity_levels(gc, gc_sh, gc_sl)

# Run SMC on DXY
dx_sh, dx_sl = find_swing_highs_lows(dxy)
dx_obs       = find_order_blocks(dxy, dx_sh, dx_sl)
dx_fvgs      = find_fvg(dxy)
dx_bos       = find_bos_choch(dxy, dx_sh, dx_sl)
dx_liq       = find_liquidity_levels(dxy, dx_sh, dx_sl)

# ── Market bias summary ──────────────────────────────────────
def market_bias(df, bos_signals, obs):
    c   = df['Close'].values.flatten()
    e20 = df['EMA20'].values.flatten()
    e50 = df['EMA50'].values.flatten()
    rsi = df['RSI'].values.flatten()

    score = 0
    notes = []

    if c[-1] > e20[-1]: score += 1
    if c[-1] > e50[-1]: score += 1
    if e20[-1] > e50[-1]: score += 1
    if rsi[-1] > 55: score += 1; notes.append(f'RSI {rsi[-1]:.0f} bullish')
    if rsi[-1] < 45: score -= 1; notes.append(f'RSI {rsi[-1]:.0f} bearish')

    bull_bos = sum(1 for b in bos_signals if b['type']=='BOS_bull')
    bear_bos = sum(1 for b in bos_signals if b['type']=='BOS_bear')
    score += bull_bos - bear_bos

    if score >= 3:   bias = 'BULLISH';  col = '#00e676'
    elif score <= -2: bias = 'BEARISH'; col = '#ff4444'
    else:             bias = 'NEUTRAL'; col = '#ffaa00'

    return bias, col, score, notes

gc_bias, gc_bias_col, gc_score, gc_notes   = market_bias(gc,  gc_bos,  gc_obs)
dxy_bias, dxy_bias_col, dxy_score, dxy_notes = market_bias(dxy, dx_bos, dx_obs)

print(f"  GC  bias: {gc_bias}  (score {gc_score})")
print(f"  DXY bias: {dxy_bias} (score {dxy_score})")

# ============================================================
# FIGURE
# ============================================================
fig = plt.figure(figsize=(24, 20), facecolor='#07070f')
fig.patch.set_facecolor('#07070f')

gs = gridspec.GridSpec(
    5, 2, figure=fig,
    height_ratios=[2.8, 0.7, 0.7, 2.8, 0.7],
    hspace=0.06, wspace=0.06,
    left=0.05, right=0.98,
    top=0.96,  bottom=0.03
)

# GC panels
ax_gc    = fig.add_subplot(gs[0, 0])
ax_gc_rs = fig.add_subplot(gs[1, 0])
ax_gc_mc = fig.add_subplot(gs[2, 0])

# DXY panels
ax_dx    = fig.add_subplot(gs[0, 1])
ax_dx_rs = fig.add_subplot(gs[1, 1])
ax_dx_mc = fig.add_subplot(gs[2, 1])

# Correlation bottom
ax_corr  = fig.add_subplot(gs[3:, :])

BG = '#07070f'
all_axes = [ax_gc, ax_gc_rs, ax_gc_mc,
            ax_dx, ax_dx_rs, ax_dx_mc, ax_corr]
for ax in all_axes:
    ax.set_facecolor(BG)
    for sp in ax.spines.values():
        sp.set_color('#111122')
        sp.set_linewidth(0.5)
    ax.tick_params(colors='#444466', labelsize=8)

# ── candlestick helper ───────────────────────────────────────
def plot_candles(ax, df, n=80):
    sub = df.iloc[-n:].copy()
    sub = sub.reset_index()
    xvals = np.arange(len(sub))
    dates = sub.iloc[:, 0]

    for i, row in sub.iterrows():
        x  = xvals[i]
        o  = float(row['Open'])
        h  = float(row['High'])
        l  = float(row['Low'])
        c  = float(row['Close'])
        col = '#00e676' if c >= o else '#ff4444'
        ax.plot([x, x], [l, h], color=col, lw=0.8, alpha=0.9)
        body_h = max(abs(c-o), 0.01)
        rect = mpatches.Rectangle(
            (x-0.35, min(o,c)), 0.70, body_h,
            facecolor=col, edgecolor=col,
            linewidth=0.3, alpha=0.90
        )
        ax.add_patch(rect)

    # x-axis date labels
    step = max(1, len(sub)//8)
    ax.set_xticks(xvals[::step])
    ax.set_xticklabels(
        [d.strftime('%b %d') for d in dates[::step]],
        fontsize=7, color='#444466'
    )
    ax.set_xlim(-1, len(sub))
    return xvals, sub, dates

def get_x(df_full, df_sub, date):
    """Get x position from date"""
    try:
        idx = list(df_sub.iloc[:, 0]).index(date)
        return idx
    except:
        return None

# ============================================================
# GOLD CHART
# ============================================================
xvals_gc, sub_gc, dates_gc = plot_candles(ax_gc, gc)
n_gc = len(sub_gc)

def get_recent_series(df, col, n=80):
    return df[col].iloc[-n:].values.flatten()

# EMAs
ax_gc.plot(xvals_gc, get_recent_series(gc,'EMA20'), color='#00aaff',
           lw=1.2, alpha=0.85, label='EMA20')
ax_gc.plot(xvals_gc, get_recent_series(gc,'EMA50'), color='#ff9900',
           lw=1.2, alpha=0.85, label='EMA50')
ax_gc.plot(xvals_gc, get_recent_series(gc,'EMA200'), color='#ff4444',
           lw=1.0, alpha=0.7, label='EMA200', linestyle='--')

# Bollinger Bands
bb_up = get_recent_series(gc,'BB_UP')
bb_lo = get_recent_series(gc,'BB_LO')
bb_mi = get_recent_series(gc,'BB_MID')
ax_gc.fill_between(xvals_gc, bb_lo, bb_up,
                    color='#ffffff', alpha=0.03)
ax_gc.plot(xvals_gc, bb_up, color='#555577', lw=0.6, linestyle='--', alpha=0.7)
ax_gc.plot(xvals_gc, bb_lo, color='#555577', lw=0.6, linestyle='--', alpha=0.7)

# SMC: Order Blocks
for ob in gc_obs:
    try:
        xi = list(dates_gc).index(ob['date'])
    except:
        xi = None
    if xi is not None:
        col   = '#00ff88' if ob['type']=='bullish' else '#ff4444'
        alpha = 0.18
        ax_gc.axhspan(ob['low'], ob['high'], color=col, alpha=alpha,
                       xmin=xi/n_gc, xmax=1.0)
        ax_gc.text(xi, ob['high'], f" OB{'↑' if ob['type']=='bullish' else '↓'}",
                    color=col, fontsize=6.5, va='bottom',
                    path_effects=[pe.withStroke(linewidth=1.5, foreground='#07070f')])

# SMC: FVGs
for fvg in gc_fvgs:
    try:
        xi = list(dates_gc).index(fvg['date'])
    except:
        xi = None
    if xi is not None:
        col = '#0044ff' if fvg['type']=='bullish' else '#884400'
        ax_gc.axhspan(fvg['bot'], fvg['top'], color=col, alpha=0.20,
                       xmin=xi/n_gc, xmax=1.0)
        ax_gc.text(xi, (fvg['top']+fvg['bot'])/2,
                    f"  FVG{'↑' if fvg['type']=='bullish' else '↓'}",
                    color=col, fontsize=6, va='center',
                    path_effects=[pe.withStroke(linewidth=1.5, foreground='#07070f')])

# SMC: Swing highs/lows
for sh in gc_sh[-8:]:
    try:
        xi = list(dates_gc).index(sh[0])
        ax_gc.scatter([xi], [sh[1]], color='#ff4444', s=25,
                       marker='^', zorder=8, alpha=0.9)
    except: pass

for sl in gc_sl[-8:]:
    try:
        xi = list(dates_gc).index(sl[0])
        ax_gc.scatter([xi], [sl[1]], color='#00e676', s=25,
                       marker='v', zorder=8, alpha=0.9)
    except: pass

# SMC: BOS / CHoCH
for bos in gc_bos:
    try:
        xi = list(dates_gc).index(bos['date'])
        col = '#00ff88' if 'bull' in bos['type'] else '#ff4444'
        lbl = 'BOS↑' if 'bull' in bos['type'] else 'BOS↓'
        ax_gc.axvline(xi, color=col, lw=0.8, linestyle=':', alpha=0.7)
        ax_gc.text(xi, bos['level'], f' {lbl}',
                    color=col, fontsize=7, fontweight='bold',
                    path_effects=[pe.withStroke(linewidth=1.5, foreground='#07070f')])
    except: pass

# SMC: Liquidity pools
for liq in gc_liq:
    col = '#ffff00' if liq['type']=='sell_side' else '#00ffff'
    lbl = 'SSL' if liq['type']=='sell_side' else 'BSL'
    ax_gc.axhline(liq['price'], color=col, lw=0.8,
                   linestyle='-.', alpha=0.7)
    ax_gc.text(n_gc-1, liq['price'], f' {lbl} ${liq["price"]:,.0f}',
                color=col, fontsize=6.5, va='center',
                path_effects=[pe.withStroke(linewidth=1.5, foreground='#07070f')])

# Current price
last_gc = float(np.asarray(gc['Close'])[-1])
ax_gc.axhline(last_gc, color='#ffd700', lw=1.0, linestyle='--', alpha=0.7)
ax_gc.text(n_gc-1, last_gc, f' ${last_gc:,.2f}',
            color='#ffd700', fontsize=8, fontweight='bold', va='center')

ax_gc.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax_gc.set_ylabel('Gold (USD)', color='#ffd700', fontsize=9)
ax_gc.yaxis.grid(True, color='#0d0d1a', lw=0.4)
ax_gc.set_axisbelow(True)
ax_gc.legend(loc='upper left', fontsize=7,
              facecolor='#111122', edgecolor='#222233',
              labelcolor='white', framealpha=0.85)
ax_gc.set_title(
    f'GOLD (GC=F)   ${last_gc:,.2f}   '
    f'Bias: {gc_bias}   Score: {gc_score}',
    color=gc_bias_col, fontsize=11, fontweight='bold',
    pad=5, loc='left'
)

# SMC legend
smc_legend = [
    mpatches.Patch(color='#00ff88', alpha=0.5, label='Bullish OB'),
    mpatches.Patch(color='#ff4444', alpha=0.5, label='Bearish OB'),
    mpatches.Patch(color='#0044ff', alpha=0.5, label='Bull FVG'),
    mpatches.Patch(color='#884400', alpha=0.5, label='Bear FVG'),
    plt.Line2D([0],[0], color='#ffff00', lw=1.2, ls='-.', label='SSL'),
    plt.Line2D([0],[0], color='#00ffff', lw=1.2, ls='-.', label='BSL'),
]
ax_gc.legend(handles=smc_legend, loc='upper right', fontsize=6.5,
              facecolor='#111122', edgecolor='#222233',
              labelcolor='white', framealpha=0.88, ncol=3)

# ── RSI ──
rsi_gc = get_recent_series(gc,'RSI')
ax_gc_rs.plot(xvals_gc, rsi_gc, color='#aa88ff', lw=1.2)
ax_gc_rs.axhline(70, color='#ff4444', lw=0.6, linestyle='--', alpha=0.7)
ax_gc_rs.axhline(30, color='#00e676', lw=0.6, linestyle='--', alpha=0.7)
ax_gc_rs.axhline(50, color='#333355', lw=0.4)
ax_gc_rs.fill_between(xvals_gc, rsi_gc, 50,
                        where=rsi_gc>=50, color='#00e676', alpha=0.12)
ax_gc_rs.fill_between(xvals_gc, rsi_gc, 50,
                        where=rsi_gc<50,  color='#ff4444', alpha=0.12)
ax_gc_rs.set_ylim(0, 100)
ax_gc_rs.set_xlim(-1, n_gc)
ax_gc_rs.set_yticks([30,50,70])
ax_gc_rs.set_yticklabels(['30','50','70'], fontsize=6)
ax_gc_rs.set_xticklabels([])
ax_gc_rs.set_ylabel('RSI', color='#aa88ff', fontsize=7)
ax_gc_rs.text(n_gc-1, float(rsi_gc[-1]),
               f' {float(rsi_gc[-1]):.0f}',
               color='#aa88ff', fontsize=7, va='center')

# ── MACD ──
macd_gc  = get_recent_series(gc,'MACD')
sig_gc   = get_recent_series(gc,'MACD_signal')
hist_gc  = get_recent_series(gc,'MACD_hist')
hcols    = ['#00e676' if v >= 0 else '#ff4444' for v in hist_gc]
ax_gc_mc.bar(xvals_gc, hist_gc, color=hcols, alpha=0.7, width=0.8)
ax_gc_mc.plot(xvals_gc, macd_gc,  color='#00aaff', lw=1.0, label='MACD')
ax_gc_mc.plot(xvals_gc, sig_gc,   color='#ff9900', lw=1.0, label='Signal')
ax_gc_mc.axhline(0, color='#333355', lw=0.4)
ax_gc_mc.set_xlim(-1, n_gc)
ax_gc_mc.set_xticklabels([])
ax_gc_mc.set_ylabel('MACD', color='#888899', fontsize=7)

# ============================================================
# DXY CHART (same structure)
# ============================================================
xvals_dx, sub_dx, dates_dx = plot_candles(ax_dx, dxy)
n_dx = len(sub_dx)

ax_dx.plot(xvals_dx, get_recent_series(dxy,'EMA20'),
            color='#00aaff', lw=1.2, alpha=0.85, label='EMA20')
ax_dx.plot(xvals_dx, get_recent_series(dxy,'EMA50'),
            color='#ff9900', lw=1.2, alpha=0.85, label='EMA50')
ax_dx.plot(xvals_dx, get_recent_series(dxy,'EMA200'),
            color='#ff4444', lw=1.0, alpha=0.7, linestyle='--', label='EMA200')

bb_up_d = get_recent_series(dxy,'BB_UP')
bb_lo_d = get_recent_series(dxy,'BB_LO')
ax_dx.fill_between(xvals_dx, bb_lo_d, bb_up_d,
                    color='#ffffff', alpha=0.03)
ax_dx.plot(xvals_dx, bb_up_d, color='#555577', lw=0.6, linestyle='--', alpha=0.7)
ax_dx.plot(xvals_dx, bb_lo_d, color='#555577', lw=0.6, linestyle='--', alpha=0.7)

for ob in dx_obs:
    try:
        xi = list(dates_dx).index(ob['date'])
    except:
        xi = None
    if xi is not None:
        col = '#00ff88' if ob['type']=='bullish' else '#ff4444'
        ax_dx.axhspan(ob['low'], ob['high'], color=col, alpha=0.18,
                       xmin=xi/n_dx, xmax=1.0)
        ax_dx.text(xi, ob['high'], f" OB{'↑' if ob['type']=='bullish' else '↓'}",
                    color=col, fontsize=6.5, va='bottom',
                    path_effects=[pe.withStroke(linewidth=1.5, foreground='#07070f')])

for fvg in dx_fvgs:
    try:
        xi = list(dates_dx).index(fvg['date'])
    except:
        xi = None
    if xi is not None:
        col = '#0044ff' if fvg['type']=='bullish' else '#884400'
        ax_dx.axhspan(fvg['bot'], fvg['top'], color=col, alpha=0.20,
                       xmin=xi/n_dx, xmax=1.0)
        ax_dx.text(xi, (fvg['top']+fvg['bot'])/2,
                    f"  FVG{'↑' if fvg['type']=='bullish' else '↓'}",
                    color=col, fontsize=6, va='center',
                    path_effects=[pe.withStroke(linewidth=1.5, foreground='#07070f')])

for sh in dx_sh[-8:]:
    try:
        xi = list(dates_dx).index(sh[0])
        ax_dx.scatter([xi], [sh[1]], color='#ff4444', s=25, marker='^', zorder=8)
    except: pass

for sl in dx_sl[-8:]:
    try:
        xi = list(dates_dx).index(sl[0])
        ax_dx.scatter([xi], [sl[1]], color='#00e676', s=25, marker='v', zorder=8)
    except: pass

for bos in dx_bos:
    try:
        xi = list(dates_dx).index(bos['date'])
        col = '#00ff88' if 'bull' in bos['type'] else '#ff4444'
        lbl = 'BOS↑' if 'bull' in bos['type'] else 'BOS↓'
        ax_dx.axvline(xi, color=col, lw=0.8, linestyle=':', alpha=0.7)
        ax_dx.text(xi, bos['level'], f' {lbl}',
                    color=col, fontsize=7, fontweight='bold',
                    path_effects=[pe.withStroke(linewidth=1.5, foreground='#07070f')])
    except: pass

for liq in dx_liq:
    col = '#ffff00' if liq['type']=='sell_side' else '#00ffff'
    lbl = 'SSL' if liq['type']=='sell_side' else 'BSL'
    ax_dx.axhline(liq['price'], color=col, lw=0.8, linestyle='-.', alpha=0.7)
    ax_dx.text(n_dx-1, liq['price'], f' {lbl} {liq["price"]:.3f}',
                color=col, fontsize=6.5, va='center',
                path_effects=[pe.withStroke(linewidth=1.5, foreground='#07070f')])

last_dx = float(np.asarray(dxy['Close'])[-1])
ax_dx.axhline(last_dx, color='#00ccff', lw=1.0, linestyle='--', alpha=0.7)
ax_dx.text(n_dx-1, last_dx, f' {last_dx:.3f}',
            color='#00ccff', fontsize=8, fontweight='bold', va='center')

ax_dx.set_ylabel('DXY', color='#00ccff', fontsize=9)
ax_dx.yaxis.grid(True, color='#0d0d1a', lw=0.4)
ax_dx.set_axisbelow(True)
ax_dx.legend(handles=smc_legend, loc='upper right', fontsize=6.5,
              facecolor='#111122', edgecolor='#222233',
              labelcolor='white', framealpha=0.88, ncol=3)
ax_dx.set_title(
    f'DXY (Dollar Index)   {last_dx:.3f}   '
    f'Bias: {dxy_bias}   Score: {dxy_score}',
    color=dxy_bias_col, fontsize=11, fontweight='bold',
    pad=5, loc='left'
)

rsi_dx = get_recent_series(dxy,'RSI')
ax_dx_rs.plot(xvals_dx, rsi_dx, color='#aa88ff', lw=1.2)
ax_dx_rs.axhline(70, color='#ff4444', lw=0.6, linestyle='--', alpha=0.7)
ax_dx_rs.axhline(30, color='#00e676', lw=0.6, linestyle='--', alpha=0.7)
ax_dx_rs.axhline(50, color='#333355', lw=0.4)
ax_dx_rs.fill_between(xvals_dx, rsi_dx, 50,
                        where=rsi_dx>=50, color='#00e676', alpha=0.12)
ax_dx_rs.fill_between(xvals_dx, rsi_dx, 50,
                        where=rsi_dx<50,  color='#ff4444', alpha=0.12)
ax_dx_rs.set_ylim(0, 100)
ax_dx_rs.set_xlim(-1, n_dx)
ax_dx_rs.set_yticks([30,50,70])
ax_dx_rs.set_yticklabels(['30','50','70'], fontsize=6)
ax_dx_rs.set_xticklabels([])
ax_dx_rs.set_ylabel('RSI', color='#aa88ff', fontsize=7)
ax_dx_rs.text(n_dx-1, float(rsi_dx[-1]),
               f' {float(rsi_dx[-1]):.0f}',
               color='#aa88ff', fontsize=7, va='center')

macd_dx  = get_recent_series(dxy,'MACD')
sig_dx   = get_recent_series(dxy,'MACD_signal')
hist_dx  = get_recent_series(dxy,'MACD_hist')
hcols_dx = ['#00e676' if v >= 0 else '#ff4444' for v in hist_dx]
ax_dx_mc.bar(xvals_dx, hist_dx, color=hcols_dx, alpha=0.7, width=0.8)
ax_dx_mc.plot(xvals_dx, macd_dx, color='#00aaff', lw=1.0)
ax_dx_mc.plot(xvals_dx, sig_dx,  color='#ff9900', lw=1.0)
ax_dx_mc.axhline(0, color='#333355', lw=0.4)
ax_dx_mc.set_xlim(-1, n_dx)
ax_dx_mc.set_xticklabels([])
ax_dx_mc.set_ylabel('MACD', color='#888899', fontsize=7)

# ============================================================
# BOTTOM — GC vs DXY correlation + summary
# ============================================================
gc_ret    = gc['Close'].squeeze().pct_change().iloc[-60:].values.flatten()
dx_ret    = dxy['Close'].squeeze().pct_change().iloc[-60:].values.flatten()
gc_close  = gc['Close'].squeeze().pct_change()
dx_close  = dxy['Close'].squeeze().pct_change()
roll_corr = gc_close.rolling(20).corr(dx_close).iloc[-60:].values.flatten()
xc = np.arange(len(roll_corr))

ax_corr.fill_between(xc, roll_corr, 0,
                      where=roll_corr >= 0,
                      color='#ff4444', alpha=0.4,
                      label='Positive corr (unusual)')
ax_corr.fill_between(xc, roll_corr, 0,
                      where=roll_corr < 0,
                      color='#00aaff', alpha=0.4,
                      label='Negative corr (normal)')
ax_corr.plot(xc, roll_corr, color='#ffffff', lw=1.2, alpha=0.8)
ax_corr.axhline(0,    color='#333355', lw=0.6)
ax_corr.axhline(-0.5, color='#00aaff', lw=0.5, linestyle='--', alpha=0.5)
ax_corr.axhline( 0.5, color='#ff4444', lw=0.5, linestyle='--', alpha=0.5)
ax_corr.set_xlim(0, len(roll_corr)-1)
ax_corr.set_ylim(-1.1, 1.1)

step = max(1, len(gc.index[-60:])//8)
xt   = list(range(0, len(roll_corr), step))
ax_corr.set_xticks(xt)
ax_corr.set_xticklabels(
    [gc.index[-60:][i].strftime('%b %d') for i in xt],
    fontsize=7, color='#444466'
)
ax_corr.set_ylabel('GC / DXY\n20d Corr', color='#aaaacc', fontsize=8)
ax_corr.set_title(
    f'GC vs DXY  20-Day Rolling Correlation   ·   '
    f'Current: {float(roll_corr[-1]):.2f}   ·   '
    f'Normal = negative (gold rises when dollar falls)',
    color='#666688', fontsize=9, pad=3, loc='left'
)
ax_corr.legend(loc='upper right', fontsize=8,
               facecolor='#111122', edgecolor='#222233',
               labelcolor='white', framealpha=0.88)
ax_corr.yaxis.grid(True, color='#0d0d1a', lw=0.4)

# Summary box
corr_now = float(roll_corr[-1])
corr_sig = 'INVERTED (normal)' if corr_now < -0.3 else \
           ('NEUTRAL' if corr_now < 0.3 else 'POSITIVE (unusual!)')

summary = (
    f"  ANALYSIS SUMMARY\n"
    f"  {'─'*28}\n"
    f"  Gold  {last_gc:>9,.2f}   {gc_bias:<8}\n"
    f"  DXY   {last_dx:>9.3f}   {dxy_bias:<8}\n"
    f"  GC/DXY corr:  {corr_now:+.2f}  {corr_sig}\n"
    f"  {'─'*28}\n"
    f"  SMC — GOLD\n"
    f"  OBs: {len(gc_obs)}  FVGs: {len(gc_fvgs)}  "
    f"BOS: {len(gc_bos)}  Liq: {len(gc_liq)}\n"
    f"  SMC — DXY\n"
    f"  OBs: {len(dx_obs)}  FVGs: {len(dx_fvgs)}  "
    f"BOS: {len(dx_bos)}  Liq: {len(dx_liq)}\n"
    f"  {'─'*28}\n"
    f"  LEGEND\n"
    f"  OB  = Order Block\n"
    f"  FVG = Fair Value Gap\n"
    f"  BOS = Break of Structure\n"
    f"  SSL = Sell-Side Liquidity\n"
    f"  BSL = Buy-Side Liquidity\n"
    f"  ▲/▼ = Swing High/Low"
)

ax_corr.text(0.995, 0.97, summary,
             transform=ax_corr.transAxes,
             color='#ccccee', fontsize=8,
             va='top', ha='right',
             fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.7',
                       facecolor='#0a0a1a',
                       edgecolor='#222233',
                       alpha=0.94))

plt.savefig(str(OUTDIR / 'gold_dxy_smc_analysis.png'),
            dpi=150, facecolor='#07070f', bbox_inches='tight')
print("Saved: /Users/elena_nael/gold_dxy_smc_analysis.png")
plt.show()


In [ ]:
# ── portable output dir (was hardcoded /Users/elena_nael/...) ─────────
from pathlib import Path as _Path
OUTDIR = _Path.cwd() / "outputs"; OUTDIR.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import gaussian_filter1d
import yfinance as yf
import warnings, requests, json, time
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)

print("=" * 65)
print("  GOLD DEEP FACTOR ANALYSIS")
print("  What is ACTUALLY driving Gold right now?")
print("=" * 65)

# ============================================================
# 1. FETCH ALL MARKET DATA
# ============================================================
print("\n[1/6] Fetching market data...")

tickers = {
    'GC=F'   : 'Gold',
    'DX-Y.NYB':'DXY (Dollar)',
    'CL=F'   : 'Oil (WTI)',
    'SI=F'   : 'Silver',
    'GDX'    : 'Gold Miners ETF',
    '^TNX'   : '10Y Yield',
    '^IRX'   : '3M Yield',
    'TIP'    : 'TIPS (Real Rates)',
    '^VIX'   : 'VIX (Fear)',
    'SPY'    : 'S&P 500',
    'BTC-USD': 'Bitcoin',
    'USO'    : 'Oil ETF',
    '^GSPC'  : 'S&P500 Index',
    'GLD'    : 'Gold ETF (retail)',
    'IAU'    : 'Gold ETF (institutional)',
    'PALL'   : 'Palladium',
    'PL=F'   : 'Platinum',
    'HG=F'   : 'Copper',
    'UUP'    : 'Dollar ETF',
    'FXE'    : 'Euro ETF',
}

raw = {}
for t, name in tickers.items():
    try:
        d = yf.download(t, period='1y', interval='1d', progress=False)
        if hasattr(d, 'columns') and isinstance(d.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
            d.columns = d.columns.get_level_values(0)
        if len(d) > 50:
            raw[t] = d['Close'].squeeze()
            print(f"  OK  {name:<28} ({len(d)} days)")
        else:
            print(f"  --  {name:<28} (insufficient data)")
    except:
        print(f"  XX  {name:<28} (failed)")
    time.sleep(0.1)

# ============================================================
# 2. FETCH MACRO INDICATORS VIA FRED (free, no key needed)
# ============================================================
print("\n[2/6] Fetching macro indicators (FRED)...")

def fetch_fred(series_id, label):
    try:
        url = (f"https://fred.stlouisfed.org/graph/fredgraph.csv"
               f"?id={series_id}")
        df = pd.read_csv(url, index_col=0, parse_dates=True)
        df.columns = [label]
        df = df[df[label] != '.']
        df[label] = pd.to_numeric(df[label], errors='coerce')
        df = df.dropna()
        print(f"  OK  {label}")
        return df[label]
    except Exception as e:
        print(f"  XX  {label}: {e}")
        return None

fred_series = {
    'CPIAUCSL'  : 'CPI Inflation',
    'DFF'       : 'Fed Funds Rate',
    'T10YIE'    : '10Y Breakeven Inflation',
    'DFII10'    : '10Y Real Yield',
    'DTWEXBGS'  : 'Trade-Weighted Dollar',
    'BAMLH0A0HYM2': 'HY Credit Spread',
    'VIXCLS'    : 'VIX (FRED)',
    'M2SL'      : 'M2 Money Supply',
    'DCOILWTICO': 'WTI Oil Price',
    'GOLDAMGBD228NLBM': 'Gold Fix (LBMA)',
}

macro = {}
for sid, lbl in fred_series.items():
    s = fetch_fred(sid, lbl)
    if s is not None:
        macro[lbl] = s

# ============================================================
# 3. GOOGLE TRENDS PROXY (fear/sentiment keywords)
# ============================================================
print("\n[3/6] Building sentiment proxies...")

# Use VIX as fear proxy, GLD volume as retail demand proxy
gold_df = raw.get('GC=F', None)
gld_df  = yf.download('GLD', period='1y', interval='1d', progress=False)
if hasattr(gld_df, 'columns') and isinstance(gld_df.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gld_df.columns = gld_df.columns.get_level_values(0)
iau_df  = yf.download('IAU', period='1y', interval='1d', progress=False)
if hasattr(iau_df, 'columns') and isinstance(iau_df.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    iau_df.columns = iau_df.columns.get_level_values(0)

# Retail vs institutional demand split
# GLD = retail ETF flows, IAU = institutional
gld_vol = gld_df['Volume'].squeeze() if len(gld_df) > 0 else None
iau_vol = iau_df['Volume'].squeeze() if len(iau_df) > 0 else None

print("  Retail demand proxy (GLD volume): OK")
print("  Institutional demand proxy (IAU volume): OK")

# ============================================================
# 4. CORRELATION ENGINE
# ============================================================
print("\n[4/6] Running correlation analysis...")

gc_price = raw['GC=F'].resample('D').last().ffill()
gc_ret   = gc_price.pct_change().dropna()

factors = {}

# ── Market factors ───────────────────────────────────────────
market_map = {
    'DX-Y.NYB' : 'Dollar Index (DXY)',
    'CL=F'     : 'Oil (WTI)',
    '^TNX'     : '10Y Treasury Yield',
    '^IRX'     : '3M Treasury Yield',
    'TIP'      : 'TIPS / Real Rates',
    '^VIX'     : 'VIX Fear Index',
    'SPY'      : 'S&P 500 (Risk-On)',
    'SI=F'     : 'Silver (Metals Complex)',
    'HG=F'     : 'Copper (Global Growth)',
    'BTC-USD'  : 'Bitcoin (Alt Asset)',
    'PALL'     : 'Palladium',
    'PL=F'     : 'Platinum',
    'GDX'      : 'Gold Miners (GDX)',
    'GLD'      : 'Retail Gold ETF Flow',
    'IAU'      : 'Institutional Gold ETF',
    'UUP'      : 'Dollar ETF',
    'FXE'      : 'Euro vs Dollar',
}

for ticker, label in market_map.items():
    if ticker in raw:
        s  = raw[ticker].resample('D').last().ffill()
        sr = s.pct_change().dropna()
        common = gc_ret.index.intersection(sr.index)
        if len(common) > 60:
            gc_c = gc_ret.loc[common].values
            sr_c = sr.loc[common].values
            mask = ~(np.isnan(gc_c) | np.isnan(sr_c))
            if mask.sum() > 60:
                corr_p, pval_p = pearsonr(gc_c[mask], sr_c[mask])
                corr_s, _      = spearmanr(gc_c[mask], sr_c[mask])

                # Recent 30-day correlation
                common30 = common[-30:]
                gc30 = gc_ret.loc[common30].values
                sr30 = sr.loc[common30].values
                m30  = ~(np.isnan(gc30)|np.isnan(sr30))
                corr_30 = pearsonr(gc30[m30], sr30[m30])[0] if m30.sum() > 10 else np.nan

                factors[label] = {
                    'corr_1y'  : corr_p,
                    'corr_30d' : corr_30,
                    'pval'     : pval_p,
                    'spearman' : corr_s,
                    'type'     : 'market',
                }

# ── Macro factors (FRED — level correlation with gold price) ─
gc_monthly = gc_price.resample('MS').last().ffill()
for lbl, s in macro.items():
    sm = s.resample('MS').last().ffill()
    common = gc_monthly.index.intersection(sm.index)
    if len(common) > 8:
        gc_c = gc_monthly.loc[common].values.flatten()
        sm_c = sm.loc[common].values.flatten()
        mask = ~(np.isnan(gc_c)|np.isnan(sm_c))
        if mask.sum() > 6:
            try:
                corr_p, pval_p = pearsonr(gc_c[mask], sm_c[mask])
                factors[lbl] = {
                    'corr_1y'  : corr_p,
                    'corr_30d' : corr_p,
                    'pval'     : pval_p,
                    'spearman' : corr_p,
                    'type'     : 'macro',
                }
            except: pass

# ── Derived factors ──────────────────────────────────────────
# Real yield = 10Y - CPI
if '^TNX' in raw and 'T10YIE' in macro:
    tnx = raw['^TNX'].resample('MS').last().ffill()
    bei = macro['T10YIE'].resample('MS').last().ffill()
    common = tnx.index.intersection(bei.index).intersection(gc_monthly.index)
    if len(common) > 6:
        real_y = (tnx.loc[common] - bei.loc[common]).values.flatten()
        gc_c   = gc_monthly.loc[common].values.flatten()
        mask   = ~(np.isnan(real_y)|np.isnan(gc_c))
        if mask.sum() > 5:
            corr_p, pval_p = pearsonr(gc_c[mask], real_y[mask])
            factors['Real Yield (10Y - Inflation)'] = {
                'corr_1y' : corr_p, 'corr_30d': corr_p,
                'pval': pval_p, 'spearman': corr_p, 'type': 'derived'
            }

# Yield curve spread
if '^TNX' in raw and '^IRX' in raw:
    tnx = raw['^TNX'].resample('D').last().ffill()
    irx = raw['^IRX'].resample('D').last().ffill()
    common = tnx.index.intersection(irx.index).intersection(gc_ret.index)
    if len(common) > 60:
        spread_r = tnx.loc[common].pct_change().dropna()
        irx_r    = irx.loc[common].pct_change().dropna()
        yc_spread = (spread_r - irx_r).dropna()
        gc_c      = gc_ret.loc[yc_spread.index].dropna()
        common2   = yc_spread.index.intersection(gc_c.index)
        if len(common2) > 40:
            corr_p, pval_p = pearsonr(
                gc_c.loc[common2].values, yc_spread.loc[common2].values)
            factors['Yield Curve (10Y-3M)'] = {
                'corr_1y': corr_p, 'corr_30d': corr_p,
                'pval': pval_p, 'spearman': corr_p, 'type': 'derived'
            }

# Retail demand pressure (GLD volume z-score)
if gld_vol is not None:
    gld_z  = (gld_vol - gld_vol.rolling(30).mean()) / gld_vol.rolling(30).std()
    gld_zr = gld_z.resample('D').last().ffill()
    common = gc_ret.index.intersection(gld_zr.index)
    if len(common) > 60:
        gc_c  = gc_ret.loc[common].values
        gld_c = gld_zr.loc[common].values
        mask  = ~(np.isnan(gc_c)|np.isnan(gld_c))
        if mask.sum() > 40:
            corr_p, pval_p = pearsonr(gc_c[mask], gld_c[mask])
            factors['Retail Buying (GLD ETF Flow)'] = {
                'corr_1y': corr_p, 'corr_30d': corr_p,
                'pval': pval_p, 'spearman': corr_p, 'type': 'sentiment'
            }

# Institutional demand (IAU volume)
if iau_vol is not None:
    iau_z  = (iau_vol - iau_vol.rolling(30).mean()) / iau_vol.rolling(30).std()
    iau_zr = iau_z.resample('D').last().ffill()
    common = gc_ret.index.intersection(iau_zr.index)
    if len(common) > 60:
        gc_c  = gc_ret.loc[common].values
        iau_c = iau_zr.loc[common].values
        mask  = ~(np.isnan(gc_c)|np.isnan(iau_c))
        if mask.sum() > 40:
            corr_p, pval_p = pearsonr(gc_c[mask], iau_c[mask])
            factors['Institutional Buying (IAU ETF Flow)'] = {
                'corr_1y': corr_p, 'corr_30d': corr_p,
                'pval': pval_p, 'spearman': corr_p, 'type': 'sentiment'
            }

print(f"  Total factors analyzed: {len(factors)}")

# ============================================================
# 5. RANK FACTORS BY CURRENT (30-DAY) INFLUENCE
# ============================================================
print("\n[5/6] Ranking factors...")

df_factors = pd.DataFrame(factors).T
df_factors['abs_30d'] = df_factors['corr_30d'].abs()
df_factors['abs_1y']  = df_factors['corr_1y'].abs()
df_factors = df_factors.dropna(subset=['corr_30d'])
df_factors = df_factors.sort_values('abs_30d', ascending=False)

# Significance filter
df_sig = df_factors[df_factors['pval'] < 0.10].copy()

print(f"\n  TOP 10 FACTORS DRIVING GOLD RIGHT NOW:")
print(f"  {'Factor':<38} {'30d Corr':>8} {'1Y Corr':>8} {'Signal'}")
print(f"  {'-'*70}")
for name, row in df_sig.head(10).iterrows():
    c30  = row['corr_30d']
    c1y  = row['corr_1y']
    sig  = '↑ BULL' if c30 > 0.3 else ('↓ BEAR' if c30 < -0.3 else '~ NEUT')
    print(f"  {name:<38} {c30:>+8.3f} {c1y:>+8.3f}  {sig}")

# ============================================================
# 6. NARRATIVE INTERPRETATION
# ============================================================
print("\n[6/6] Building narrative...")

def interpret_factor(name, c30, c1y):
    """Return human-readable interpretation"""
    inv = c30 < 0

    interpretations = {
        'Dollar Index (DXY)': {
            True : "Dollar WEAKENING → Gold rising (inverse confirmed)",
            False: "Dollar STRENGTHENING but Gold still rising (decoupling!)"
        },
        'VIX Fear Index': {
            True : "Fear/volatility is NOT driving gold currently",
            False: "Market FEAR is boosting gold (safe haven demand active)"
        },
        'S&P 500 (Risk-On)': {
            True : "Stocks falling → gold rising (flight to safety)",
            False: "Stocks AND gold rising (liquidity/inflation trade)"
        },
        '10Y Treasury Yield': {
            True : "Rising yields hurting gold (opportunity cost)",
            False: "Gold rising WITH yields (inflation fears dominant)"
        },
        '10Y Breakeven Inflation': {
            False: "Inflation expectations DRIVING gold higher",
            True : "Inflation expectations not the main driver"
        },
        'Real Yield (10Y - Inflation)': {
            True : "Falling real yields → gold's STRONGEST driver",
            False: "Rising real yields — gold resilient despite headwind"
        },
        'Oil (WTI)': {
            False: "Oil rising WITH gold → commodity supercycle / inflation",
            True : "Oil falling, gold rising → pure safe haven demand"
        },
        'Bitcoin (Alt Asset)': {
            False: "Crypto AND gold rising → broad hard asset demand",
            True : "Crypto falling, gold rising → quality flight"
        },
        'M2 Money Supply': {
            False: "Money supply expansion fueling gold",
            True : "M2 contraction — gold rising on other factors"
        },
        'Silver (Metals Complex)': {
            False: "Metals complex rallying together → broad demand",
            True : "Gold outperforming silver → pure safe haven"
        },
        'Retail Buying (GLD ETF Flow)': {
            False: "Retail investors BUYING gold ETFs actively",
            True : "Retail selling but gold rising → institutional driven"
        },
        'Institutional Buying (IAU ETF Flow)': {
            False: "Institutions ACCUMULATING gold",
            True : "Institutions reducing — gold driven by other flows"
        },
        'Fed Funds Rate': {
            True : "High rates headwind for gold — rising anyway",
            False: "Rate expectations easing → tailwind for gold"
        },
        '10Y Real Yield': {
            True : "Real yields falling → MAJOR gold tailwind",
            False: "Real yields rising — gold resilient"
        },
        'HY Credit Spread': {
            False: "Credit stress rising → safe haven gold demand",
            True : "Credit calm — not a fear-driven gold rally"
        },
    }

    for key, msgs in interpretations.items():
        if key.lower() in name.lower() or name.lower() in key.lower():
            return msgs.get(inv, f"Correlation {c30:+.2f}")
    return f"30d corr {c30:+.2f} vs 1Y {c1y:+.2f}"

# ============================================================
# FIGURE
# ============================================================
print("\nBuilding chart...")
fig = plt.figure(figsize=(22, 26), facecolor='#07070f')
fig.patch.set_facecolor('#07070f')

gs = gridspec.GridSpec(
    4, 2, figure=fig,
    height_ratios=[0.08, 1.6, 1.6, 2.2],
    hspace=0.10, wspace=0.10,
    left=0.04, right=0.97,
    top=0.97, bottom=0.03
)

ax_title  = fig.add_subplot(gs[0, :])
ax_rank   = fig.add_subplot(gs[1, :])
ax_heat   = fig.add_subplot(gs[2, 0])
ax_corr   = fig.add_subplot(gs[2, 1])
ax_interp = fig.add_subplot(gs[3, :])

BG = '#07070f'
for ax in [ax_title, ax_rank, ax_heat, ax_corr, ax_interp]:
    ax.set_facecolor(BG)
    for sp in ax.spines.values():
        sp.set_color('#111122')
        sp.set_linewidth(0.5)
    ax.tick_params(colors='#444466', labelsize=8)

# ── TITLE ────────────────────────────────────────────────────
ax_title.axis('off')
gc_last = float(np.asarray(raw['GC=F'])[-1])
ax_title.text(
    0.5, 0.5,
    f'GOLD  DEEP  FACTOR  ANALYSIS   ·   '
    f'What is ACTUALLY driving gold?   ·   '
    f'${gc_last:,.2f}   ·   {pd.Timestamp.now().strftime("%B %d, %Y")}',
    transform=ax_title.transAxes,
    color='#ffd700', fontsize=15, fontweight='bold',
    va='center', ha='center',
    path_effects=[pe.withStroke(linewidth=5, foreground='#332200')]
)

# ── RANK BAR CHART ───────────────────────────────────────────
top_n  = min(20, len(df_sig))
top_df = df_sig.head(top_n).copy()
names  = [n[:42] for n in top_df.index]
c30s   = top_df['corr_30d'].values.astype(float)
c1ys   = top_df['corr_1y'].values.astype(float)
y_pos  = np.arange(len(names))

bar_colors = []
for c in c30s:
    if   c >  0.5: bar_colors.append('#00ff88')
    elif c >  0.2: bar_colors.append('#00cc66')
    elif c > -0.2: bar_colors.append('#888888')
    elif c > -0.5: bar_colors.append('#ff6600')
    else:          bar_colors.append('#ff2244')

bars = ax_rank.barh(y_pos, c30s, color=bar_colors,
                     alpha=0.85, height=0.55, zorder=3)
ax_rank.scatter(c1ys, y_pos, color='#ffffff',
                s=40, zorder=5, marker='D',
                alpha=0.8, label='1-Year corr ◆')

ax_rank.axvline(0, color='#333355', lw=1.0)
ax_rank.axvline( 0.5, color='#00ff88', lw=0.5, linestyle='--', alpha=0.5)
ax_rank.axvline(-0.5, color='#ff2244', lw=0.5, linestyle='--', alpha=0.5)
ax_rank.axvline( 0.3, color='#00cc66', lw=0.4, linestyle=':', alpha=0.4)
ax_rank.axvline(-0.3, color='#ff6600', lw=0.4, linestyle=':', alpha=0.4)

ax_rank.set_yticks(y_pos)
ax_rank.set_yticklabels(names, fontsize=9, color='#ccccdd')
ax_rank.set_xlim(-1.05, 1.05)
ax_rank.set_xlabel('Correlation with Gold Returns (−1 = inverse, +1 = direct)',
                    color='#888899', fontsize=9)
ax_rank.xaxis.grid(True, color='#0d0d1a', lw=0.5)
ax_rank.set_axisbelow(True)
ax_rank.set_title(
    'FACTOR RANKING   ·   Sorted by 30-Day Correlation Strength   ·   '
    'Bar = 30d   ◆ = 1-Year   Green = positive driver   Red = inverse driver',
    color='#888899', fontsize=10, pad=6, loc='left'
)
ax_rank.legend(loc='lower right', fontsize=8,
               facecolor='#111122', edgecolor='#222233',
               labelcolor='white', framealpha=0.9)

# Value labels on bars
for i, (bar, val) in enumerate(zip(bars, c30s)):
    col = '#00ff88' if val > 0 else '#ff4444'
    ax_rank.text(val + (0.02 if val >= 0 else -0.02), i,
                 f'{val:+.2f}', va='center',
                 ha='left' if val >= 0 else 'right',
                 color=col, fontsize=7.5, fontweight='bold')

# ── ROLLING CORRELATION HEATMAP ──────────────────────────────
top5 = df_sig.head(5).index.tolist()
roll_matrix = {}

for name in top5:
    # Find matching ticker
    ticker_match = None
    for t, lbl in market_map.items():
        if lbl == name and t in raw:
            ticker_match = t
            break
    if ticker_match:
        s      = raw[ticker_match].resample('D').last().ffill()
        sr     = s.pct_change().dropna()
        common = gc_ret.index.intersection(sr.index)
        if len(common) > 60:
            roll = gc_ret.loc[common].rolling(20).corr(
                sr.loc[common]).dropna()
            roll_matrix[name[:30]] = roll

if roll_matrix:
    roll_df  = pd.DataFrame(roll_matrix).dropna()
    roll_arr = roll_df.values.T
    im = ax_heat.imshow(
        roll_arr, aspect='auto', cmap='RdYlGn',
        vmin=-1, vmax=1, interpolation='nearest'
    )
    plt.colorbar(im, ax=ax_heat, shrink=0.8,
                 label='Rolling 20d Correlation')
    ax_heat.set_yticks(range(len(roll_df.columns)))
    ax_heat.set_yticklabels(roll_df.columns, fontsize=8, color='#ccccdd')
    step = max(1, len(roll_df)//6)
    ax_heat.set_xticks(range(0, len(roll_df), step))
    ax_heat.set_xticklabels(
        [d.strftime('%b %d') for d in roll_df.index[::step]],
        fontsize=7, color='#444466', rotation=30
    )
    ax_heat.set_title('ROLLING CORRELATION HEATMAP (Top 5 factors, 20-day window)',
                       color='#888899', fontsize=9, pad=4, loc='left')
else:
    ax_heat.text(0.5, 0.5, 'Insufficient data for heatmap',
                 transform=ax_heat.transAxes, color='#666688',
                 ha='center', va='center')

# ── ROLLING CORR LINES (top 4 factors) ──────────────────────
colors_line = ['#ffd700','#00ff88','#ff4444','#00aaff','#ff9900']
for i, name in enumerate(top5[:4]):
    ticker_match = None
    for t, lbl in market_map.items():
        if lbl == name and t in raw:
            ticker_match = t
            break
    if ticker_match:
        s      = raw[ticker_match].resample('D').last().ffill()
        sr     = s.pct_change().dropna()
        common = gc_ret.index.intersection(sr.index)
        if len(common) > 60:
            roll = gc_ret.loc[common].rolling(30).corr(
                sr.loc[common]).dropna()
            xr   = np.arange(len(roll))
            ax_corr.plot(xr, roll.values,
                         color=colors_line[i], lw=1.4,
                         alpha=0.9, label=name[:30])

ax_corr.axhline(0,    color='#333355', lw=0.8)
ax_corr.axhline( 0.5, color='#00ff88', lw=0.5, linestyle='--', alpha=0.5)
ax_corr.axhline(-0.5, color='#ff4444', lw=0.5, linestyle='--', alpha=0.5)
ax_corr.set_ylim(-1.1, 1.1)
ax_corr.set_title('ROLLING 30-DAY CORRELATION  (Top 4 factors over time)',
                   color='#888899', fontsize=9, pad=4, loc='left')
ax_corr.legend(fontsize=7.5, facecolor='#111122', edgecolor='#222233',
               labelcolor='white', framealpha=0.9)
ax_corr.yaxis.grid(True, color='#0d0d1a', lw=0.4)
ax_corr.axhspan( 0.3,  1.1, color='#00ff88', alpha=0.03)
ax_corr.axhspan(-1.1, -0.3, color='#ff4444', alpha=0.03)

# ── INTERPRETATION PANEL ─────────────────────────────────────
ax_interp.axis('off')
ax_interp.set_facecolor('#060610')
for sp in ax_interp.spines.values():
    sp.set_color('#1a1a33')

# Build full narrative
categories = {
    'MONETARY / MACRO':    ['macro','derived'],
    'MARKET FORCES':       ['market'],
    'SENTIMENT / FLOWS':   ['sentiment'],
}

cat_colors = {
    'MONETARY / MACRO'  : '#ffd700',
    'MARKET FORCES'     : '#00aaff',
    'SENTIMENT / FLOWS' : '#ff88aa',
}

# Collect all interpretations
all_interps = []
for name, row in df_sig.iterrows():
    c30 = float(row['corr_30d'])
    c1y = float(row['corr_1y'])
    txt = interpret_factor(name, c30, c1y)
    strength = abs(c30)
    sig = '🔴 STRONG' if strength > 0.5 else \
          ('🟡 MODERATE' if strength > 0.3 else '⚪ WEAK')
    direction = '▲ POSITIVE' if c30 > 0 else '▼ INVERSE'
    all_interps.append((strength, name, c30, c1y, txt, sig, direction,
                        row['type']))

all_interps.sort(reverse=True)

# Draw interpretation table
ax_interp.text(0.5, 0.99,
    'PROFESSIONAL INTERPRETATION  —  Ranked from Most to Least Influential',
    transform=ax_interp.transAxes,
    color='#ffd700', fontsize=12, fontweight='bold',
    va='top', ha='center',
    path_effects=[pe.withStroke(linewidth=3, foreground='#332200')]
)

# Column headers
hx = [0.01, 0.01, 0.32, 0.48, 0.63, 0.78]
hy = 0.94
headers = ['#', 'FACTOR', '30d CORR', 'SIGNAL', 'STRENGTH', 'INTERPRETATION']
hcols   = ['#888899']*6
for hdr, hxp in zip(headers, hx):
    ax_interp.text(hxp, hy, hdr,
                   transform=ax_interp.transAxes,
                   color='#888899', fontsize=8,
                   fontweight='bold', va='top')

ax_interp.plot([0, 1], [0.925, 0.925],
               color='#1a1a33', lw=1.0,
               transform=ax_interp.transAxes,
               clip_on=False)

row_h = 0.058
y     = 0.905
for rank, (strength, name, c30, c1y, txt, sig, direction, ftype) in \
        enumerate(all_interps[:14], 1):

    # Row background alternating
    if rank % 2 == 0:
        rect = mpatches.FancyBboxPatch(
            (0.0, y - row_h*0.85), 1.0, row_h*0.9,
            boxstyle='square,pad=0', transform=ax_interp.transAxes,
            facecolor='#0c0c1a', edgecolor='none', alpha=0.5
        )
        ax_interp.add_patch(rect)

    # Rank number
    rank_col = '#ffd700' if rank <= 3 else '#888899'
    ax_interp.text(0.01, y, f'{rank}.',
                   transform=ax_interp.transAxes,
                   color=rank_col, fontsize=9,
                   fontweight='bold', va='top')

    # Factor name
    col_name = '#00e676' if ftype=='macro' else \
               ('#00aaff' if ftype=='market' else '#ff88aa')
    ax_interp.text(0.035, y, name[:34],
                   transform=ax_interp.transAxes,
                   color=col_name, fontsize=8.5,
                   fontweight='bold' if rank<=3 else 'normal',
                   va='top')

    # 30d correlation value
    corr_col = '#00ff88' if c30 > 0.3 else \
               ('#ff4444' if c30 < -0.3 else '#888888')
    ax_interp.text(0.32, y, f'{c30:+.3f}',
                   transform=ax_interp.transAxes,
                   color=corr_col, fontsize=8.5,
                   fontweight='bold', va='top',
                   fontfamily='monospace')

    # Signal
    dir_col = '#00e676' if '▲' in direction else '#ff4444'
    ax_interp.text(0.40, y, direction,
                   transform=ax_interp.transAxes,
                   color=dir_col, fontsize=8, va='top')

    # Strength
    sig_col = '#ff4444' if 'STRONG' in sig else \
              ('#ffaa00' if 'MOD' in sig else '#666688')
    ax_interp.text(0.55, y, sig,
                   transform=ax_interp.transAxes,
                   color=sig_col, fontsize=7.5, va='top')

    # Interpretation text
    ax_interp.text(0.72, y, txt[:55],
                   transform=ax_interp.transAxes,
                   color='#aaaacc', fontsize=7.5, va='top')

    y -= row_h

# Bottom legend
ax_interp.text(0.01, 0.04,
    'COLOR CODE:  🟡 MACRO/MONETARY   🔵 MARKET FORCES   🩷 SENTIMENT/FLOWS     '
    'CORR: > +0.5 = strong positive driver   < -0.5 = strong inverse driver',
    transform=ax_interp.transAxes,
    color='#555577', fontsize=7.5, va='bottom'
)

plt.savefig(str(OUTDIR / 'gold_deep_factor_analysis.png'),
            dpi=150, facecolor='#07070f', bbox_inches='tight')
print("Saved: /Users/elena_nael/gold_deep_factor_analysis.png")
plt.show()

# ── Print final console summary ──────────────────────────────
print("\n" + "=" * 65)
print("  FINAL VERDICT — WHAT IS DRIVING GOLD RIGHT NOW?")
print("=" * 65)
for rank, (strength, name, c30, c1y, txt, sig, direction, ftype) in \
        enumerate(all_interps[:8], 1):
    print(f"\n  #{rank}  {name}")
    print(f"      30d corr: {c30:+.3f}  |  {sig}  |  {direction}")
    print(f"      → {txt}")
print("\n" + "=" * 65)


In [ ]:
# ── portable output dir (was hardcoded /Users/elena_nael/...) ─────────
from pathlib import Path as _Path
OUTDIR = _Path.cwd() / "outputs"; OUTDIR.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import gaussian_filter1d
from scipy.signal import argrelextrema
import yfinance as yf
import warnings, time, datetime
warnings.filterwarnings('ignore')

print("=" * 70)
print("   GOLD MASTER VERDICT ENGINE")
print("   Full systematic analysis → single trading decision")
print("=" * 70)

# ══════════════════════════════════════════════════════════════════════
# MODULE 1 — DATA FETCH
# ══════════════════════════════════════════════════════════════════════
print("\n[1/7] Fetching market data...")

tickers = {
    'GC=F'      : 'Gold',
    'DX-Y.NYB'  : 'DXY',
    'CL=F'      : 'Oil',
    'SI=F'      : 'Silver',
    'GDX'       : 'Gold Miners',
    '^TNX'      : '10Y Yield',
    '^IRX'      : '3M Yield',
    'TIP'       : 'TIPS',
    '^VIX'      : 'VIX',
    'SPY'       : 'SPY',
    'BTC-USD'   : 'Bitcoin',
    'HG=F'      : 'Copper',
    'GLD'       : 'GLD ETF',
    'IAU'       : 'IAU ETF',
    '^GSPC'     : 'SP500',
}

raw = {}
for t, name in tickers.items():
    try:
        d = yf.download(t, period='1y', interval='1d', progress=False)
        if hasattr(d, 'columns') and isinstance(d.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
            d.columns = d.columns.get_level_values(0)
        if len(d) > 50:
            raw[t] = d['Close'].squeeze()
    except: pass
    time.sleep(0.05)

# Get volume for ETF flow proxies
gld_full = yf.download('GLD', period='1y', interval='1d', progress=False)
if hasattr(gld_full, 'columns') and isinstance(gld_full.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gld_full.columns = gld_full.columns.get_level_values(0)
iau_full = yf.download('IAU', period='1y', interval='1d', progress=False)
if hasattr(iau_full, 'columns') and isinstance(iau_full.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    iau_full.columns = iau_full.columns.get_level_values(0)
gdx_full = yf.download('GDX', period='1y', interval='1d', progress=False)
if hasattr(gdx_full, 'columns') and isinstance(gdx_full.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gdx_full.columns = gdx_full.columns.get_level_values(0)
gc_full  = yf.download('GC=F', period='1y', interval='1d', progress=False)
if hasattr(gc_full, 'columns') and isinstance(gc_full.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    gc_full.columns = gc_full.columns.get_level_values(0)

def fetch_fred(sid):
    try:
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}"
        df  = pd.read_csv(url, index_col=0, parse_dates=True)
        df  = df[df.iloc[:,0] != '.']
        return pd.to_numeric(df.iloc[:,0], errors='coerce').dropna()
    except: return None

print("  Fetching FRED macro...")
cpi      = fetch_fred('CPIAUCSL')
fed_rate = fetch_fred('DFF')
m2       = fetch_fred('M2SL')
real_y   = fetch_fred('DFII10')
breakeven= fetch_fred('T10YIE')
credit   = fetch_fred('BAMLH0A0HYM2')
print("  Data fetch complete")

# ══════════════════════════════════════════════════════════════════════
# MODULE 2 — HMM (Hidden Markov Model — Bull/Bear regime)
# ══════════════════════════════════════════════════════════════════════
print("\n[2/7] Running HMM regime detection...")

gc_price = raw['GC=F']
gc_ret   = gc_price.pct_change().dropna()
last_price = float(np.asarray(gc_price)[-1])

# Simple 2-state HMM via EM
def run_hmm(returns, n_states=2, n_iter=50):
    r = returns.values.flatten().astype(float)
    # Init
    mu    = np.array([np.percentile(r, 25), np.percentile(r, 75)])
    sigma = np.array([r.std(), r.std()])
    trans = np.array([[0.95, 0.05],[0.05, 0.95]], dtype=float)
    pi    = np.array([0.5, 0.5])

    def gauss(x, m, s):
        return np.exp(-0.5*((x-m)/s)**2) / (s*np.sqrt(2*np.pi)+1e-12)

    gamma = np.ones((len(r), n_states)) / n_states

    for _ in range(n_iter):
        T = len(r)
        B = np.zeros((T, n_states))
        for k in range(n_states):
            B[:, k] = gauss(r, mu[k], sigma[k])
        B = np.clip(B, 1e-300, None)

        # Forward
        alpha = np.zeros((T, n_states))
        alpha[0] = pi * B[0]
        alpha[0] /= alpha[0].sum() + 1e-12
        for t in range(1, T):
            alpha[t] = (alpha[t-1] @ trans) * B[t]
            s = alpha[t].sum()
            alpha[t] /= s + 1e-12

        # Backward
        beta = np.ones((T, n_states))
        for t in range(T-2, -1, -1):
            beta[t] = (trans * B[t+1] * beta[t+1]).sum(axis=1)
            s = beta[t].sum()
            beta[t] /= s + 1e-12

        # Gamma
        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True) + 1e-12

        # Xi (joint probability of consecutive states)
        xi = np.zeros((T-1, n_states, n_states))
        for t in range(T-1):
            for i in range(n_states):
                for j in range(n_states):
                    xi[t, i, j] = (alpha[t, i] * trans[i, j] *
                                   B[t+1, j] * beta[t+1, j])
            xi[t] /= xi[t].sum() + 1e-12

        # Update parameters
        mu    = (gamma * r[:, None]).sum(axis=0) / (gamma.sum(axis=0) + 1e-12)
        sigma = np.sqrt((gamma * (r[:, None] - mu)**2).sum(axis=0) /
                        (gamma.sum(axis=0) + 1e-12))
        sigma = np.maximum(sigma, 1e-4)

        # Update transition matrix using xi
        for i in range(n_states):
            denom = xi[:, i, :].sum()
            for j in range(n_states):
                trans[i, j] = xi[:, i, j].sum() / (denom + 1e-12)

    bull_state = 1 if mu[1] > mu[0] else 0
    p_bull = float(gamma[-1, bull_state])
    regime = 'BULL' if p_bull > 0.5 else 'BEAR'
    return p_bull, regime, gamma[-1]

hmm_pbull, hmm_regime, hmm_probs = run_hmm(gc_ret)
print(f"  HMM: P(Bull)={hmm_pbull:.1%}  Regime={hmm_regime}")

# ══════════════════════════════════════════════════════════════════════
# MODULE 3 — GARCH VOLATILITY REGIME
# ══════════════════════════════════════════════════════════════════════
print("\n[3/7] GARCH volatility regime...")

def garch11(returns, omega=1e-6, alpha=0.1, beta=0.85, n_iter=30):
    r = returns.values
    T = len(r)
    h = np.full(T, r.var())
    for _ in range(n_iter):
        for t in range(1, T):
            h[t] = omega + alpha*r[t-1]**2 + beta*h[t-1]
    vol_ann = np.sqrt(h[-1]*252)*100
    avg_vol = np.sqrt(h.mean()*252)*100
    regime  = 'HIGH' if vol_ann > avg_vol*1.2 else \
              ('LOW' if vol_ann < avg_vol*0.8 else 'NORMAL')
    return vol_ann, avg_vol, regime, h

garch_vol, garch_avg, garch_regime, h_series = garch11(gc_ret)
size_mult = 0.5 if garch_regime=='HIGH' else (1.5 if garch_regime=='LOW' else 1.0)
print(f"  GARCH: Vol={garch_vol:.1f}%  Regime={garch_regime}  SizeMult={size_mult}x")

# ══════════════════════════════════════════════════════════════════════
# MODULE 4 — KALMAN FILTER (fair value)
# ══════════════════════════════════════════════════════════════════════
print("\n[4/7] Kalman filter fair value...")

def kalman_filter(prices, Q=1e-4, R=0.01):
    p = prices.values
    x = p[0]; P = 1.0
    xs = []
    for z in p:
        x = x; P = P + Q
        K = P/(P+R)
        x = x + K*(z-x)
        P = (1-K)*P
        xs.append(x)
    fair  = xs[-1]
    gap   = (p[-1] - fair)/fair*100
    z_gap = gap / (np.std(np.diff(xs))*100+1e-9)
    return fair, gap, z_gap

kalman_fair, kalman_gap, kalman_z = kalman_filter(gc_price)
kalman_signal = 'CHEAP' if kalman_gap < -0.5 else \
                ('EXPENSIVE' if kalman_gap > 0.5 else 'FAIR')
print(f"  Kalman: Fair=${kalman_fair:,.0f}  Gap={kalman_gap:+.2f}%  {kalman_signal}")

# ══════════════════════════════════════════════════════════════════════
# MODULE 5 — MONTE CARLO 15-DAY FORECAST
# ══════════════════════════════════════════════════════════════════════
print("\n[5/7] Monte Carlo forecast...")

np.random.seed(42)
N_PATHS = 800
N_DAYS  = 15
sig_d   = garch_vol/100/np.sqrt(252)
mu_d    = 0.154/252

paths = np.zeros((N_PATHS, N_DAYS+1))
paths[:,0] = last_price
for t in range(1, N_DAYS+1):
    z = np.random.randn(N_PATHS)
    paths[:,t] = paths[:,t-1] * np.exp((mu_d - 0.5*sig_d**2) + sig_d*z)

mc_median = float(np.median(paths[:,-1]))
mc_p10    = float(np.percentile(paths[:,-1], 10))
mc_p90    = float(np.percentile(paths[:,-1], 90))
mc_bull   = float((paths[:,-1] > last_price).mean()*100)
print(f"  MC: Median=${mc_median:,.0f}  Bull%={mc_bull:.0f}%")

# ══════════════════════════════════════════════════════════════════════
# MODULE 6 — FACTOR ANALYSIS (top drivers)
# ══════════════════════════════════════════════════════════════════════
print("\n[6/7] Factor correlation engine...")

gc_ret_clean = gc_ret.copy()

factor_map = {
    'DX-Y.NYB': ('Dollar (DXY)',       'market',    'inverse'),
    'CL=F'    : ('Oil (WTI)',           'commodity', 'positive'),
    '^TNX'    : ('10Y Yield',           'rates',     'inverse'),
    '^VIX'    : ('VIX Fear',            'sentiment', 'positive'),
    'SPY'     : ('S&P 500',             'risk',      'varies'),
    'SI=F'    : ('Silver',              'metals',    'positive'),
    'HG=F'    : ('Copper',              'growth',    'positive'),
    'BTC-USD' : ('Bitcoin',             'altasset',  'positive'),
    'GDX'     : ('Gold Miners',         'miners',    'positive'),
    'GLD'     : ('Retail ETF Flow',     'sentiment', 'positive'),
    'IAU'     : ('Institutional Flow',  'sentiment', 'positive'),
}

factors_result = {}
for ticker, (label, ftype, expected) in factor_map.items():
    if ticker not in raw: continue
    s  = raw[ticker].resample('D').last().ffill()
    sr = s.pct_change().dropna()
    # FIX: same tz-mismatch guard as the monthly block below
    _g = gc_ret_clean.tz_localize(None) if getattr(gc_ret_clean.index,'tz',None) is not None else gc_ret_clean
    _s = sr.tz_localize(None) if getattr(sr.index,'tz',None) is not None else sr
    common = _g.index.intersection(_s.index)
    gc_ret_clean, sr = _g, _s
    if len(common) < 30: continue
    gc_c = gc_ret_clean.loc[common].values
    sr_c = sr.loc[common].values
    mask = ~(np.isnan(gc_c)|np.isnan(sr_c))
    if mask.sum() < 30: continue
    c1y,_ = pearsonr(gc_c[mask], sr_c[mask])
    # 30-day
    c30 = np.nan
    common30 = common[-30:]
    gc30 = gc_ret_clean.loc[common30].values
    sr30 = sr.loc[common30].values
    m30  = ~(np.isnan(gc30)|np.isnan(sr30))
    if m30.sum() > 10:
        c30,_ = pearsonr(gc30[m30], sr30[m30])
    factors_result[label] = {'c30': c30, 'c1y': c1y,
                              'type': ftype, 'expected': expected}

# Macro factors
macro_factors = {}
def add_macro(series, label, invert=False):
    if series is None: return
    gc_m = gc_price.resample('MS').last().ffill()
    sm   = series.resample('MS').last().ffill()
    # FIX: resample('MS') can yield a tz-aware vs tz-naive mismatch between the
    # two series, so .intersection() returned labels absent from one of them and
    # .loc[common] raised KeyError. Normalise both to tz-naive first.
    if getattr(gc_m.index, 'tz', None) is not None:
        gc_m = gc_m.tz_localize(None)
    if getattr(sm.index, 'tz', None) is not None:
        sm = sm.tz_localize(None)
    common = gc_m.index.intersection(sm.index)
    common = common[common.isin(gc_m.index) & common.isin(sm.index)]
    if len(common) < 6: return
    gc_c = gc_m.loc[common].values.flatten()
    sm_c = sm.loc[common].values.flatten()
    mask = ~(np.isnan(gc_c)|np.isnan(sm_c))
    if mask.sum() < 5: return
    c,_ = pearsonr(gc_c[mask], sm_c[mask])
    macro_factors[label] = {'c30': c, 'c1y': c, 'type': 'macro'}

add_macro(cpi,      'CPI Inflation')
add_macro(fed_rate, 'Fed Funds Rate')
add_macro(m2,       'M2 Money Supply')
add_macro(real_y,   'Real Yield (FRED)')
add_macro(breakeven,'Breakeven Inflation')
add_macro(credit,   'Credit Spread (HY)')

all_factors = {**factors_result, **macro_factors}

# Sort by |30d corr|
ranked = sorted(all_factors.items(),
                key=lambda x: abs(x[1].get('c30',0)), reverse=True)

# Identify primary driver regime
top3 = ranked[:3]
monetary_score = sum(1 for n,v in top3
                     if v['type']=='macro' and abs(v['c30'])>0.5)
fear_score     = sum(1 for n,v in top3
                     if n in ['VIX Fear','S&P 500'] and abs(v['c30'])>0.4)
flow_score     = sum(1 for n,v in top3
                     if v['type']=='sentiment' and abs(v['c30'])>0.4)
metals_score   = sum(1 for n,v in top3
                     if v['type'] in ['metals','commodity','growth'])

if monetary_score >= 2:
    driver_regime = 'MONETARY DEBASEMENT'
    driver_col    = '#ffd700'
    driver_desc   = 'CPI/Fed/M2 aligned — inflation & rate cut expectations driving gold'
elif fear_score >= 1:
    driver_regime = 'SAFE HAVEN FEAR'
    driver_col    = '#ff8800'
    driver_desc   = 'Risk-off / VIX elevated — flight to safety driving gold'
elif flow_score >= 2:
    driver_regime = 'ETF FLOW DEMAND'
    driver_col    = '#00ff88'
    driver_desc   = 'Retail + institutional buying — demand-side pressure dominant'
elif metals_score >= 2:
    driver_regime = 'COMMODITY SUPERCYCLE'
    driver_col    = '#00aaff'
    driver_desc   = 'Broad metals/commodities bid — inflation + growth trade'
else:
    driver_regime = 'MIXED / UNCLEAR'
    driver_col    = '#888888'
    driver_desc   = 'No single dominant driver — wait for clearer regime'

# GDX divergence check
gdx_div = False
if 'Gold Miners' in factors_result:
    gdx_c30 = factors_result['Gold Miners']['c30']
    # Miners should outperform in true bull — check recent price ratio
    if len(gdx_full) > 20 and len(gc_full) > 20:
        gdx_ret_20 = float(gdx_full['Close'].squeeze().iloc[-1] /
                           gdx_full['Close'].squeeze().iloc[-20] - 1)
        gc_ret_20  = float(gc_full['Close'].squeeze().iloc[-1] /
                           gc_full['Close'].squeeze().iloc[-20] - 1)
        gdx_div = gdx_ret_20 < gc_ret_20 * 0.5  # miners lagging by >50%

copper_confirm = False
if 'Copper' in factors_result:
    copper_confirm = factors_result['Copper']['c30'] > 0.3

print(f"  Driver regime: {driver_regime}")
print(f"  GDX divergence warning: {gdx_div}")
print(f"  Copper confirming: {copper_confirm}")

# ══════════════════════════════════════════════════════════════════════
# MODULE 7 — SMC QUICK SCAN (key levels)
# ══════════════════════════════════════════════════════════════════════
print("\n[7/7] SMC key level scan...")

gc_df = gc_full[['Open','High','Low','Close']].dropna()
gc_df.columns = ['Open','High','Low','Close']

# Swing highs/lows
h_arr = gc_df['High'].values.flatten()
l_arr = gc_df['Low'].values.flatten()
sh_idx = argrelextrema(h_arr, np.greater, order=5)[0]
sl_idx = argrelextrema(l_arr, np.less,    order=5)[0]

# Nearest resistance (swing high above price)
resistances = sorted([h_arr[i] for i in sh_idx if h_arr[i] > last_price])
supports    = sorted([l_arr[i] for i in sl_idx if l_arr[i] < last_price], reverse=True)

r1 = resistances[0] if resistances else last_price*1.02
r2 = resistances[1] if len(resistances)>1 else last_price*1.04
s1 = supports[0]    if supports    else last_price*0.98
s2 = supports[1]    if len(supports)>1 else last_price*0.96

# EMA trend
ema20 = float(gc_df['Close'].ewm(span=20).mean().iloc[-1])
ema50 = float(gc_df['Close'].ewm(span=50).mean().iloc[-1])
ema200= float(gc_df['Close'].ewm(span=200).mean().iloc[-1])
trend_score = sum([last_price>ema20, last_price>ema50, last_price>ema200,
                   ema20>ema50, ema50>ema200])

rsi_raw = gc_df['Close'].diff()
gain = rsi_raw.clip(lower=0).rolling(14).mean()
loss = (-rsi_raw.clip(upper=0)).rolling(14).mean()
rsi  = float((100 - 100/(1+gain/loss.replace(0,1e-9))).iloc[-1])

print(f"  R1=${r1:,.0f}  R2=${r2:,.0f}  S1=${s1:,.0f}  S2=${s2:,.0f}")
print(f"  EMA trend score: {trend_score}/5  RSI: {rsi:.0f}")

# ══════════════════════════════════════════════════════════════════════
# MASTER VERDICT COMPUTATION
# ══════════════════════════════════════════════════════════════════════

# Score each module
scores = {}

# HMM
if hmm_pbull > 0.8:   scores['HMM'] = ('STRONG BULL', 2, '#00ff88')
elif hmm_pbull > 0.6: scores['HMM'] = ('BULL',         1, '#88ff44')
elif hmm_pbull < 0.3: scores['HMM'] = ('BEAR',        -2, '#ff4444')
else:                  scores['HMM'] = ('NEUTRAL',      0, '#888888')

# GARCH
scores['GARCH'] = (f'VOL {garch_regime}', 0, '#ffaa00')

# Kalman
if kalman_gap < -1:    scores['Kalman'] = ('CHEAP +BUY',  1, '#00ff88')
elif kalman_gap > 1:   scores['Kalman'] = ('EXPENSIVE',  -1, '#ff4444')
else:                   scores['Kalman'] = ('FAIR VALUE',  0, '#ffff00')

# Monte Carlo
if mc_bull > 70:   scores['MonteCarlo'] = (f'BULL {mc_bull:.0f}%',  2, '#00ff88')
elif mc_bull > 55: scores['MonteCarlo'] = (f'MILD BULL {mc_bull:.0f}%', 1, '#88ff44')
elif mc_bull < 35: scores['MonteCarlo'] = (f'BEAR {mc_bull:.0f}%', -2, '#ff4444')
else:               scores['MonteCarlo'] = (f'NEUTRAL {mc_bull:.0f}%', 0, '#888888')

# Trend
if trend_score >= 4:  scores['EMA Trend'] = ('BULL TREND',   2, '#00ff88')
elif trend_score >= 3: scores['EMA Trend'] = ('MILD TREND',  1, '#88ff44')
elif trend_score <= 1: scores['EMA Trend'] = ('DOWNTREND',  -2, '#ff4444')
else:                   scores['EMA Trend'] = ('NEUTRAL',     0, '#888888')

# RSI
if rsi > 70:    scores['RSI'] = ('OVERBOUGHT', -1, '#ff6600')
elif rsi < 30:  scores['RSI'] = ('OVERSOLD',    1, '#00ff88')
elif rsi > 55:  scores['RSI'] = ('BULLISH',     1, '#88ff44')
elif rsi < 45:  scores['RSI'] = ('BEARISH',    -1, '#ff4444')
else:            scores['RSI'] = ('NEUTRAL',     0, '#888888')

# Factor regime
if driver_regime in ('MONETARY DEBASEMENT','ETF FLOW DEMAND'):
    scores['Factors'] = (driver_regime, 2, '#ffd700')
elif driver_regime == 'COMMODITY SUPERCYCLE':
    scores['Factors'] = (driver_regime, 1, '#00aaff')
elif driver_regime == 'SAFE HAVEN FEAR':
    scores['Factors'] = (driver_regime, 1, '#ff8800')
else:
    scores['Factors'] = (driver_regime, 0, '#888888')

# GDX warning
if gdx_div:
    scores['GDX'] = ('MINERS LAGGING ⚠', -1, '#ff6600')
else:
    scores['GDX'] = ('MINERS CONFIRM',    1, '#00ff88')

# Copper
if copper_confirm:
    scores['Copper'] = ('GROWTH CONFIRMS',  1, '#00aaff')
else:
    scores['Copper'] = ('COPPER NEUTRAL',   0, '#888888')

# Total score
total_score = sum(v[1] for v in scores.values())
max_score   = 14

conviction = min(100, max(0, int((total_score / max_score)*100 + 50)))

# Final direction
if total_score >= 6:
    direction = 'LONG'; dir_col = '#00ff88'
elif total_score <= -3:
    direction = 'SHORT'; dir_col = '#ff4444'
else:
    direction = 'NEUTRAL / WAIT'; dir_col = '#ffaa00'

# Entry / target / stop
entry  = max(s1, last_price * 0.998)
target = min(r1, mc_median)
stop   = s1 * 0.997
rr     = abs(target - entry) / abs(entry - stop + 1e-9)

# Key risk
risks = []
if rsi > 65:         risks.append(f'RSI overbought ({rsi:.0f})')
if gdx_div:          risks.append('GDX miners lagging gold')
if garch_regime=='HIGH': risks.append(f'High vol ({garch_vol:.0f}%) — widen stops')
if not copper_confirm:   risks.append('Copper not confirming')
key_risk = ' · '.join(risks) if risks else 'No major risk flags'

# Next catalyst — check upcoming macro
catalysts = []
now = pd.Timestamp.now()
# Approximate next CPI (usually 2nd week of month)
next_cpi = pd.Timestamp(now.year, now.month + (1 if now.day > 12 else 0), 12)
catalysts.append(f"CPI release ~{next_cpi.strftime('%b %d')}")
catalysts.append("Fed speakers / FOMC minutes")
catalysts.append("Non-Farm Payrolls")

# ══════════════════════════════════════════════════════════════════════
# PRINT CONSOLE VERDICT
# ══════════════════════════════════════════════════════════════════════
print("\n" + "═"*70)
print(f"  GOLD MASTER VERDICT — {pd.Timestamp.now().strftime('%B %d, %Y')}")
print("═"*70)
print(f"  Direction    :  {direction}")
print(f"  Conviction   :  {conviction}%  (score {total_score}/{max_score})")
print(f"  Position Size:  {size_mult}x  (GARCH vol-adjusted)")
print(f"  Entry Zone   :  ${entry:,.0f}")
print(f"  Target (T1)  :  ${target:,.0f}  ({(target-last_price)/last_price*100:+.1f}%)")
print(f"  Stop Loss    :  ${stop:,.0f}  ({(stop-last_price)/last_price*100:+.1f}%)")
print(f"  R:R Ratio    :  {rr:.1f}:1")
print(f"  ─────────────────────────────────────────────────")
print(f"  Driver Regime:  {driver_regime}")
print(f"  {driver_desc}")
print(f"  ─────────────────────────────────────────────────")
print(f"  Key Risk     :  {key_risk}")
print(f"  Watch        :  {catalysts[0]}")
print("═"*70)

# ══════════════════════════════════════════════════════════════════════
# FIGURE — MASTER VERDICT DASHBOARD
# ══════════════════════════════════════════════════════════════════════
print("\nBuilding master verdict chart...")

fig = plt.figure(figsize=(24, 22), facecolor='#05050f')
fig.patch.set_facecolor('#05050f')

gs = gridspec.GridSpec(
    4, 3, figure=fig,
    height_ratios=[1.1, 1.8, 1.8, 1.4],
    width_ratios=[1.2, 1.0, 1.0],
    hspace=0.08, wspace=0.07,
    left=0.04, right=0.97,
    top=0.97, bottom=0.03
)

ax_verdict  = fig.add_subplot(gs[0, :])      # top — big verdict
ax_price    = fig.add_subplot(gs[1, :2])     # price chart
ax_modules  = fig.add_subplot(gs[1, 2])      # module scores
ax_factors  = fig.add_subplot(gs[2, :2])     # factor ranking
ax_mc       = fig.add_subplot(gs[2, 2])      # MC distribution
ax_interp   = fig.add_subplot(gs[3, :])      # interpretation table

BG = '#05050f'
for ax in [ax_verdict,ax_price,ax_modules,ax_factors,ax_mc,ax_interp]:
    ax.set_facecolor(BG)
    for sp in ax.spines.values():
        sp.set_color('#111128')
        sp.set_linewidth(0.5)
    ax.tick_params(colors='#444466', labelsize=8)

# ── VERDICT PANEL (top) ──────────────────────────────────────
ax_verdict.axis('off')
ax_verdict.set_xlim(0,1); ax_verdict.set_ylim(0,1)

# Background glow
verdict_bg = '#001a00' if direction=='LONG' else \
             ('#1a0000' if direction=='SHORT' else '#0a0a00')
rect = FancyBboxPatch((0.01,0.05), 0.98, 0.90,
    boxstyle='round,pad=0.01', transform=ax_verdict.transAxes,
    facecolor=verdict_bg, edgecolor=dir_col,
    linewidth=2.0, alpha=0.8)
ax_verdict.add_patch(rect)

# Main verdict text
ax_verdict.text(0.5, 0.80,
    f'GOLD  MASTER  VERDICT   ·   {pd.Timestamp.now().strftime("%B %d, %Y")}',
    transform=ax_verdict.transAxes, color='#888899',
    fontsize=10, ha='center', va='center')

ax_verdict.text(0.18, 0.42,
    direction, transform=ax_verdict.transAxes,
    color=dir_col, fontsize=38, fontweight='bold',
    ha='center', va='center',
    path_effects=[pe.withStroke(linewidth=8, foreground='#000000')])

ax_verdict.text(0.18, 0.12,
    f'${last_price:,.2f}', transform=ax_verdict.transAxes,
    color='#ffd700', fontsize=14, fontweight='bold',
    ha='center', va='center')

# Conviction meter
ax_verdict.text(0.42, 0.72, 'CONVICTION', transform=ax_verdict.transAxes,
                color='#888899', fontsize=9, ha='center')
conv_col = '#00ff88' if conviction>65 else ('#ffaa00' if conviction>45 else '#ff4444')
ax_verdict.text(0.42, 0.38, f'{conviction}%',
    transform=ax_verdict.transAxes, color=conv_col,
    fontsize=32, fontweight='bold', ha='center', va='center',
    path_effects=[pe.withStroke(linewidth=6, foreground='#000000')])
ax_verdict.text(0.42, 0.12, f'Score {total_score}/{max_score}',
    transform=ax_verdict.transAxes, color='#666688',
    fontsize=10, ha='center')

# Trade params
params = [
    ('SIZE',   f'{size_mult}x',         '#ffaa00'),
    ('ENTRY',  f'${entry:,.0f}',        '#ffffff'),
    ('TARGET', f'${target:,.0f}',       '#00ff88'),
    ('STOP',   f'${stop:,.0f}',         '#ff4444'),
    ('R:R',    f'{rr:.1f}:1',           '#00aaff'),
    ('REGIME', driver_regime[:16],      driver_col),
]
for i, (lbl, val, col) in enumerate(params):
    x = 0.58 + (i%3)*0.137
    y = 0.72 if i < 3 else 0.28
    ax_verdict.text(x, y+0.12, lbl,
        transform=ax_verdict.transAxes, color='#666688',
        fontsize=8, ha='center')
    ax_verdict.text(x, y-0.02, val,
        transform=ax_verdict.transAxes, color=col,
        fontsize=13, fontweight='bold', ha='center',
        path_effects=[pe.withStroke(linewidth=3, foreground='#000000')])

# ── PRICE CHART ──────────────────────────────────────────────
sub = gc_df.iloc[-60:].reset_index()
xv  = np.arange(len(sub))

for i, row in sub.iterrows():
    o = float(row['Open']); h = float(row['High'])
    l = float(row['Low']);  c = float(row['Close'])
    col = '#00e676' if c>=o else '#ff4444'
    ax_price.plot([i,i],[l,h], color=col, lw=0.8)
    rect = mpatches.Rectangle((i-0.35,min(o,c)),0.70,max(abs(c-o),0.1),
        facecolor=col, edgecolor=col, lw=0.2, alpha=0.9)
    ax_price.add_patch(rect)

# EMAs
e20  = gc_df['Close'].ewm(span=20).mean().iloc[-60:].values
e50  = gc_df['Close'].ewm(span=50).mean().iloc[-60:].values
e200 = gc_df['Close'].ewm(span=200).mean().iloc[-60:].values
ax_price.plot(xv, e20,  color='#00aaff', lw=1.2, alpha=0.8, label='EMA20')
ax_price.plot(xv, e50,  color='#ff9900', lw=1.2, alpha=0.8, label='EMA50')
ax_price.plot(xv, e200, color='#ff4444', lw=1.0, alpha=0.7,
              label='EMA200', linestyle='--')

# Key levels
for lvl, lbl, col, ls in [(r1,'R1','#ff6666','-.'),(r2,'R2','#ff3333','--'),
                            (s1,'S1','#66ff66','-.'),(s2,'S2','#33ff33','--'),
                            (entry,'ENTRY','#ffffff',':'),
                            (target,'TARGET','#00ff88','-'),
                            (stop,'STOP','#ff4444','-')]:
    ax_price.axhline(lvl, color=col, lw=1.0, linestyle=ls, alpha=0.75)
    ax_price.text(len(sub)-1, lvl, f' {lbl} ${lvl:,.0f}',
                  color=col, fontsize=7, va='center',
                  path_effects=[pe.withStroke(linewidth=1.5,
                                               foreground='#05050f')])

ax_price.axhline(last_price, color='#ffd700', lw=1.5, alpha=0.9)
step = max(1, len(sub)//7)
ax_price.set_xticks(xv[::step])
ax_price.set_xticklabels([sub.iloc[i,0].strftime('%b %d')
                           for i in range(0,len(sub),step)],
                          fontsize=7, color='#444466')
ax_price.set_xlim(-1, len(sub))
ax_price.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax_price.yaxis.grid(True, color='#0d0d1a', lw=0.4)
ax_price.set_axisbelow(True)
ax_price.set_title('Price Chart (60 days) · EMAs · Key Levels · Entry/Target/Stop',
                    color='#888899', fontsize=9, pad=4, loc='left')
ax_price.legend(loc='upper left', fontsize=7.5,
                facecolor='#111122', edgecolor='#222233',
                labelcolor='white', framealpha=0.9)

# ── MODULE SCORECARD ─────────────────────────────────────────
ax_modules.axis('off')
ax_modules.set_xlim(0,1); ax_modules.set_ylim(0,1)
ax_modules.text(0.5, 0.97, 'MODEL SCORECARD',
                transform=ax_modules.transAxes,
                color='#ffd700', fontsize=10, fontweight='bold',
                ha='center', va='top')

row_items = list(scores.items())
rh = 0.88 / len(row_items)
for i, (mod, (label, val, col)) in enumerate(row_items):
    y = 0.92 - i*rh
    # Score bar background
    bw = max(0.01, abs(val)/3)
    bx = 0.5 if val >= 0 else 0.5 - bw
    rect = FancyBboxPatch((bx, y-rh*0.35), bw, rh*0.7,
        boxstyle='round,pad=0.005',
        transform=ax_modules.transAxes,
        facecolor=col, alpha=0.25, edgecolor='none')
    ax_modules.add_patch(rect)
    ax_modules.text(0.02, y, mod,
        transform=ax_modules.transAxes,
        color='#aaaacc', fontsize=8, va='center')
    ax_modules.text(0.98, y, label,
        transform=ax_modules.transAxes,
        color=col, fontsize=8, va='center', ha='right',
        fontweight='bold')
    # Score dot
    score_col = '#00ff88' if val>0 else ('#ff4444' if val<0 else '#888888')
    ax_modules.text(0.50, y, f'{val:+d}',
        transform=ax_modules.transAxes,
        color=score_col, fontsize=8, va='center', ha='center',
        fontweight='bold')

ax_modules.plot([0,1],[0.0,0.0], transform=ax_modules.transAxes,
                 color='#1a1a33', lw=0.5, clip_on=False)
ax_modules.set_title('Model Signals', color='#888899',
                      fontsize=9, pad=4)

# ── FACTOR RANKING ───────────────────────────────────────────
top_n   = min(16, len(ranked))
top_f   = ranked[:top_n]
f_names = [n[:36] for n,v in top_f]
f_c30   = [v['c30'] for n,v in top_f]
f_c1y   = [v['c1y'] for n,v in top_f]
f_types = [v['type'] for n,v in top_f]
ypos    = np.arange(len(f_names))

type_colors = {
    'macro':'#ffd700', 'market':'#00aaff', 'sentiment':'#ff88aa',
    'metals':'#88ffcc','commodity':'#ff9900','rates':'#ff6666',
    'risk':'#aa88ff', 'miners':'#ffcc00','altasset':'#ff44aa',
    'growth':'#44ffaa','derived':'#ffaa44'
}
bcolors = [type_colors.get(t,'#888888') for t in f_types]

ax_factors.barh(ypos, f_c30, color=bcolors, alpha=0.80,
                height=0.55, zorder=3)
ax_factors.scatter(f_c1y, ypos, color='#ffffff', s=30,
                   marker='D', zorder=5, alpha=0.75)
ax_factors.axvline(0, color='#333355', lw=1.0)
for v in [0.3, 0.5, -0.3, -0.5]:
    ax_factors.axvline(v, color='#222244', lw=0.5, linestyle='--')

ax_factors.set_yticks(ypos)
ax_factors.set_yticklabels(f_names, fontsize=8, color='#ccccdd')
ax_factors.set_xlim(-1.05, 1.05)
ax_factors.xaxis.grid(True, color='#0d0d1a', lw=0.4)
ax_factors.set_axisbelow(True)

for i, (val, nm) in enumerate(zip(f_c30, f_names)):
    c = '#00ff88' if val>0 else '#ff4444'
    ax_factors.text(val+(0.02 if val>=0 else -0.02), i,
                    f'{val:+.2f}', va='center',
                    ha='left' if val>=0 else 'right',
                    color=c, fontsize=7, fontweight='bold')

ax_factors.set_title(
    f'FACTOR RANKING (30d)   ·   Bar=30d  ◆=1Y   '
    f'🟡Macro  🔵Market  🩷Sentiment  🟢Metals',
    color='#888899', fontsize=8.5, pad=4, loc='left')

# ── MONTE CARLO DISTRIBUTION ─────────────────────────────────
finals  = paths[:,-1]
kde_x   = np.linspace(finals.min(), finals.max(), 300)
from scipy.stats import gaussian_kde
kde_fn  = gaussian_kde(finals, bw_method=0.12)
kde_y   = kde_fn(kde_x)
kde_y  /= kde_y.max()

ax_mc.fill_between(kde_x, kde_y, 0,
    where=kde_x>=last_price, color='#003322', alpha=0.85)
ax_mc.fill_between(kde_x, kde_y, 0,
    where=kde_x<last_price, color='#220000', alpha=0.85)
ax_mc.plot(kde_x, kde_y, color='#ffffff', lw=1.0, alpha=0.7)

for val, lbl, col in [
    (mc_median,   f'Med ${mc_median:,.0f}', '#ffd700'),
    (mc_p90,      f'P90 ${mc_p90:,.0f}',   '#00ff88'),
    (mc_p10,      f'P10 ${mc_p10:,.0f}',   '#ff4444'),
    (last_price,  f'Now ${last_price:,.0f}','#ffffff'),
    (target,      f'T1  ${target:,.0f}',    '#00ffaa'),
]:
    ax_mc.axvline(val, color=col, lw=1.2 if val!=last_price else 1.8,
                   linestyle='--' if val==last_price else '-', alpha=0.85)
    ax_mc.text(val, 0.92, f'\n{lbl}', color=col, fontsize=6.5,
                ha='center', va='top', rotation=90)

ax_mc.set_ylim(0, 1.15)
ax_mc.yaxis.set_visible(False)
ax_mc.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax_mc.tick_params(axis='x', labelsize=6.5, rotation=30)
ax_mc.set_title(f'15-Day MC Distribution\nBull: {mc_bull:.0f}%',
                 color='#888899', fontsize=8.5, pad=4)

# ── INTERPRETATION TABLE ─────────────────────────────────────
ax_interp.axis('off')
ax_interp.set_facecolor('#060610')

ax_interp.text(0.5, 0.97,
    f'FULL INTERPRETATION  —  {driver_regime}  —  Key Risk: {key_risk}',
    transform=ax_interp.transAxes, color='#ffd700',
    fontsize=11, fontweight='bold', ha='center', va='top',
    path_effects=[pe.withStroke(linewidth=3, foreground='#000000')])

interp_lines = [
    ('HMM Regime',   f'P(Bull)={hmm_pbull:.1%}',
     f'Market is in a {hmm_regime} regime. '
     f'{"Strong conviction — proceed with trend." if hmm_pbull>0.8 else "Mixed signals — size down."}',
     scores["HMM"][2]),

    ('GARCH Vol',    f'{garch_vol:.1f}% ann',
     f'Volatility is {garch_regime}. Position size adjusted to {size_mult}x. '
     f'{"Widen stops by 20% in high vol." if garch_regime=="HIGH" else "Normal stops apply."}',
     '#ffaa00'),

    ('Kalman Filter', f'Fair=${kalman_fair:,.0f}  Gap={kalman_gap:+.1f}%',
     f'Gold is {kalman_signal} vs Kalman fair value. '
     f'{"Entry has positive mean reversion edge." if kalman_signal=="CHEAP" else "No valuation edge — momentum driven."}',
     scores["Kalman"][2]),

    ('Driver Regime', driver_regime,
     driver_desc,
     driver_col),

    ('GDX Signal',
     'LAGGING ⚠' if gdx_div else 'CONFIRMING',
     ('Miners underperforming gold by >50% over 20 days. '
      'This is a warning — true bull markets see miners outperform. Watch closely.'
      if gdx_div else
      'Miners moving with gold — institutional equity market confirming the move.'),
     '#ff6600' if gdx_div else '#00ff88'),

    ('Copper / Global', 'CONFIRMING' if copper_confirm else 'NEUTRAL',
     ('Copper rising with gold = global growth + inflation trade. '
      'This is a broad commodity bid, not just safe haven.' if copper_confirm else
      'Copper not confirming. Gold may be running on fear/monetary factors alone — more fragile.'),
     '#00aaff' if copper_confirm else '#888888'),

    ('Key Catalysts',  catalysts[0],
     f'Watch: {" | ".join(catalysts)}. '
     f'CPI beat = gold higher. Fed hawkish surprise = biggest downside risk.',
     '#ff8800'),

    ('Trade Setup',
     f'Entry ${entry:,.0f} → T1 ${target:,.0f} → Stop ${stop:,.0f}',
     f'Risk/Reward {rr:.1f}:1. '
     f'{"R:R acceptable — proceed." if rr>=1.5 else "R:R below 1.5 — consider waiting for better entry."} '
     f'Size: {size_mult}x of normal position.',
     dir_col),
]

rh2 = 0.87 / len(interp_lines)
for i, (mod, val, desc, col) in enumerate(interp_lines):
    y = 0.90 - i*rh2
    if i%2==0:
        bg = FancyBboxPatch((0.0, y-rh2*0.75), 1.0, rh2*0.90,
            boxstyle='square,pad=0',
            transform=ax_interp.transAxes,
            facecolor='#0c0c1c', edgecolor='none', alpha=0.5)
        ax_interp.add_patch(bg)

    ax_interp.text(0.005, y, mod,
        transform=ax_interp.transAxes,
        color=col, fontsize=8.5, fontweight='bold', va='top')
    ax_interp.text(0.15, y, val,
        transform=ax_interp.transAxes,
        color='#ffffff', fontsize=8, va='top', fontfamily='monospace')
    ax_interp.text(0.35, y, desc[:90],
        transform=ax_interp.transAxes,
        color='#aaaacc', fontsize=8, va='top')

plt.savefig(str(OUTDIR / 'gold_master_verdict.png'),
            dpi=150, facecolor='#05050f', bbox_inches='tight')
print("Saved: /Users/elena_nael/gold_master_verdict.png")
plt.show()


In [ ]:
# ── export this notebook to HTML ──────────────────────────────────────
# FIX: was a hardcoded interpreter (/opt/anaconda3/bin/jupyter) and a
# hardcoded notebook path. Both break on any other machine or env.
import subprocess, sys, glob, os
from pathlib import Path

_here = Path.cwd()
_nb = sorted(glob.glob(str(_here / "Daily_systematic_gold_macro_model*.ipynb")),
             key=os.path.getmtime)
if _nb:
    subprocess.run([sys.executable, "-m", "jupyter", "nbconvert",
                    "--to", "html", "--no-input", "--no-prompt", _nb[-1]])
    print(f"  exported {Path(_nb[-1]).name}")
else:
    print(f"  notebook not found in {_here} — run this from the folder it lives in")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ELENA GOLD — LIQUIDITY HEATMAP
#  Shows price levels with highest/lowest liquidity concentration
#  High liquidity = yellow/red  |  Low liquidity = blue
#  Paste entire cell into Jupyter and run
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
from datetime import datetime, timedelta
import pytz
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter

warnings.filterwarnings('ignore')

ATHENS = pytz.timezone('Europe/Athens')
NOW    = datetime.now(ATHENS)

# ── CONFIG ────────────────────────────────────────────────────────────────────
LOOKBACK_DAYS  = 60       # how many days of data to build the heatmap from
PRICE_BINS     = 300      # vertical resolution (price levels)
TIME_BINS      = 120      # horizontal resolution (time buckets)
SMOOTH_SIGMA   = 2.5      # gaussian blur — higher = smoother heatmap
VOL_WEIGHT     = True     # weight by volume (True = institutional, False = pure price)

# ── COLOUR MAP — blue (no liquidity) → yellow → red (high liquidity) ─────────
cmap = LinearSegmentedColormap.from_list(
    'liquidity',
    [
        '#0a0e2a',   # deep blue    — no liquidity
        '#0d2b6e',   # navy         — very low
        '#1a4fa0',   # blue         — low
        '#1a7abf',   # steel blue   — below average
        '#12a693',   # teal         — average
        '#f0c040',   # gold/yellow  — high liquidity
        '#f07010',   # orange       — very high
        '#e02020',   # red          — extreme liquidity cluster
    ],
    N=512
)

print("📥  Fetching GC=F data...")

# ── FETCH DATA ────────────────────────────────────────────────────────────────
def fetch_ohlcv(ticker, period, interval):
    raw = yf.download(ticker, period=period, interval=interval,
                      auto_adjust=True, progress=False)
    if hasattr(raw, 'columns') and isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    raw.dropna(inplace=True)
    if raw.index.tz is None:
        raw.index = raw.index.tz_localize('UTC').tz_convert(ATHENS)
    else:
        raw.index = raw.index.tz_convert(ATHENS)
    return raw[raw['Volume'] > 0].copy()

# Use 1H for richer intrabar data
df = fetch_ohlcv('GC=F', f'{LOOKBACK_DAYS+5}d', '1h')
df = df[df.index >= (NOW - timedelta(days=LOOKBACK_DAYS))]

if df.empty:
    raise RuntimeError("No data fetched — check connection")

print(f"  ✓  {len(df)} bars loaded  |  "
      f"{df.index[0].strftime('%d %b')} → {df.index[-1].strftime('%d %b %Y')}")

price_min = float(df['Low'].min())
price_max = float(df['High'].max())
price_pad = (price_max - price_min) * 0.02
price_min -= price_pad
price_max += price_pad

# ── BUILD LIQUIDITY MATRIX ────────────────────────────────────────────────────
# For each bar: distribute volume across the High–Low range
# This approximates where trades actually occurred (TPO-style volume profile)

price_edges = np.linspace(price_min, price_max, PRICE_BINS + 1)
time_edges  = np.linspace(0, len(df), TIME_BINS + 1)
price_centers = (price_edges[:-1] + price_edges[1:]) / 2
price_step    = price_edges[1] - price_edges[0]

matrix = np.zeros((PRICE_BINS, TIME_BINS))

for t_idx, (ts, row) in enumerate(df.iterrows()):
    t_bin = int(np.clip(t_idx / len(df) * TIME_BINS, 0, TIME_BINS - 1))
    lo    = float(row['Low'])
    hi    = float(row['High'])
    vol   = float(row['Volume']) if VOL_WEIGHT else 1.0
    cl    = float(row['Close'])
    op    = float(row['Open'])

    # Price bins touched by this bar
    p_lo_bin = int(np.clip((lo - price_min) / (price_max - price_min) * PRICE_BINS, 0, PRICE_BINS-1))
    p_hi_bin = int(np.clip((hi - price_min) / (price_max - price_min) * PRICE_BINS, 0, PRICE_BINS-1))

    if p_lo_bin == p_hi_bin:
        matrix[p_lo_bin, t_bin] += vol
        continue

    n_bins = p_hi_bin - p_lo_bin + 1

    # Volume distribution — bell-shaped within the bar's range
    # More volume near open/close (where price spent most time)
    bin_prices   = price_centers[p_lo_bin:p_hi_bin+1]
    close_dist   = np.exp(-0.5 * ((bin_prices - cl) / (price_step * n_bins * 0.3))**2)
    open_dist    = np.exp(-0.5 * ((bin_prices - op) / (price_step * n_bins * 0.3))**2)
    weights      = close_dist + open_dist
    weights_sum  = weights.sum()
    if weights_sum > 0:
        weights = weights / weights_sum
    else:
        weights = np.ones(n_bins) / n_bins

    matrix[p_lo_bin:p_hi_bin+1, t_bin] += weights * vol

# ── SMOOTH ────────────────────────────────────────────────────────────────────
matrix_smooth = gaussian_filter(matrix, sigma=SMOOTH_SIGMA)

# ── KEY LEVELS ────────────────────────────────────────────────────────────────
# Volume profile (sum across time) → find POCs and high-volume nodes
vol_profile    = matrix_smooth.sum(axis=1)
vol_profile_n  = vol_profile / vol_profile.max()

# POC — single highest liquidity price
poc_bin   = int(np.argmax(vol_profile))
poc_price = float(price_centers[poc_bin])

# High volume nodes (HVN) — top 10% of volume profile
hvn_thresh  = np.percentile(vol_profile, 90)
hvn_bins    = np.where(vol_profile >= hvn_thresh)[0]

# Low volume nodes (LVN) — bottom 15% of volume profile
lvn_thresh  = np.percentile(vol_profile, 15)
lvn_bins    = np.where(vol_profile <= lvn_thresh)[0]

# Session levels
def session_level(session='asia'):
    today = NOW.date()
    if session == 'asia':
        mask = (df.index.date == today) & (df.index.hour >= 1)  & (df.index.hour < 8)
    elif session == 'london':
        mask = (df.index.date == today) & (df.index.hour >= 10) & (df.index.hour < 16)
    elif session == 'prev_day':
        yesterday = (NOW - timedelta(days=1)).date()
        mask = df.index.date == yesterday
    s = df[mask]
    if s.empty: return None, None
    return float(s['Low'].min()), float(s['High'].max())

asia_l,   asia_h   = session_level('asia')
london_l, london_h = session_level('london')
pd_l,     pd_h     = session_level('prev_day')
current_price       = float(np.asarray(df['Close'])[-1])

# ── VWAP (full lookback) ──────────────────────────────────────────────────────
df['tp']    = (df['High'] + df['Low'] + df['Close']) / 3
df['tpvol'] = df['tp'] * df['Volume']
vwap_global = float(df['tpvol'].cumsum().iloc[-1] / df['Volume'].cumsum().iloc[-1])

# ── TIME AXIS LABELS ──────────────────────────────────────────────────────────
n_labels   = 8
label_idxs = np.linspace(0, len(df)-1, n_labels, dtype=int)
label_bins = (label_idxs / len(df) * TIME_BINS).astype(int)
label_strs = [df.index[i].strftime('%d/%m\n%H:%M') for i in label_idxs]

# ── PLOT ──────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 12), facecolor='#0a0c10')
gs  = plt.GridSpec(1, 2, figure=fig, width_ratios=[5, 1],
                   wspace=0.02, left=0.05, right=0.96, top=0.91, bottom=0.09)

ax_heat  = fig.add_subplot(gs[0])   # main heatmap
ax_prof  = fig.add_subplot(gs[1])   # volume profile sidebar

# ── HEATMAP ───────────────────────────────────────────────────────────────────
im = ax_heat.imshow(
    matrix_smooth,
    aspect='auto',
    origin='lower',
    cmap=cmap,
    extent=[0, TIME_BINS, price_min, price_max],
    interpolation='bilinear',
    vmin=0,
    vmax=np.percentile(matrix_smooth, 99),  # cap at 99th pct to avoid outlier domination
)

# ── OVERLAY KEY LEVELS ────────────────────────────────────────────────────────
def hline(ax, price, color, label, ls='-', lw=1.2, alpha=0.85, xmax=1.0):
    if price is None: return
    ax.axhline(price, color=color, lw=lw, ls=ls, alpha=alpha, xmax=xmax)
    ax.text(TIME_BINS * 0.005, price, f' {label}: {price:,.1f}',
            color=color, fontsize=8, va='bottom', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.15', facecolor='#0a0c10', alpha=0.6, edgecolor='none'))

# POC — brightest level
hline(ax_heat, poc_price, '#ffffff', 'POC', lw=1.8, ls='-')

# Session levels
hline(ax_heat, asia_h,   '#44ff88', 'Asia H',   ls='--', lw=1.0)
hline(ax_heat, asia_l,   '#44ff88', 'Asia L',   ls='--', lw=1.0)
hline(ax_heat, london_h, '#44aaff', 'London H', ls='--', lw=1.0)
hline(ax_heat, london_l, '#44aaff', 'London L', ls='--', lw=1.0)
hline(ax_heat, pd_h,     '#ffaa33', 'PDH',      ls=':',  lw=1.0)
hline(ax_heat, pd_l,     '#ffaa33', 'PDL',      ls=':',  lw=1.0)
hline(ax_heat, vwap_global, '#cc88ff', 'VWAP', ls='-.', lw=1.0)

# Current price — bright white tick
ax_heat.axhline(current_price, color='#ffffff', lw=2.2, ls='-', alpha=1.0)
ax_heat.text(TIME_BINS * 0.99, current_price, f' ▶ {current_price:,.1f}',
             color='#ffffff', fontsize=9, va='center', ha='right', fontweight='bold')

# LVN bands — shade low-liquidity zones (price magnets / fast travel zones)
lvn_runs, in_run, run_start = [], False, 0
for i in range(PRICE_BINS):
    in_lvn = vol_profile[i] <= lvn_thresh
    if in_lvn and not in_run:
        in_run, run_start = True, i
    elif not in_lvn and in_run:
        in_run = False
        if i - run_start > 3:  # only shade meaningful LVN stretches
            ax_heat.axhspan(price_centers[run_start], price_centers[i-1],
                            alpha=0.08, color='#ffffff', zorder=0)
if in_run:
    ax_heat.axhspan(price_centers[run_start], price_centers[-1],
                    alpha=0.08, color='#ffffff', zorder=0)

# ── AXES FORMATTING ───────────────────────────────────────────────────────────
ax_heat.set_facecolor('#0a0c10')
ax_heat.tick_params(colors='#8a95a8', labelsize=8)
for sp in ax_heat.spines.values(): sp.set_color('#1e2330')
ax_heat.set_ylim(price_min, price_max)
ax_heat.set_xlim(0, TIME_BINS)
ax_heat.set_xticks(label_bins)
ax_heat.set_xticklabels(label_strs, color='#8a95a8', fontsize=7)
ax_heat.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax_heat.set_ylabel('Price (USD/oz)', color='#8a95a8', fontsize=9)
ax_heat.set_xlabel('Time (Athens)', color='#8a95a8', fontsize=9)

# ── VOLUME PROFILE SIDEBAR ────────────────────────────────────────────────────
ax_prof.set_facecolor('#0a0c10')
ax_prof.tick_params(colors='#8a95a8', labelsize=7)
for sp in ax_prof.spines.values(): sp.set_color('#1e2330')

# Colour the profile bars by their own intensity
profile_colors = cmap(vol_profile_n)
ax_prof.barh(price_centers, vol_profile_n,
             height=price_step * 1.05,
             color=profile_colors,
             edgecolor='none',
             align='center')

# POC line on profile
ax_prof.axhline(poc_price, color='#ffffff', lw=1.5, ls='-')

ax_prof.set_ylim(price_min, price_max)
ax_prof.set_xlim(0, 1.05)
ax_prof.set_xticks([])
ax_prof.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax_prof.yaxis.tick_right()
ax_prof.set_title('Vol Profile', color='#C9A84C', fontsize=8, pad=6)

# ── COLORBAR ──────────────────────────────────────────────────────────────────
cbar = fig.colorbar(im, ax=ax_heat, orientation='vertical',
                    fraction=0.012, pad=0.01, aspect=40)
cbar.ax.tick_params(colors='#8a95a8', labelsize=7)
cbar.set_label('Liquidity Intensity', color='#8a95a8', fontsize=8)
cbar.set_ticks([])

# ── TITLE & ANNOTATIONS ───────────────────────────────────────────────────────
fig.suptitle(
    f"GOLD LIQUIDITY HEATMAP  ·  GC Futures  ·  "
    f"Last {LOOKBACK_DAYS} days  ·  "
    f"{NOW.strftime('%A %d %B %Y  %H:%M Athens')}  ·  "
    f"Current: ${current_price:,.1f}  ·  POC: ${poc_price:,.1f}",
    color='#F0C040', fontsize=12, fontweight='bold', y=0.97
)

# Legend
from matplotlib.lines import Line2D
legend_items = [
    Line2D([0],[0], color='#ffffff', lw=1.8,  label=f'POC ${poc_price:,.1f}'),
    Line2D([0],[0], color='#44ff88', lw=1,    ls='--', label='Asia H/L'),
    Line2D([0],[0], color='#44aaff', lw=1,    ls='--', label='London H/L'),
    Line2D([0],[0], color='#ffaa33', lw=1,    ls=':',  label='PDH/PDL'),
    Line2D([0],[0], color='#cc88ff', lw=1,    ls='-.', label='VWAP'),
    Line2D([0],[0], color='#ffffff', lw=0.5,  alpha=0.3,
           label='LVN zones (fast travel)', linestyle='-'),
]
ax_heat.legend(handles=legend_items, loc='upper left',
               facecolor='#12151c', edgecolor='#1e2330',
               labelcolor='#8a95a8', fontsize=7.5, framealpha=0.85)

plt.savefig('Elena_Gold_Liquidity_Heatmap.png', dpi=160,
            bbox_inches='tight', facecolor='#0a0c10')
plt.show()

# ── CONSOLE SUMMARY ───────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  LIQUIDITY SUMMARY")
print(f"{'═'*60}")
print(f"  Current price : ${current_price:,.2f}")
print(f"  POC           : ${poc_price:,.2f}  ← highest liquidity level")
print(f"  VWAP          : ${vwap_global:,.2f}")
if asia_h:   print(f"  Asia H/L      : ${asia_h:,.2f} / ${asia_l:,.2f}")
if london_h: print(f"  London H/L    : ${london_h:,.2f} / ${london_l:,.2f}")
if pd_h:     print(f"  PDH/PDL       : ${pd_h:,.2f} / ${pd_l:,.2f}")

# Top 5 HVN clusters
hvn_prices = price_centers[hvn_bins]
hvn_vols   = vol_profile[hvn_bins]
top5_idx   = np.argsort(hvn_vols)[::-1][:5]
print(f"\n  TOP 5 LIQUIDITY CLUSTERS (HVN):")
for i, idx in enumerate(top5_idx):
    dist = hvn_prices[idx] - current_price
    print(f"  {i+1}. ${hvn_prices[idx]:,.2f}  ({dist:+.1f} from current)")

print(f"\n  LVN zones (fast travel, price moves quickly through these):")
lvn_prices = price_centers[lvn_bins]
if len(lvn_prices) > 0:
    # Group into contiguous ranges
    gaps = np.where(np.diff(lvn_bins) > 3)[0]
    starts = np.concatenate([[0], gaps+1])
    ends   = np.concatenate([gaps, [len(lvn_bins)-1]])
    for s, e in zip(starts, ends):
        if e - s >= 3:
            print(f"  ${lvn_prices[s]:,.1f} → ${lvn_prices[e]:,.1f}")

print(f"\n  📊  Saved: Elena_Gold_Liquidity_Heatmap.png")
print(f"{'═'*60}\n")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ELENA GOLD — LARGE ORDER DETECTION  (QUANT EDITION)
#  Yellow candles = bullish  |  Black candles = bearish  |  Dark blue bg
#  Paste entire cell into Jupyter and run
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
from datetime import datetime, timedelta
import pytz
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter1d

warnings.filterwarnings('ignore')

ATHENS = pytz.timezone('Europe/Athens')
NOW    = datetime.now(ATHENS)

# ── CONFIG ────────────────────────────────────────────────────────────────────
LOOKBACK_DAYS      = 10
VOL_SPIKE_MULT     = 2.5
VOL_ROLLING_WINDOW = 20
PRICE_MOVE_MULT    = 1.5
ATR_PERIOD         = 14
ABSORPTION_RATIO   = 0.35
LARGE_WICK_RATIO   = 0.55
MIN_CLUSTER_BARS   = 2

# ── PALETTE ───────────────────────────────────────────────────────────────────
C = dict(
    bg          = '#020818',   # very dark navy
    panel       = '#030d24',   # dark blue panel
    panel2      = '#040f2a',   # slightly lighter panel
    border      = '#0a1f4a',   # blue border
    grid        = '#071430',   # subtle grid lines
    bull        = '#F0C040',   # gold/yellow candles
    bull_wick   = '#C9A030',   # slightly darker for wicks
    bear        = '#0a0a0f',   # near-black candles
    bear_wick   = '#1a1a2e',   # dark wick
    bear_edge   = '#2a2a4a',   # bear candle edge so it's visible on dark bg
    gold        = '#F0C040',
    gold2       = '#FFD700',
    green       = '#00FF88',
    red         = '#FF2244',
    blue        = '#1E90FF',
    cyan        = '#00E5FF',
    orange      = '#FF6600',
    purple      = '#BB66FF',
    amber       = '#FF9900',
    white       = '#E8F0FF',
    lgrey       = '#3A5080',
    grey        = '#1A2A50',
    vwap        = '#CC44FF',
    # signal colours
    sig_abuy    = '#00FF88',
    sig_asell   = '#FF2244',
    sig_absbuy  = '#00E5FF',
    sig_abssell = '#FF6600',
    sig_stop    = '#FFD700',
    sig_iceberg = '#1E90FF',
    sig_cluster = '#FF990022',
)

print("📥  Fetching GC=F 1H data...")

# ── FETCH ─────────────────────────────────────────────────────────────────────
raw = yf.download('GC=F', period=f'{LOOKBACK_DAYS+3}d', interval='1h',
                  auto_adjust=True, progress=False)
if hasattr(raw, 'columns') and isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
if raw.index.tz is None:
    raw.index = raw.index.tz_localize('UTC').tz_convert(ATHENS)
else:
    raw.index = raw.index.tz_convert(ATHENS)

df = raw[raw['Volume'] > 0].copy()
df = df[df.index >= (NOW - timedelta(days=LOOKBACK_DAYS))]
df.dropna(inplace=True)

print(f"  ✓  {len(df)} bars  |  "
      f"{df.index[0].strftime('%d %b %H:%M')} → {df.index[-1].strftime('%d %b %H:%M')} Athens")

# ── INDICATORS ────────────────────────────────────────────────────────────────
df['vol_ma']     = df['Volume'].rolling(VOL_ROLLING_WINDOW).mean()
df['vol_ratio']  = df['Volume'] / df['vol_ma']
df['vol_zscore'] = ((df['Volume'] - df['Volume'].rolling(VOL_ROLLING_WINDOW).mean()) /
                     df['Volume'].rolling(VOL_ROLLING_WINDOW).std())

df['tr']  = np.maximum(df['High'] - df['Low'],
            np.maximum(abs(df['High'] - df['Close'].shift(1)),
                       abs(df['Low']  - df['Close'].shift(1))))
df['atr'] = df['tr'].rolling(ATR_PERIOD).mean()

df['bar_range']  = df['High'] - df['Low']
df['body_size']  = abs(df['Close'] - df['Open'])
df['upper_wick'] = df['High']  - df[['Close','Open']].max(axis=1)
df['lower_wick'] = df[['Close','Open']].min(axis=1) - df['Low']
df['body_ratio'] = df['body_size'] / df['bar_range'].replace(0, np.nan)
df['price_move'] = abs(df['Close'] - df['Open'])
df['is_bullish'] = df['Close'] > df['Open']

df['date']      = df.index.date
df['tp']        = (df['High'] + df['Low'] + df['Close']) / 3
df['tpvol']     = df['tp'] * df['Volume']
df['cum_vol']   = df.groupby('date')['Volume'].cumsum()
df['cum_tpvol'] = df.groupby('date')['tpvol'].cumsum()
df['vwap']      = df['cum_tpvol'] / df['cum_vol']

df['bull_vol']  = np.where(df['is_bullish'], df['Volume'], df['Volume'] * 0.3)
df['bear_vol']  = np.where(~df['is_bullish'], df['Volume'], df['Volume'] * 0.3)
df['delta']     = df['bull_vol'] - df['bear_vol']
df['cum_delta'] = df['delta'].cumsum()

# ── SIGNAL DETECTION ──────────────────────────────────────────────────────────
sigs = ['sig_aggressive_buy','sig_aggressive_sell','sig_absorption_buy',
        'sig_absorption_sell','sig_stop_hunt_low','sig_stop_hunt_high',
        'sig_iceberg','sig_volume_spike']
for s in sigs:
    df[s] = False

for i in range(ATR_PERIOD, len(df)):
    r       = df.iloc[i]
    atr     = r['atr']
    vr      = r['vol_ratio']
    brng    = r['bar_range']
    body    = r['body_size']
    move    = r['price_move']
    uw      = r['upper_wick']
    lw      = r['lower_wick']
    bull    = r['is_bullish']
    if pd.isna(atr) or pd.isna(vr) or brng == 0: continue

    if vr >= VOL_SPIKE_MULT and bull    and move >= atr*PRICE_MOVE_MULT and body/brng > 0.6:
        df.iloc[i, df.columns.get_loc('sig_aggressive_buy')]   = True
    if vr >= VOL_SPIKE_MULT and not bull and move >= atr*PRICE_MOVE_MULT and body/brng > 0.6:
        df.iloc[i, df.columns.get_loc('sig_aggressive_sell')]  = True
    if vr >= VOL_SPIKE_MULT and body/brng < ABSORPTION_RATIO and lw/brng > 0.3:
        df.iloc[i, df.columns.get_loc('sig_absorption_buy')]   = True
    if vr >= VOL_SPIKE_MULT and body/brng < ABSORPTION_RATIO and uw/brng > 0.3:
        df.iloc[i, df.columns.get_loc('sig_absorption_sell')]  = True
    if lw/brng > LARGE_WICK_RATIO and vr >= 1.5 and bull:
        df.iloc[i, df.columns.get_loc('sig_stop_hunt_low')]    = True
    if uw/brng > LARGE_WICK_RATIO and vr >= 1.5 and not bull:
        df.iloc[i, df.columns.get_loc('sig_stop_hunt_high')]   = True
    if vr >= VOL_SPIKE_MULT * 1.2 and brng < atr * 0.4:
        df.iloc[i, df.columns.get_loc('sig_iceberg')]          = True
    if vr >= VOL_SPIKE_MULT:
        df.iloc[i, df.columns.get_loc('sig_volume_spike')]     = True

df['sig_cluster'] = False
spk = df['sig_volume_spike'].values
for i in range(len(spk) - MIN_CLUSTER_BARS + 1):
    if all(spk[i:i+MIN_CLUSTER_BARS]):
        df.iloc[i:i+MIN_CLUSTER_BARS, df.columns.get_loc('sig_cluster')] = True

# ── FIGURE SETUP ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'monospace',
    'axes.facecolor':    C['panel'],
    'figure.facecolor':  C['bg'],
    'text.color':        C['white'],
    'axes.labelcolor':   C['lgrey'],
    'xtick.color':       C['lgrey'],
    'ytick.color':       C['lgrey'],
    'grid.color':        C['grid'],
    'grid.linewidth':    0.4,
    'grid.alpha':        1.0,
})

fig = plt.figure(figsize=(26, 18), facecolor=C['bg'])
gs  = gridspec.GridSpec(5, 1, figure=fig,
                        height_ratios=[5, 0.08, 1.4, 1.4, 1.0],
                        hspace=0.0,
                        left=0.055, right=0.975, top=0.945, bottom=0.048)

ax_price  = fig.add_subplot(gs[0])
ax_div    = fig.add_subplot(gs[1])   # thin divider
ax_vol    = fig.add_subplot(gs[2], sharex=ax_price)
ax_delta  = fig.add_subplot(gs[3], sharex=ax_price)
ax_zscore = fig.add_subplot(gs[4], sharex=ax_price)

ax_div.set_visible(False)

def style(ax, title='', ylabel='', grid=True):
    ax.set_facecolor(C['panel'])
    ax.tick_params(colors=C['lgrey'], labelsize=7.5, length=3)
    for sp in ax.spines.values():
        sp.set_color(C['border'])
        sp.set_linewidth(0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if grid:
        ax.yaxis.grid(True, color=C['grid'], linewidth=0.4, alpha=1)
        ax.set_axisbelow(True)
    if title:
        ax.text(0.005, 0.97, title, transform=ax.transAxes,
                color=C['gold'], fontsize=8.5, fontweight='bold',
                va='top', ha='left', fontfamily='monospace')
    if ylabel:
        ax.set_ylabel(ylabel, color=C['lgrey'], fontsize=7.5)

x = np.arange(len(df))

# ── PANEL 1: CANDLESTICK CHART ────────────────────────────────────────────────
ax_price.set_facecolor(C['panel'])

# Volume cluster background glow
for i, (ts, row) in enumerate(df.iterrows()):
    if row['sig_cluster']:
        ax_price.axvspan(i-0.5, i+0.5, alpha=0.07, color=C['amber'], zorder=0)

# Candlesticks — yellow/gold for bull, near-black for bear
for i, (ts, row) in enumerate(df.iterrows()):
    o, h, l, c = row['Open'], row['High'], row['Low'], row['Close']
    bull = c >= o

    if bull:
        wk_col   = C['bull_wick']
        body_col = C['bull']
        edge_col = C['bull']
        alpha_b  = 0.92
    else:
        wk_col   = C['bear_edge']
        body_col = C['bear']
        edge_col = C['bear_edge']
        alpha_b  = 1.0

    # Wick
    ax_price.plot([i, i], [l, h], color=wk_col, lw=0.9, alpha=0.8, zorder=2)

    # Body
    rect_y = min(o, c)
    rect_h = max(abs(c - o), 0.3)
    ax_price.add_patch(plt.Rectangle(
        (i - 0.38, rect_y), 0.76, rect_h,
        facecolor=body_col, edgecolor=edge_col,
        linewidth=0.5, alpha=alpha_b, zorder=3
    ))

# VWAP with glow effect
ax_price.plot(x, df['vwap'].values, color=C['vwap'], lw=1.8,
              alpha=0.5, zorder=4, label='VWAP')
ax_price.plot(x, df['vwap'].values, color=C['vwap'], lw=0.8,
              alpha=0.9, zorder=5)

# EMA 20 subtle
ema20 = df['Close'].ewm(span=20).mean()
ax_price.plot(x, ema20.values, color=C['blue'], lw=0.8,
              alpha=0.5, zorder=4, ls='--', label='EMA 20')

atr_mean = df['atr'].mean()

# ── SIGNAL MARKERS ────────────────────────────────────────────────────────────
for i, (ts, row) in enumerate(df.iterrows()):
    atr_o = row['atr'] if not pd.isna(row['atr']) else atr_mean

    # Aggressive Buy — bright green arrow + label
    if row['sig_aggressive_buy']:
        y_pos = row['Low'] - atr_o * 0.9
        ax_price.annotate('', xy=(i, row['Low'] - atr_o*0.1),
                          xytext=(i, y_pos),
                          arrowprops=dict(arrowstyle='->', color=C['sig_abuy'],
                                         lw=1.8), zorder=8)
        ax_price.text(i, y_pos - atr_o*0.15, 'AGG\nBUY',
                      color=C['sig_abuy'], fontsize=5, ha='center', va='top',
                      fontfamily='monospace', fontweight='bold', zorder=9)

    # Aggressive Sell — red arrow down
    if row['sig_aggressive_sell']:
        y_pos = row['High'] + atr_o * 0.9
        ax_price.annotate('', xy=(i, row['High'] + atr_o*0.1),
                          xytext=(i, y_pos),
                          arrowprops=dict(arrowstyle='->', color=C['sig_asell'],
                                         lw=1.8), zorder=8)
        ax_price.text(i, y_pos + atr_o*0.15, 'AGG\nSELL',
                      color=C['sig_asell'], fontsize=5, ha='center', va='bottom',
                      fontfamily='monospace', fontweight='bold', zorder=9)

    # Absorption Buy — cyan diamond below
    if row['sig_absorption_buy']:
        ax_price.scatter(i, row['Low'] - atr_o*0.3, marker='D', s=55,
                        color=C['sig_absbuy'], zorder=7, linewidth=0, alpha=0.9)

    # Absorption Sell — orange diamond above
    if row['sig_absorption_sell']:
        ax_price.scatter(i, row['High'] + atr_o*0.3, marker='D', s=55,
                        color=C['sig_abssell'], zorder=7, linewidth=0, alpha=0.9)

    # Stop Hunt — gold X
    if row['sig_stop_hunt_low']:
        ax_price.scatter(i, row['Low'] - atr_o*0.2, marker='x', s=90,
                        color=C['sig_stop'], zorder=7, linewidth=2.0, alpha=1.0)
    if row['sig_stop_hunt_high']:
        ax_price.scatter(i, row['High'] + atr_o*0.2, marker='x', s=90,
                        color=C['sig_stop'], zorder=7, linewidth=2.0, alpha=1.0)

    # Iceberg — blue square on candle body
    if row['sig_iceberg']:
        ax_price.scatter(i, (row['High'] + row['Low'])/2, marker='s', s=65,
                        color=C['sig_iceberg'], zorder=7, alpha=0.85,
                        linewidth=0.8, edgecolors=C['cyan'])

# Current price dotted line + label
last_price = float(np.asarray(df['Close'])[-1])
ax_price.axhline(last_price, color=C['gold'], lw=0.7, ls=':', alpha=0.7, zorder=6)
ax_price.text(len(df) - 0.5, last_price,
              f'  ▶ {last_price:,.1f}',
              color=C['gold2'], fontsize=9, va='center',
              fontweight='bold', fontfamily='monospace', zorder=10)

style(ax_price,
      title=f'GC FUTURES  ·  LARGE ORDER DETECTION  ·  1H  ·  '
            f'LAST {LOOKBACK_DAYS} DAYS  ·  CURRENT ${last_price:,.1f}',
      ylabel='USD/oz')
ax_price.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax_price.yaxis.grid(True, color=C['grid'], linewidth=0.4)

# Legend
legend_items = [
    mpatches.Patch(facecolor=C['bull'],       edgecolor=C['bull'],       label='Bullish candle'),
    mpatches.Patch(facecolor=C['bear'],       edgecolor=C['bear_edge'],  label='Bearish candle'),
    mpatches.Patch(facecolor=C['sig_abuy'],   label='▲ Aggressive Buy'),
    mpatches.Patch(facecolor=C['sig_asell'],  label='▼ Aggressive Sell'),
    mpatches.Patch(facecolor=C['sig_absbuy'], label='◆ Absorption Buy'),
    mpatches.Patch(facecolor=C['sig_abssell'],label='◆ Absorption Sell'),
    mpatches.Patch(facecolor=C['sig_stop'],   label='✕ Stop Hunt'),
    mpatches.Patch(facecolor=C['sig_iceberg'],label='■ Iceberg'),
    mpatches.Patch(facecolor=C['vwap'],       label='— VWAP'),
    mpatches.Patch(facecolor=C['blue'],       label='-- EMA 20'),
    mpatches.Patch(facecolor=C['amber'],      alpha=0.4, label='░ Vol Cluster'),
]
ax_price.legend(handles=legend_items, loc='upper left',
                facecolor=C['bg'], edgecolor=C['border'],
                labelcolor=C['lgrey'], fontsize=6.8,
                ncol=6, framealpha=0.95,
                prop={'family': 'monospace'})

# ── PANEL 2: VOLUME ───────────────────────────────────────────────────────────
vol_vals = df['Volume'].values
vol_cols = []
for i, (ts, row) in enumerate(df.iterrows()):
    if row['sig_aggressive_buy']:    vol_cols.append(C['sig_abuy'])
    elif row['sig_aggressive_sell']: vol_cols.append(C['sig_asell'])
    elif row['sig_iceberg']:         vol_cols.append(C['sig_iceberg'])
    elif row['sig_absorption_buy'] or row['sig_absorption_sell']:
                                     vol_cols.append(C['cyan'])
    elif row['sig_cluster']:         vol_cols.append(C['amber'])
    elif row['is_bullish']:          vol_cols.append('#1A3A1A')
    else:                            vol_cols.append('#1A0A0F')

ax_vol.bar(x, vol_vals, color=vol_cols, edgecolor='none', width=0.85, zorder=3)

# Volume MA line with glow
vol_ma_s = gaussian_filter1d(df['vol_ma'].fillna(0).values, sigma=1.5)
ax_vol.plot(x, vol_ma_s, color=C['gold'], lw=1.4, alpha=0.9, zorder=4)
ax_vol.plot(x, vol_ma_s, color=C['gold'], lw=3.0, alpha=0.15, zorder=3)

# Spike threshold
thresh_line = df['vol_ma'].mean() * VOL_SPIKE_MULT
ax_vol.axhline(thresh_line, color=C['red'], lw=0.7, ls='--', alpha=0.6, zorder=5)

style(ax_vol,
      title='VOLUME  ·  bright=signal  dim=normal  gold=MA  red--=spike threshold',
      ylabel='contracts')
ax_vol.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))

# ── PANEL 3: CUMULATIVE DELTA ─────────────────────────────────────────────────
cd    = df['cum_delta'].values
cd_s  = gaussian_filter1d(cd, sigma=1.5)

# Gradient fill
ax_delta.fill_between(x, cd_s, 0,
                      where=cd_s >= 0, alpha=0.25, color=C['green'], zorder=2)
ax_delta.fill_between(x, cd_s, 0,
                      where=cd_s <  0, alpha=0.25, color=C['red'],   zorder=2)
ax_delta.plot(x, cd_s, color=C['white'], lw=1.0, alpha=0.6, zorder=3)

# Highlight positive/negative regions with stronger line
ax_delta.plot(x, np.where(cd_s >= 0, cd_s, np.nan),
              color=C['green'], lw=1.4, alpha=0.9, zorder=4)
ax_delta.plot(x, np.where(cd_s <  0, cd_s, np.nan),
              color=C['red'],   lw=1.4, alpha=0.9, zorder=4)
ax_delta.axhline(0, color=C['lgrey'], lw=0.5, zorder=3)

# Divergence markers
price_arr = df['Close'].values
win = 10
for i in range(win, len(df)):
    pt = price_arr[i] - price_arr[i-win]
    dt = cd[i] - cd[i-win]
    if pt > 0 and dt < -abs(cd[i-win])*0.15:
        ax_delta.scatter(i, cd_s[i], marker='v', s=45,
                        color=C['red'], zorder=6, alpha=0.9)
    elif pt < 0 and dt > abs(cd[i-win])*0.15:
        ax_delta.scatter(i, cd_s[i], marker='^', s=45,
                        color=C['green'], zorder=6, alpha=0.9)

style(ax_delta,
      title='CUMULATIVE DELTA PROXY  ·  green=buy pressure  red=sell pressure  ▲▼=divergence',
      ylabel='Δvol')
ax_delta.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.1f}k'))

# ── PANEL 4: VOLUME Z-SCORE ───────────────────────────────────────────────────
zs = df['vol_zscore'].fillna(0).values

# Coloured bars by intensity
zs_cols = []
for z in zs:
    if z >= VOL_SPIKE_MULT:    zs_cols.append(C['red'])
    elif z >= 2.0:             zs_cols.append(C['orange'])
    elif z >= 1.5:             zs_cols.append(C['amber'])
    elif z >= 0:               zs_cols.append('#1a3a1a')
    else:                      zs_cols.append('#0d0d1a')

ax_zscore.bar(x, zs, color=zs_cols, edgecolor='none', width=0.85, zorder=3)
ax_zscore.axhline(VOL_SPIKE_MULT, color=C['red'],    lw=0.9, ls='--', alpha=0.7, zorder=4)
ax_zscore.axhline(1.5,            color=C['amber'],  lw=0.7, ls='--', alpha=0.5, zorder=4)
ax_zscore.axhline(0,              color=C['lgrey'],  lw=0.4,          alpha=0.6, zorder=3)

# Label threshold
ax_zscore.text(len(df)*0.99, VOL_SPIKE_MULT + 0.1,
               f'{VOL_SPIKE_MULT}σ', color=C['red'],
               fontsize=6.5, ha='right', fontfamily='monospace')

style(ax_zscore,
      title='VOLUME Z-SCORE  ·  red=institutional spike  orange=elevated  amber=above avg',
      ylabel='σ')

# ── SHARED X-AXIS ─────────────────────────────────────────────────────────────
n_ticks   = 14
tick_idxs = np.linspace(0, len(df)-1, n_ticks, dtype=int)
tick_lbls = [df.index[i].strftime('%d/%m\n%H:%M') for i in tick_idxs]
ax_zscore.set_xticks(tick_idxs)
ax_zscore.set_xticklabels(tick_lbls, color=C['lgrey'], fontsize=7,
                           fontfamily='monospace')
for ax in [ax_price, ax_vol, ax_delta]:
    plt.setp(ax.get_xticklabels(), visible=False)

# Sync x limits
for ax in [ax_price, ax_vol, ax_delta, ax_zscore]:
    ax.set_xlim(-0.8, len(df) - 0.2)

# ── TITLE BAR ─────────────────────────────────────────────────────────────────
fig.text(0.50, 0.974,
         f'ELENA GOLD  ·  INSTITUTIONAL ORDER FLOW  ·  GC FUTURES 1H  ·  '
         f'{NOW.strftime("%A %d %B %Y  %H:%M")} ATHENS',
         color=C['gold2'], fontsize=11, fontweight='bold',
         ha='center', va='center', fontfamily='monospace')

# ── SEPARATOR LINES between panels ───────────────────────────────────────────
for ax in [ax_price, ax_vol, ax_delta]:
    ax.spines['bottom'].set_color(C['border'])
    ax.spines['bottom'].set_linewidth(0.8)

plt.savefig('Elena_Gold_Order_Flow_Quant.png', dpi=160,
            bbox_inches='tight', facecolor=C['bg'])
plt.show()

# ── CONSOLE SUMMARY ───────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  SIGNAL SUMMARY  ·  LAST {LOOKBACK_DAYS} DAYS")
print(f"{'═'*60}")
sig_map = {
    'sig_aggressive_buy':   ('AGG BUY',    '▲', C['sig_abuy']),
    'sig_aggressive_sell':  ('AGG SELL',   '▼', C['sig_asell']),
    'sig_absorption_buy':   ('ABS BUY',    '◆', C['sig_absbuy']),
    'sig_absorption_sell':  ('ABS SELL',   '◆', C['sig_abssell']),
    'sig_stop_hunt_low':    ('STOP ↓',     '✕', C['sig_stop']),
    'sig_stop_hunt_high':   ('STOP ↑',     '✕', C['sig_stop']),
    'sig_iceberg':          ('ICEBERG',    '■', C['sig_iceberg']),
    'sig_cluster':          ('CLUSTER',    '░', C['amber']),
}
for col, (name, icon, _) in sig_map.items():
    n   = int(df[col].sum())
    bar = '█' * n + '░' * max(0, 20-n)
    print(f"  {icon} {name:<14} {n:3d}  {bar}")

print(f"\n{'─'*60}")
print(f"  LAST 15 BARS WITH SIGNALS")
print(f"{'─'*60}")
sig_cols  = list(sig_map.keys())[:-1]
sig_names = [v[0] for v in list(sig_map.values())[:-1]]
for idx, row in df.tail(30).iterrows():
    fired = [sig_names[i] for i, col in enumerate(sig_cols) if row[col]]
    if fired:
        vr  = row['vol_ratio']
        dir = '↑' if row['is_bullish'] else '↓'
        print(f"  {idx.strftime('%d/%m %H:%M')}  {dir}  ${row['Close']:,.1f}"
              f"  vol×{vr:.1f}  →  {' + '.join(fired)}")

print(f"\n  📊  Saved: Elena_Gold_Order_Flow_Quant.png")
print(f"{'═'*60}\n")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ELENA GOLD — LARGE ORDER DETECTION  (XAUUSD 15M  QUANT EDITION)
#  Yellow candles = bullish  |  Black candles = bearish  |  Dark blue bg
#  Paste entire cell into Jupyter and run
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
from datetime import datetime, timedelta
import pytz
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter1d

warnings.filterwarnings('ignore')

ATHENS = pytz.timezone('Europe/Athens')
NOW    = datetime.now(ATHENS)

# ── CONFIG ────────────────────────────────────────────────────────────────────
LOOKBACK_DAYS      = 2        # 2 days of 15m = ~192 bars
VOL_SPIKE_MULT     = 1.8      # lower than 1H — 15m bars have smaller vol spikes
VOL_ROLLING_WINDOW = 14
PRICE_MOVE_MULT    = 1.2      # lower for 15m — moves are smaller per bar
ATR_PERIOD         = 14
ABSORPTION_RATIO   = 0.35
LARGE_WICK_RATIO   = 0.50
MIN_CLUSTER_BARS   = 2

# Tickers to try in order — first one with data wins
TICKER_CANDIDATES = ['XAUUSD=X', 'GC=F', 'GLD']

# ── PALETTE ───────────────────────────────────────────────────────────────────
C = dict(
    bg          = '#020818',
    panel       = '#030d24',
    border      = '#0a1f4a',
    grid        = '#071430',
    bull        = '#F0C040',
    bull_wick   = '#C9A030',
    bear        = '#0a0a0f',
    bear_edge   = '#2a2a4a',
    gold        = '#F0C040',
    gold2       = '#FFD700',
    green       = '#00FF88',
    red         = '#FF2244',
    blue        = '#1E90FF',
    cyan        = '#00E5FF',
    orange      = '#FF6600',
    purple      = '#BB66FF',
    amber       = '#FF9900',
    white       = '#E8F0FF',
    lgrey       = '#3A5080',
    grey        = '#1A2A50',
    vwap        = '#CC44FF',
    sig_abuy    = '#00FF88',
    sig_asell   = '#FF2244',
    sig_absbuy  = '#00E5FF',
    sig_abssell = '#FF6600',
    sig_stop    = '#FFD700',
    sig_iceberg = '#1E90FF',
)

# ── FETCH — try candidates until one works ────────────────────────────────────
print("📥  Fetching 15m data...")

df       = pd.DataFrame()
TICKER   = None

for _t in TICKER_CANDIDATES:
    try:
        raw = yf.download(_t, period='5d', interval='15m',
                          auto_adjust=True, progress=False)
        if hasattr(raw, 'columns') and isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        if raw.index.tz is None:
            raw.index = raw.index.tz_localize('UTC').tz_convert(ATHENS)
        else:
            raw.index = raw.index.tz_convert(ATHENS)
        raw = raw[raw['Volume'] > 0].copy()
        raw.dropna(inplace=True)
        cutoff = NOW - timedelta(days=LOOKBACK_DAYS)
        raw    = raw[raw.index >= cutoff]
        if len(raw) > 20:
            df     = raw.copy()
            TICKER = _t
            break
    except Exception as e:
        print(f"  ✗  {_t}: {e}")

if df.empty:
    raise RuntimeError("No data found for any ticker. Check connection.")

print(f"  ✓  {TICKER}  |  {len(df)} bars  |  "
      f"{df.index[0].strftime('%d %b %H:%M')} → "
      f"{df.index[-1].strftime('%d %b %H:%M')} Athens")

# ── INDICATORS ────────────────────────────────────────────────────────────────
df['vol_ma']     = df['Volume'].rolling(VOL_ROLLING_WINDOW).mean()
df['vol_ratio']  = df['Volume'] / df['vol_ma']
df['vol_zscore'] = ((df['Volume'] - df['Volume'].rolling(VOL_ROLLING_WINDOW).mean()) /
                     df['Volume'].rolling(VOL_ROLLING_WINDOW).std())

df['tr']  = np.maximum(df['High'] - df['Low'],
            np.maximum(abs(df['High'] - df['Close'].shift(1)),
                       abs(df['Low']  - df['Close'].shift(1))))
df['atr'] = df['tr'].rolling(ATR_PERIOD).mean()

df['bar_range']  = df['High'] - df['Low']
df['body_size']  = abs(df['Close'] - df['Open'])
df['upper_wick'] = df['High']  - df[['Close','Open']].max(axis=1)
df['lower_wick'] = df[['Close','Open']].min(axis=1) - df['Low']
df['body_ratio'] = df['body_size'] / df['bar_range'].replace(0, np.nan)
df['price_move'] = abs(df['Close'] - df['Open'])
df['is_bullish'] = df['Close'] >= df['Open']

# VWAP — daily reset
df['date']      = df.index.date
df['tp']        = (df['High'] + df['Low'] + df['Close']) / 3
df['tpvol']     = df['tp'] * df['Volume']
df['cum_vol']   = df.groupby('date')['Volume'].cumsum()
df['cum_tpvol'] = df.groupby('date')['tpvol'].cumsum()
df['vwap']      = df['cum_tpvol'] / df['cum_vol']

# EMA 20 and 50
df['ema20'] = df['Close'].ewm(span=20).mean()
df['ema50'] = df['Close'].ewm(span=50).mean()

# Cumulative delta proxy
df['bull_vol']  = np.where(df['is_bullish'], df['Volume'], df['Volume'] * 0.3)
df['bear_vol']  = np.where(~df['is_bullish'], df['Volume'], df['Volume'] * 0.3)
df['delta']     = df['bull_vol'] - df['bear_vol']
df['cum_delta'] = df['delta'].cumsum()

# ── SIGNAL DETECTION ──────────────────────────────────────────────────────────
sigs = ['sig_aggressive_buy','sig_aggressive_sell','sig_absorption_buy',
        'sig_absorption_sell','sig_stop_hunt_low','sig_stop_hunt_high',
        'sig_iceberg','sig_volume_spike','sig_cluster']
for s in sigs:
    df[s] = False

for i in range(ATR_PERIOD, len(df)):
    r    = df.iloc[i]
    atr  = r['atr']
    vr   = r['vol_ratio']
    brng = r['bar_range']
    body = r['body_size']
    move = r['price_move']
    uw   = r['upper_wick']
    lw   = r['lower_wick']
    bull = r['is_bullish']
    if pd.isna(atr) or pd.isna(vr) or brng == 0: continue

    if vr >= VOL_SPIKE_MULT and bull     and move >= atr*PRICE_MOVE_MULT and body/brng > 0.55:
        df.iloc[i, df.columns.get_loc('sig_aggressive_buy')]  = True
    if vr >= VOL_SPIKE_MULT and not bull and move >= atr*PRICE_MOVE_MULT and body/brng > 0.55:
        df.iloc[i, df.columns.get_loc('sig_aggressive_sell')] = True
    if vr >= VOL_SPIKE_MULT and body/brng < ABSORPTION_RATIO and lw/brng > 0.3:
        df.iloc[i, df.columns.get_loc('sig_absorption_buy')]  = True
    if vr >= VOL_SPIKE_MULT and body/brng < ABSORPTION_RATIO and uw/brng > 0.3:
        df.iloc[i, df.columns.get_loc('sig_absorption_sell')] = True
    if lw/brng > LARGE_WICK_RATIO and vr >= 1.3 and bull:
        df.iloc[i, df.columns.get_loc('sig_stop_hunt_low')]   = True
    if uw/brng > LARGE_WICK_RATIO and vr >= 1.3 and not bull:
        df.iloc[i, df.columns.get_loc('sig_stop_hunt_high')]  = True
    if vr >= VOL_SPIKE_MULT * 1.1 and brng < atr * 0.4:
        df.iloc[i, df.columns.get_loc('sig_iceberg')]         = True
    if vr >= VOL_SPIKE_MULT:
        df.iloc[i, df.columns.get_loc('sig_volume_spike')]    = True

# Cluster detection
spk = df['sig_volume_spike'].values
for i in range(len(spk) - MIN_CLUSTER_BARS + 1):
    if all(spk[i:i+MIN_CLUSTER_BARS]):
        df.iloc[i:i+MIN_CLUSTER_BARS, df.columns.get_loc('sig_cluster')] = True

# ── FIGURE ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'monospace',
    'axes.facecolor':   C['panel'],
    'figure.facecolor': C['bg'],
    'text.color':       C['white'],
    'axes.labelcolor':  C['lgrey'],
    'xtick.color':      C['lgrey'],
    'ytick.color':      C['lgrey'],
    'grid.color':       C['grid'],
    'grid.linewidth':   0.4,
})

fig = plt.figure(figsize=(26, 18), facecolor=C['bg'])
gs  = gridspec.GridSpec(4, 1, figure=fig,
                        height_ratios=[5, 1.4, 1.4, 1.0],
                        hspace=0.0,
                        left=0.055, right=0.975,
                        top=0.945, bottom=0.048)

ax_price  = fig.add_subplot(gs[0])
ax_vol    = fig.add_subplot(gs[1], sharex=ax_price)
ax_delta  = fig.add_subplot(gs[2], sharex=ax_price)
ax_zscore = fig.add_subplot(gs[3], sharex=ax_price)

def style(ax, title='', ylabel=''):
    ax.set_facecolor(C['panel'])
    ax.tick_params(colors=C['lgrey'], labelsize=7.5, length=3)
    for sp in ax.spines.values():
        sp.set_color(C['border'])
        sp.set_linewidth(0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, color=C['grid'], linewidth=0.4, alpha=1)
    ax.set_axisbelow(True)
    if title:
        ax.text(0.005, 0.97, title, transform=ax.transAxes,
                color=C['gold'], fontsize=8, fontweight='bold',
                va='top', ha='left', fontfamily='monospace')
    if ylabel:
        ax.set_ylabel(ylabel, color=C['lgrey'], fontsize=7.5)

x       = np.arange(len(df))
atr_avg = df['atr'].mean()

# ── PANEL 1: CANDLESTICKS ─────────────────────────────────────────────────────

# Session background shading — London and NY
for i, (ts, row) in enumerate(df.iterrows()):
    h = ts.hour
    if 10 <= h < 16:   # London
        ax_price.axvspan(i-0.5, i+0.5, alpha=0.04, color=C['blue'],  zorder=0)
    elif 16 <= h < 23: # NY
        ax_price.axvspan(i-0.5, i+0.5, alpha=0.04, color=C['green'], zorder=0)

# Volume cluster glow
for i, (ts, row) in enumerate(df.iterrows()):
    if row['sig_cluster']:
        ax_price.axvspan(i-0.5, i+0.5, alpha=0.09, color=C['amber'], zorder=0)

# Candles
for i, (ts, row) in enumerate(df.iterrows()):
    o, h, l, c = row['Open'], row['High'], row['Low'], row['Close']
    bull = c >= o
    if bull:
        wk_col, body_col, edge_col, ab = C['bull_wick'], C['bull'],  C['bull'],      0.90
    else:
        wk_col, body_col, edge_col, ab = C['bear_edge'], C['bear'],  C['bear_edge'], 1.00

    ax_price.plot([i, i], [l, h], color=wk_col, lw=0.8, alpha=0.75, zorder=2)
    rect_h = max(abs(c - o), 0.05)
    ax_price.add_patch(plt.Rectangle(
        (i-0.38, min(o,c)), 0.76, rect_h,
        facecolor=body_col, edgecolor=edge_col,
        linewidth=0.4, alpha=ab, zorder=3))

# VWAP
ax_price.plot(x, df['vwap'].values,  color=C['vwap'], lw=1.6, alpha=0.50, zorder=4)
ax_price.plot(x, df['vwap'].values,  color=C['vwap'], lw=0.8, alpha=0.90, zorder=5,
              label='VWAP')

# EMAs
ax_price.plot(x, df['ema20'].values, color=C['blue'],  lw=0.9, alpha=0.60,
              zorder=4, ls='--', label='EMA 20')
ax_price.plot(x, df['ema50'].values, color=C['orange'], lw=0.9, alpha=0.60,
              zorder=4, ls='--', label='EMA 50')

# ── SIGNALS ───────────────────────────────────────────────────────────────────
for i, (ts, row) in enumerate(df.iterrows()):
    atr_o = row['atr'] if not pd.isna(row['atr']) else atr_avg
    if atr_o == 0: atr_o = atr_avg

    if row['sig_aggressive_buy']:
        ax_price.annotate('',
            xy=(i, row['Low'] - atr_o*0.15),
            xytext=(i, row['Low'] - atr_o*0.95),
            arrowprops=dict(arrowstyle='->', color=C['sig_abuy'], lw=2.0), zorder=8)
        ax_price.text(i, row['Low'] - atr_o*1.05, 'AGG\nBUY',
                      color=C['sig_abuy'], fontsize=5.5, ha='center', va='top',
                      fontweight='bold', fontfamily='monospace', zorder=9)

    if row['sig_aggressive_sell']:
        ax_price.annotate('',
            xy=(i, row['High'] + atr_o*0.15),
            xytext=(i, row['High'] + atr_o*0.95),
            arrowprops=dict(arrowstyle='->', color=C['sig_asell'], lw=2.0), zorder=8)
        ax_price.text(i, row['High'] + atr_o*1.05, 'AGG\nSELL',
                      color=C['sig_asell'], fontsize=5.5, ha='center', va='bottom',
                      fontweight='bold', fontfamily='monospace', zorder=9)

    if row['sig_absorption_buy']:
        ax_price.scatter(i, row['Low'] - atr_o*0.25, marker='D', s=50,
                        color=C['sig_absbuy'], zorder=7, linewidth=0, alpha=0.9)

    if row['sig_absorption_sell']:
        ax_price.scatter(i, row['High'] + atr_o*0.25, marker='D', s=50,
                        color=C['sig_abssell'], zorder=7, linewidth=0, alpha=0.9)

    if row['sig_stop_hunt_low']:
        ax_price.scatter(i, row['Low'] - atr_o*0.15, marker='x', s=80,
                        color=C['sig_stop'], zorder=7, linewidth=2.2)

    if row['sig_stop_hunt_high']:
        ax_price.scatter(i, row['High'] + atr_o*0.15, marker='x', s=80,
                        color=C['sig_stop'], zorder=7, linewidth=2.2)

    if row['sig_iceberg']:
        ax_price.scatter(i, (row['High']+row['Low'])/2, marker='s', s=60,
                        color=C['sig_iceberg'], zorder=7, alpha=0.85,
                        linewidth=0.8, edgecolors=C['cyan'])

# Current price line
last_price = float(np.asarray(df['Close'])[-1])
ax_price.axhline(last_price, color=C['gold'], lw=0.7, ls=':', alpha=0.7, zorder=6)
ax_price.text(len(df)-0.3, last_price, f'  ▶ {last_price:,.2f}',
              color=C['gold2'], fontsize=9, va='center',
              fontweight='bold', fontfamily='monospace', zorder=10)

# Session legend patches
london_patch = mpatches.Patch(color=C['blue'],  alpha=0.25, label='London 10–16')
ny_patch     = mpatches.Patch(color=C['green'], alpha=0.25, label='NY 16–23')
legend_items = [
    mpatches.Patch(facecolor=C['bull'],        edgecolor=C['bull'],      label='Bullish'),
    mpatches.Patch(facecolor=C['bear'],        edgecolor=C['bear_edge'], label='Bearish'),
    mpatches.Patch(facecolor=C['sig_abuy'],    label='▲ Agg Buy'),
    mpatches.Patch(facecolor=C['sig_asell'],   label='▼ Agg Sell'),
    mpatches.Patch(facecolor=C['sig_absbuy'],  label='◆ Abs Buy'),
    mpatches.Patch(facecolor=C['sig_abssell'], label='◆ Abs Sell'),
    mpatches.Patch(facecolor=C['sig_stop'],    label='✕ Stop Hunt'),
    mpatches.Patch(facecolor=C['sig_iceberg'], label='■ Iceberg'),
    mpatches.Patch(facecolor=C['vwap'],        label='VWAP'),
    mpatches.Patch(facecolor=C['blue'],        label='EMA20'),
    mpatches.Patch(facecolor=C['orange'],      label='EMA50'),
    mpatches.Patch(facecolor=C['amber'],       alpha=0.4, label='Vol Cluster'),
    london_patch, ny_patch,
]
ax_price.legend(handles=legend_items, loc='upper left',
                facecolor=C['bg'], edgecolor=C['border'],
                labelcolor=C['lgrey'], fontsize=6.5,
                ncol=7, framealpha=0.95,
                prop={'family': 'monospace'})

style(ax_price,
      title=f'XAUUSD  ·  15M  ·  LARGE ORDER DETECTION  ·  '
            f'LAST {LOOKBACK_DAYS} DAYS  ·  '
            f'{NOW.strftime("%d %b %Y  %H:%M")} ATHENS  ·  '
            f'SOURCE: {TICKER}',
      ylabel='USD/oz')
ax_price.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v,_: f'{v:,.1f}'))

# ── PANEL 2: VOLUME ───────────────────────────────────────────────────────────
vol_cols = []
for _, row in df.iterrows():
    if   row['sig_aggressive_buy']:                           vol_cols.append(C['sig_abuy'])
    elif row['sig_aggressive_sell']:                          vol_cols.append(C['sig_asell'])
    elif row['sig_iceberg']:                                  vol_cols.append(C['sig_iceberg'])
    elif row['sig_absorption_buy'] or row['sig_absorption_sell']: vol_cols.append(C['cyan'])
    elif row['sig_cluster']:                                  vol_cols.append(C['amber'])
    elif row['is_bullish']:                                   vol_cols.append('#1A3A1A')
    else:                                                     vol_cols.append('#1A0A0F')

ax_vol.bar(x, df['Volume'].values, color=vol_cols, edgecolor='none', width=0.85, zorder=3)
vol_ma_s = gaussian_filter1d(df['vol_ma'].fillna(0).values, sigma=1.5)
ax_vol.plot(x, vol_ma_s,           color=C['gold'], lw=1.4, alpha=0.9,  zorder=4)
ax_vol.plot(x, vol_ma_s,           color=C['gold'], lw=3.0, alpha=0.12, zorder=3)
ax_vol.axhline(df['vol_ma'].mean() * VOL_SPIKE_MULT,
               color=C['red'], lw=0.7, ls='--', alpha=0.6, zorder=5)

style(ax_vol,
      title='VOLUME  ·  bright=signal  dim=normal  gold=MA  red--=spike threshold',
      ylabel='contracts')
ax_vol.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v,_: f'{v/1000:.1f}k'))

# ── PANEL 3: CUMULATIVE DELTA ─────────────────────────────────────────────────
cd   = df['cum_delta'].values
cd_s = gaussian_filter1d(cd, sigma=1.5)

ax_delta.fill_between(x, cd_s, 0, where=cd_s >= 0, alpha=0.25, color=C['green'], zorder=2)
ax_delta.fill_between(x, cd_s, 0, where=cd_s <  0, alpha=0.25, color=C['red'],   zorder=2)
ax_delta.plot(x, np.where(cd_s >= 0, cd_s, np.nan),
              color=C['green'], lw=1.4, alpha=0.9, zorder=4)
ax_delta.plot(x, np.where(cd_s <  0, cd_s, np.nan),
              color=C['red'],   lw=1.4, alpha=0.9, zorder=4)
ax_delta.axhline(0, color=C['lgrey'], lw=0.5, zorder=3)

# Divergence
price_arr = df['Close'].values
win = 8
for i in range(win, len(df)):
    pt = price_arr[i] - price_arr[i-win]
    dt = cd[i] - cd[i-win]
    ref = abs(cd[i-win]) if abs(cd[i-win]) > 0 else 1
    if pt > 0 and dt < -ref*0.15:
        ax_delta.scatter(i, cd_s[i], marker='v', s=40,
                        color=C['red'], zorder=6, alpha=0.9)
    elif pt < 0 and dt > ref*0.15:
        ax_delta.scatter(i, cd_s[i], marker='^', s=40,
                        color=C['green'], zorder=6, alpha=0.9)

style(ax_delta,
      title='CUMULATIVE DELTA  ·  green=buy pressure  red=sell pressure  ▲▼=price divergence',
      ylabel='Δvol')
ax_delta.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v,_: f'{v/1000:.1f}k'))

# ── PANEL 4: VOLUME Z-SCORE ───────────────────────────────────────────────────
zs = df['vol_zscore'].fillna(0).values
zs_cols = [C['red']    if z >= VOL_SPIKE_MULT else
           C['orange'] if z >= 2.0            else
           C['amber']  if z >= 1.5            else
           '#1a3a1a'   if z >= 0              else
           '#0d0d1a'   for z in zs]

ax_zscore.bar(x, zs, color=zs_cols, edgecolor='none', width=0.85, zorder=3)
ax_zscore.axhline(VOL_SPIKE_MULT, color=C['red'],   lw=0.9, ls='--', alpha=0.7, zorder=4)
ax_zscore.axhline(1.5,            color=C['amber'], lw=0.7, ls='--', alpha=0.5, zorder=4)
ax_zscore.axhline(0,              color=C['lgrey'], lw=0.4,          alpha=0.6, zorder=3)
ax_zscore.text(len(df)*0.995, VOL_SPIKE_MULT+0.08,
               f'{VOL_SPIKE_MULT}σ', color=C['red'],
               fontsize=6.5, ha='right', fontfamily='monospace')

style(ax_zscore,
      title='VOLUME Z-SCORE  ·  red=institutional spike  orange=elevated  amber=above avg',
      ylabel='σ')

# ── X-AXIS ────────────────────────────────────────────────────────────────────
n_ticks   = 16
tick_idxs = np.linspace(0, len(df)-1, n_ticks, dtype=int)
tick_lbls = [df.index[i].strftime('%d/%m\n%H:%M') for i in tick_idxs]
ax_zscore.set_xticks(tick_idxs)
ax_zscore.set_xticklabels(tick_lbls, color=C['lgrey'], fontsize=7,
                           fontfamily='monospace')
for ax in [ax_price, ax_vol, ax_delta]:
    plt.setp(ax.get_xticklabels(), visible=False)

for ax in [ax_price, ax_vol, ax_delta, ax_zscore]:
    ax.set_xlim(-0.8, len(df) - 0.2)

# ── TITLE ─────────────────────────────────────────────────────────────────────
fig.text(0.50, 0.974,
         f'ELENA GOLD  ·  INSTITUTIONAL ORDER FLOW  ·  XAUUSD 15M  ·  '
         f'{NOW.strftime("%A %d %B %Y  %H:%M")} ATHENS  ·  '
         f'${last_price:,.2f}',
         color=C['gold2'], fontsize=11, fontweight='bold',
         ha='center', va='center', fontfamily='monospace')

plt.savefig('Elena_XAUUSD_15m_OrderFlow.png', dpi=160,
            bbox_inches='tight', facecolor=C['bg'])
plt.show()

# ── CONSOLE SUMMARY ───────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  SIGNAL SUMMARY  ·  {TICKER}  ·  15M  ·  LAST {LOOKBACK_DAYS} DAYS")
print(f"{'═'*60}")
sig_map = {
    'sig_aggressive_buy':  ('AGG BUY',   '▲'),
    'sig_aggressive_sell': ('AGG SELL',  '▼'),
    'sig_absorption_buy':  ('ABS BUY',   '◆'),
    'sig_absorption_sell': ('ABS SELL',  '◆'),
    'sig_stop_hunt_low':   ('STOP ↓',    '✕'),
    'sig_stop_hunt_high':  ('STOP ↑',    '✕'),
    'sig_iceberg':         ('ICEBERG',   '■'),
    'sig_cluster':         ('CLUSTER',   '░'),
}
for col, (name, icon) in sig_map.items():
    n   = int(df[col].sum())
    bar = '█' * n + '░' * max(0, 25-n)
    print(f"  {icon} {name:<14} {n:3d}  {bar}")

print(f"\n{'─'*60}")
print(f"  MOST RECENT SIGNALS")
print(f"{'─'*60}")
sig_cols  = [c for c in sig_map if c != 'sig_cluster']
sig_names = [sig_map[c][0] for c in sig_cols]
for idx, row in df.tail(50).iterrows():
    fired = [sig_names[i] for i, col in enumerate(sig_cols) if row[col]]
    if fired:
        d = '↑' if row['is_bullish'] else '↓'
        print(f"  {idx.strftime('%d/%m %H:%M')}  {d}  "
              f"${row['Close']:,.2f}  "
              f"vol×{row['vol_ratio']:.1f}  →  "
              f"{' + '.join(fired)}")

print(f"\n  📊  Saved: Elena_XAUUSD_15m_OrderFlow.png")
print(f"{'═'*60}\n")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────────────────
BG        = '#0d1117'
PANEL_BG  = '#161b22'
GRID_COL  = '#21262d'
TEXT_COL  = '#e6edf3'
MUTED     = '#8b949e'
GREEN     = '#3fb950'
RED       = '#f85149'
BLUE      = '#58a6ff'
YELLOW    = '#e3b341'
PURPLE    = '#bc8cff'
ORANGE    = '#ff7b72'

plt.rcParams.update({
    'figure.facecolor':  BG,
    'axes.facecolor':    PANEL_BG,
    'axes.edgecolor':    GRID_COL,
    'axes.labelcolor':   MUTED,
    'axes.grid':         True,
    'grid.color':        GRID_COL,
    'grid.linewidth':    0.5,
    'grid.alpha':        0.6,
    'xtick.color':       MUTED,
    'ytick.color':       MUTED,
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
    'text.color':        TEXT_COL,
    'font.family':       'monospace',
    'lines.antialiased': True,
    'patch.antialiased': True,
})

# ── 1. FETCH REAL GOLD DATA ───────────────────────────────────────────────────
import yfinance as yf

raw = yf.download("GC=F", period="30d", interval="1h", auto_adjust=True, progress=False)
if hasattr(raw, 'columns') and isinstance(raw.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
    raw.columns = raw.columns.get_level_values(0)

if hasattr(raw, 'columns') and isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw.rename(columns={
    'Open': 'open', 'High': 'high',
    'Low': 'low', 'Close': 'close', 'Volume': 'volume'
})[['open', 'high', 'low', 'close', 'volume']].copy()

df = df.dropna().reset_index(drop=True)
df['bar']      = df.index
df['buy_vol']  = (df['volume'] * 0.55).astype(int)
df['sell_vol'] = (df['volume'] * 0.45).astype(int)
df['delta']    = df['buy_vol'] - df['sell_vol']

# ── 2. VWAP + BANDS ───────────────────────────────────────────────────────────
df['tp']     = (df['high'] + df['low'] + df['close']) / 3
df['cum_pv'] = (df['tp'] * df['volume']).cumsum()
df['cum_v']  = df['volume'].cumsum()
df['vwap']   = df['cum_pv'] / df['cum_v']

def rolling_vwap_sd(df, w=20):
    sds = []
    for i in range(len(df)):
        sl  = df.iloc[max(0, i - w): i + 1]
        vw  = sl['vwap'].iloc[-1]
        var = (sl['volume'] * (sl['tp'] - vw) ** 2).sum() / sl['volume'].sum()
        sds.append(np.sqrt(var))
    return np.array(sds)

df['vwap_sd'] = rolling_vwap_sd(df)

# ── 3. CVD ────────────────────────────────────────────────────────────────────
df['cvd'] = df['delta'].cumsum()

# ── 4. VOLUME PROFILE ─────────────────────────────────────────────────────────
def volume_profile(df, buckets=40):
    lo   = df['low'].min()
    hi   = df['high'].max()
    step = (hi - lo) / buckets
    centers, vols, buy_vols, sell_vols = [], [], [], []
    for b in range(buckets):
        blo  = lo + step * b
        bhi  = blo + step
        vol  = buy = sell = 0
        for _, row in df.iterrows():
            body_lo = min(row['open'], row['close'])
            body_hi = max(row['open'], row['close'])
            rng     = body_hi - body_lo or (row['high'] - row['low']) or 0.01
            overlap = max(0, min(body_hi, bhi) - max(body_lo, blo))
            frac    = overlap / rng
            vol    += row['volume']   * frac
            buy    += row['buy_vol']  * frac
            sell   += row['sell_vol'] * frac
        centers.append(round(lo + step * (b + 0.5), 2))
        vols.append(vol)
        buy_vols.append(buy)
        sell_vols.append(sell)

    prof   = pd.DataFrame({'price': centers, 'vol': vols,
                            'buy_vol': buy_vols, 'sell_vol': sell_vols})
    poc    = prof.loc[prof['vol'].idxmax(), 'price']
    sorted_p = prof.sort_values('vol', ascending=False)
    cum    = 0
    target = prof['vol'].sum() * 0.70
    va     = []
    for _, row in sorted_p.iterrows():
        cum += row['vol']
        va.append(row['price'])
        if cum >= target:
            break
    vah = max(va)
    val = min(va)
    return prof, round(poc, 2), round(vah, 2), round(val, 2), step

prof, poc, vah, val, vp_step = volume_profile(df)

# ── 5. ORDER BLOCKS ───────────────────────────────────────────────────────────
def detect_order_blocks(df, n=8):
    obs = []
    for i in range(2, len(df) - 1):
        prev = df.iloc[i - 1]
        nxt  = df.iloc[i + 1]
        avg_vol = df['volume'].rolling(10).mean().iloc[i]
        if (prev['close'] < prev['open'] and
                nxt['close'] > nxt['open'] * 1.001 and
                nxt['volume'] > avg_vol * 1.2):
            obs.append({'bar': i, 'type': 'bull',
                         'top': prev['high'], 'bot': prev['low'],
                         'strength': nxt['volume'] / avg_vol})
        if (prev['close'] > prev['open'] and
                nxt['close'] < nxt['open'] * 0.999 and
                nxt['volume'] > avg_vol * 1.2):
            obs.append({'bar': i, 'type': 'bear',
                         'top': prev['high'], 'bot': prev['low'],
                         'strength': nxt['volume'] / avg_vol})
    obs.sort(key=lambda x: x['strength'], reverse=True)
    return obs[:n]

order_blocks = detect_order_blocks(df)

# ── 6. STOP / LIQUIDITY ZONES ─────────────────────────────────────────────────
def detect_stop_zones(df, lookback=15, n=6):
    zones = []
    for i in range(lookback, len(df)):
        w  = df.iloc[i - lookback: i]
        sh = w['high'].max()
        sl = w['low'].min()
        if df.iloc[i]['high'] > sh:
            zones.append({'bar': i, 'price': round(sh, 2), 'type': 'above',
                           'label': 'Buy Stops'})
        if df.iloc[i]['low'] < sl:
            zones.append({'bar': i, 'price': round(sl, 2), 'type': 'below',
                           'label': 'Sell Stops'})
    return zones[-n:]

stop_zones = detect_stop_zones(df)

# ── 7. ABSORPTION ─────────────────────────────────────────────────────────────
def absorption_score(df, w=10):
    scores = []
    for i in range(len(df)):
        row        = df.iloc[i]
        body       = abs(row['close'] - row['open'])
        rng        = row['high'] - row['low'] or 0.01
        body_ratio = body / rng
        avg_vol    = df.iloc[max(0, i - w): i + 1]['volume'].mean()
        vol_ratio  = row['volume'] / (avg_vol or 1)
        scores.append(round(np.clip((vol_ratio - 1) * (1 - body_ratio) * 100, 0, 100), 1))
    return scores

df['absorption'] = absorption_score(df)

# ── 8. WYCKOFF PHASE ──────────────────────────────────────────────────────────
def wyckoff_phase(df, w=20):
    phases = []
    for i in range(len(df)):
        sl          = df.iloc[max(0, i - w): i + 1]
        prices      = sl['close'].values
        price_slope = (prices[-1] - prices[0]) / len(prices) if len(prices) > 1 else 0
        cvd_slope   = (sl['cvd'].iloc[-1] - sl['cvd'].iloc[0]) / len(sl)
        if price_slope > 0.3 and cvd_slope > 0:
            phases.append('Markup')
        elif price_slope < -0.3 and cvd_slope < 0:
            phases.append('Markdown')
        elif abs(price_slope) < 0.2 and cvd_slope > 0:
            phases.append('Accumulation')
        elif abs(price_slope) < 0.2 and cvd_slope < 0:
            phases.append('Distribution')
        else:
            phases.append('Ranging')
    return phases

df['phase'] = wyckoff_phase(df)

# ── 9. SIGNALS ────────────────────────────────────────────────────────────────
last         = df.iloc[-1]
cvd_trend    = df['cvd'].iloc[-1] - df['cvd'].iloc[-min(20, len(df)-1)]
price_trend  = last['close'] - df.iloc[-min(20, len(df)-1)]['close']
current_phase = df['phase'].iloc[-1]

if cvd_trend > 0 and price_trend < 0:
    divergence = 'BULLISH DIVERGENCE'
    div_color  = GREEN
elif cvd_trend < 0 and price_trend > 0:
    divergence = 'BEARISH DIVERGENCE'
    div_color  = RED
elif cvd_trend > 0:
    divergence = 'ALIGNED BULLISH'
    div_color  = GREEN
else:
    divergence = 'ALIGNED BEARISH'
    div_color  = RED

if cvd_trend > 5000:
    bias = 'INSTITUTIONS ADDING LONGS'
    bias_col = GREEN
elif cvd_trend < -5000:
    bias = 'INSTITUTIONS ADDING SHORTS'
    bias_col = RED
elif df['absorption'].iloc[-1] > 55:
    bias = 'HEAVY ABSORPTION'
    bias_col = YELLOW
else:
    bias = 'NEUTRAL / RANGING'
    bias_col = MUTED

# ── 10. FIGURE LAYOUT ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 14), facecolor=BG, dpi=130)
fig.subplots_adjust(left=0.04, right=0.87, top=0.95, bottom=0.05,
                     hspace=0.04)

gs = GridSpec(4, 2, figure=fig,
              height_ratios=[5, 1.2, 1, 0.8],
              width_ratios=[1, 0.14],
              hspace=0.04, wspace=0.02)

ax_p   = fig.add_subplot(gs[0, 0])   # price
ax_vp  = fig.add_subplot(gs[0, 1], sharey=ax_p)   # volume profile
ax_cvd = fig.add_subplot(gs[1, 0], sharex=ax_p)   # CVD
ax_abs = fig.add_subplot(gs[2, 0], sharex=ax_p)   # absorption
ax_vol = fig.add_subplot(gs[3, 0], sharex=ax_p)   # volume

for ax in [ax_p, ax_vp, ax_cvd, ax_abs, ax_vol]:
    ax.set_facecolor(PANEL_BG)
    for sp in ax.spines.values():
        sp.set_color(GRID_COL)
        sp.set_linewidth(0.8)

bars  = df['bar'].values
N     = len(df)
CW    = 0.38    # candle half-width

# ── PRICE: VWAP BANDS ─────────────────────────────────────────────────────────
ax_p.fill_between(bars, df['vwap'] - df['vwap_sd'] * 3,
                   df['vwap'] + df['vwap_sd'] * 3,
                   color=RED, alpha=0.05, zorder=1)
ax_p.fill_between(bars, df['vwap'] - df['vwap_sd'] * 2,
                   df['vwap'] + df['vwap_sd'] * 2,
                   color=ORANGE, alpha=0.07, zorder=1)
ax_p.fill_between(bars, df['vwap'] - df['vwap_sd'],
                   df['vwap'] + df['vwap_sd'],
                   color=BLUE, alpha=0.09, zorder=1)
ax_p.plot(bars, df['vwap'], color=BLUE, lw=1.4,
           label='VWAP', zorder=4, alpha=0.9)

# ── PRICE: POC / VAH / VAL LINES ─────────────────────────────────────────────
level_specs = [
    (poc, YELLOW, '--', 1.4, f'POC  {poc:.2f}'),
    (vah, BLUE,   ':',  1.2, f'VAH  {vah:.2f}'),
    (val, RED,    ':',  1.2, f'VAL  {val:.2f}'),
]
for price_lvl, col, ls, lw, lbl in level_specs:
    ax_p.axhline(price_lvl, color=col, lw=lw, ls=ls, alpha=0.85, zorder=3)
    ax_p.text(N + 0.5, price_lvl, lbl,
               color=col, fontsize=8, va='center',
               fontweight='bold', clip_on=False)

# ── PRICE: ORDER BLOCKS ───────────────────────────────────────────────────────
for ob in order_blocks:
    bx    = ob['bar']
    width = max(6, N * 0.04)
    fc    = '#1a3a28' if ob['type'] == 'bull' else '#3a1a1a'
    ec    = GREEN    if ob['type'] == 'bull' else RED
    lbl   = 'Bull OB' if ob['type'] == 'bull' else 'Bear OB'
    rect  = mpatches.FancyBboxPatch(
        (bx - width / 2, ob['bot']),
        width, ob['top'] - ob['bot'],
        boxstyle='square,pad=0',
        facecolor=fc, edgecolor=ec, lw=1.0,
        alpha=0.80, zorder=5)
    ax_p.add_patch(rect)
    ax_p.text(bx, (ob['top'] + ob['bot']) / 2, lbl,
               color=ec, fontsize=7, ha='center', va='center',
               fontweight='bold', zorder=6)
    # Target arrow
    arrow_y   = ob['top'] if ob['type'] == 'bull' else ob['bot']
    arrow_dir = 8 if ob['type'] == 'bull' else -8
    ax_p.annotate('', xy=(bx, arrow_y + arrow_dir),
                   xytext=(bx, arrow_y),
                   arrowprops=dict(arrowstyle='->', color=ec,
                                   lw=1.2, mutation_scale=10),
                   zorder=6)

# ── PRICE: STOP ZONES ─────────────────────────────────────────────────────────
for sz in stop_zones:
    col = GREEN if sz['type'] == 'above' else RED
    ax_p.axhline(sz['price'], color=col, lw=0.8, ls=(0, (2, 3)),
                  alpha=0.7, zorder=3)
    ax_p.scatter(sz['bar'], sz['price'], marker='D',
                  color=col, s=28, zorder=7, alpha=0.9)
    ax_p.text(sz['bar'] + 1, sz['price'],
               f"  {sz['label']}  {sz['price']:.0f}",
               color=col, fontsize=7, va='center', zorder=7)

# ── PRICE: CANDLES ────────────────────────────────────────────────────────────
for _, row in df.iterrows():
    bull  = row['close'] >= row['open']
    col   = GREEN if bull else RED
    fc    = '#1a3a28' if bull else '#3a1a1a'
    # Wick
    ax_p.plot([row['bar'], row['bar']],
               [row['low'], row['high']],
               color=col, lw=0.8, zorder=8, solid_capstyle='round')
    # Body
    bot = min(row['open'], row['close'])
    ht  = abs(row['close'] - row['open']) or (row['high'] - row['low']) * 0.1
    rect = mpatches.Rectangle(
        (row['bar'] - CW, bot), CW * 2, ht,
        facecolor=fc, edgecolor=col, lw=0.6, zorder=9)
    ax_p.add_patch(rect)

# ── PRICE: AXIS FORMATTING ────────────────────────────────────────────────────
ax_p.set_xlim(-1, N + 6)
price_range = df['high'].max() - df['low'].min()
ax_p.set_ylim(df['low'].min() - price_range * 0.05,
               df['high'].max() + price_range * 0.08)
ax_p.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
ax_p.yaxis.tick_right()
ax_p.yaxis.set_label_position('right')
ax_p.set_ylabel('Price  (USD)', color=MUTED, fontsize=9)
plt.setp(ax_p.get_xticklabels(), visible=False)

# Bias label top-left inside chart
ax_p.text(0.01, 0.98, f'  {bias}  ',
           transform=ax_p.transAxes,
           color=BG, fontsize=9, fontweight='bold', va='top',
           bbox=dict(facecolor=bias_col, edgecolor='none',
                     boxstyle='round,pad=0.3', alpha=0.9))
ax_p.text(0.01, 0.92, f'  {divergence}  ',
           transform=ax_p.transAxes,
           color=BG, fontsize=8, fontweight='bold', va='top',
           bbox=dict(facecolor=div_color, edgecolor='none',
                     boxstyle='round,pad=0.25', alpha=0.75))
ax_p.text(0.01, 0.86, f'  Phase: {current_phase}  ',
           transform=ax_p.transAxes,
           color=TEXT_COL, fontsize=8, va='top',
           bbox=dict(facecolor='#21262d', edgecolor=GRID_COL,
                     boxstyle='round,pad=0.25', alpha=0.9))

# ── VOLUME PROFILE (right panel) ──────────────────────────────────────────────
max_v = prof['vol'].max()
for _, row in prof.iterrows():
    total_w = (row['vol'] / max_v)
    buy_w   = total_w * (row['buy_vol'] / row['vol']) if row['vol'] > 0 else 0
    sell_w  = total_w - buy_w
    ht      = vp_step * 0.85
    ax_vp.barh(row['price'], buy_w,  height=ht,
                color=GREEN, alpha=0.65, left=0)
    ax_vp.barh(row['price'], sell_w, height=ht,
                color=RED,   alpha=0.55, left=buy_w)
ax_vp.axhline(poc, color=YELLOW, lw=1.5, ls='--', alpha=0.9)
ax_vp.set_xlim(0, 1.15)
ax_vp.set_xticks([])
ax_vp.tick_params(labelleft=False, left=False)
for sp in ax_vp.spines.values():
    sp.set_color(GRID_COL)
ax_vp.text(0.5, 1.01, 'Vol Profile', transform=ax_vp.transAxes,
            fontsize=7, color=MUTED, ha='center', va='bottom')

# ── CVD ───────────────────────────────────────────────────────────────────────
cvd_vals  = df['cvd'].values
cvd_color = GREEN if cvd_trend > 0 else RED
ax_cvd.fill_between(bars, cvd_vals, 0,
                     where=cvd_vals >= 0, color=GREEN, alpha=0.20)
ax_cvd.fill_between(bars, cvd_vals, 0,
                     where=cvd_vals < 0,  color=RED,   alpha=0.20)
ax_cvd.plot(bars, cvd_vals, color=cvd_color, lw=1.3)
ax_cvd.axhline(0, color=GRID_COL, lw=0.8)
ax_cvd.set_ylabel('CVD', color=MUTED, fontsize=8)
ax_cvd.yaxis.tick_right()
ax_cvd.yaxis.set_label_position('right')
ax_cvd.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x/1000:.0f}K'))
plt.setp(ax_cvd.get_xticklabels(), visible=False)

# CVD annotation
ax_cvd.text(0.01, 0.85,
             f"CVD trend: {'▲' if cvd_trend > 0 else '▼'} {abs(cvd_trend):,.0f}",
             transform=ax_cvd.transAxes, color=cvd_color,
             fontsize=8, va='top', fontweight='bold')

# ── ABSORPTION ────────────────────────────────────────────────────────────────
abs_vals = df['absorption'].values
abs_cols = [YELLOW if v > 60 else BLUE if v > 35 else GRID_COL
             for v in abs_vals]
ax_abs.bar(bars, abs_vals, color=abs_cols, width=0.8, zorder=3)
ax_abs.axhline(60, color=YELLOW, lw=0.8, ls='--', alpha=0.6)
ax_abs.set_ylim(0, 110)
ax_abs.set_ylabel('Absorb %', color=MUTED, fontsize=8)
ax_abs.yaxis.tick_right()
ax_abs.yaxis.set_label_position('right')
ax_abs.set_yticks([0, 60, 100])
plt.setp(ax_abs.get_xticklabels(), visible=False)

# ── VOLUME BARS ───────────────────────────────────────────────────────────────
vol_cols = [GREEN + '99' if r['close'] >= r['open'] else RED + '99'
             for _, r in df.iterrows()]
ax_vol.bar(bars, df['volume'], color=vol_cols, width=0.8)
avg_vol_line = df['volume'].rolling(20).mean()
ax_vol.plot(bars, avg_vol_line, color=BLUE, lw=1.0, alpha=0.7)
ax_vol.set_ylabel('Volume', color=MUTED, fontsize=8)
ax_vol.yaxis.tick_right()
ax_vol.yaxis.set_label_position('right')
ax_vol.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x/1000:.0f}K'))

# X axis: show dates every N bars
step_x = max(1, N // 10)
ax_vol.set_xticks(bars[::step_x])
if hasattr(df.index, 'strftime'):
    pass
date_labels = pd.date_range(start=0, periods=0)
ax_vol.set_xticklabels(
    [str(i) for i in bars[::step_x]], fontsize=7, color=MUTED)
ax_vol.set_xlabel('Bar  (1H candles, most recent → right)', color=MUTED, fontsize=8)

# ── LEGEND ────────────────────────────────────────────────────────────────────
legend_items = [
    Line2D([0],[0], color=BLUE,   lw=1.5,          label='VWAP'),
    Line2D([0],[0], color=YELLOW, lw=1.2, ls='--', label=f'POC {poc:.0f}'),
    Line2D([0],[0], color=BLUE,   lw=1.0, ls=':',  label=f'VAH {vah:.0f}'),
    Line2D([0],[0], color=RED,    lw=1.0, ls=':',  label=f'VAL {val:.0f}'),
    mpatches.Patch(facecolor='#1a3a28', edgecolor=GREEN, label='Bull Order Block'),
    mpatches.Patch(facecolor='#3a1a1a', edgecolor=RED,   label='Bear Order Block'),
    Line2D([0],[0], color=GREEN,  lw=0, marker='D', ms=5, label='Stop zone (above)'),
    Line2D([0],[0], color=RED,    lw=0, marker='D', ms=5, label='Stop zone (below)'),
    mpatches.Patch(facecolor=GREEN+'33', edgecolor=GREEN, label='Vol Profile Buy'),
    mpatches.Patch(facecolor=RED+'33',   edgecolor=RED,   label='Vol Profile Sell'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=5,
            facecolor='#161b22', edgecolor=GRID_COL,
            labelcolor=TEXT_COL, fontsize=8,
            framealpha=0.95, bbox_to_anchor=(0.44, 0.0))

# ── TITLE ─────────────────────────────────────────────────────────────────────
fig.suptitle(
    f'XAUUSD  ·  Gold Institutional Flow  ·  1H  ·  '
    f'Last: ${last["close"]:.2f}   '
    f'VWAP: ${df["vwap"].iloc[-1]:.2f}   '
    f'Abs: {df["absorption"].iloc[-1]:.0f}%',
    fontsize=11, color=TEXT_COL, fontweight='bold', y=0.975)

plt.savefig('gold_pro.png', dpi=150, bbox_inches='tight',
             facecolor=BG, edgecolor='none')
plt.show()

# ── PRINT SUMMARY ─────────────────────────────────────────────────────────────
print('=' * 55)
print(f'  XAUUSD  |  {len(df)} bars  |  1H')
print('=' * 55)
print(f'  Last close : ${last["close"]:.2f}')
print(f'  VWAP       : ${df["vwap"].iloc[-1]:.2f}')
print(f'  POC        : ${poc:.2f}')
print(f'  VAH        : ${vah:.2f}')
print(f'  VAL        : ${val:.2f}')
print(f'  Phase      : {current_phase}')
print(f'  Bias       : {bias}')
print(f'  CVD signal : {divergence}')
print(f'  Absorption : {df["absorption"].iloc[-1]:.1f}%')
print('-' * 55)
print('  Order Blocks detected:')
for ob in order_blocks[:5]:
    print(f'    {"BULL" if ob["type"] == "bull" else "BEAR"}  '
          f'${ob["bot"]:.0f} – ${ob["top"]:.0f}  '
          f'(strength x{ob["strength"]:.1f})')
print('-' * 55)
print('  Stop / Liquidity Zones:')
for sz in stop_zones:
    print(f'    {sz["label"]:12s}  ${sz["price"]:.2f}  @ bar {sz["bar"]}')
print('=' * 55)
